# DR-VERGE — Final Research Notebook

**A rigorous investigation of complementarity-aware knowledge transfer and INT8 deployment for
lightweight two-field diabetic retinopathy grading.**

Implements `revision/dr-verge-rev.md` on top of the validated rev3 core. Supersedes
`full_pipeline_notebook_rev3.ipynb`.

---

## Research questions (locked before running)

**RQ1 — Knowledge transfer.** *To what extent can Complementarity-Shift Distillation transfer the
dual-view decision benefit of a two-field teacher to a lightweight student, compared with no
distillation, standard logit distillation, and feature distillation?*

Judged on **two independent axes**, so the finding is informative regardless of which way it lands:
- *Predictive*: QWK (primary), Accuracy, Macro-F1, MAE, Severe-Error Rate
- *Mechanistic*: ShiftL1/ShiftMAE, Cosine agreement, Benefit correlation, internal/external dual-view gain

**The comparison ladder, named precisely.** Feature-KD and CSD each add ONE term on top of the same
tuned logit-KD baseline, so the four conditions are

```
no distillation  ->  logit-KD  ->  logit-KD + feature-KD  ->  logit-KD + CSD
```

Write it as *"CSD augmentation and feature-distillation augmentation over a standard logit-KD
baseline"* — never a bare "CSD vs Feature-KD", which would imply two disjoint methods.
`table_condition_labels.csv` carries the exact label for every condition.

**Δ is an operational proxy.** `Δ = p_dual − (p_macula + p_disc)/2` is measured through three heads,
so it can also absorb head discrepancy and calibration discrepancy. Describe it as an **operational
proxy of the dual-view decision shift**, not as pure anatomical complementarity. The same-head
counterfactual ablation (`abl_csd_counterfactual`) is the control that bounds this concern.

If QWK(CSD) ≈ QWK(KD) but ShiftFidelity(CSD) > ShiftFidelity(KD), that is still a scientific
finding. If CSD fails on both, that is a valid answer too.

**RQ2 — Quantization / deployment.** *To what extent can INT8 post-training quantization and
quantization-aware training reduce model size and CPU latency while preserving categorical and
ordinal grading performance of the best lightweight dual-view model?*

Compares `M*_FP32` vs `M*_PTQ-INT8` vs `M*_QAT-INT8` (plus a matched FP32 fine-tuning control),
where `M*` is selected **on validation only**. PTQ and QAT quantize the **identical operator set**
(eager, backbone-only) so the comparison isolates the training procedure, not the scope.

---

## Locked protocol (do not change after the first full run)

| Item | Value |
|---|---|
| Primary metric | **QWK** (ordinal; DR grades are 0<1<2<3<4) |
| Core seeds | 42, 123, 2026, 3407, 8888 (**5**) |
| Baseline seeds | 42, 123, 2026 (**3**) |
| Model selection | `argmax QWK_val`; ties (<0.005) → Macro-F1 → lower SER → lower MAE → simpler method |
| Test set | DRTiD official test — not used for selection **within this locked run** (see note) |
| External validation | DeepDRiD — frozen, no tuning, evaluated last |
| Statistics | Hierarchical paired cluster bootstrap over **matched seeds** + cluster permutation test, B=10,000, Holm-corrected |
| Deployment | Every exported artifact is **re-loaded from disk** and re-checked, quantized ones included |
| Pre-registered comparisons | RQ1: CSD vs {NoDistill, LogitKD, FeatureKD}. RQ2: {PTQ, QAT} vs FP32, QAT vs PTQ |

**Everything is saved.** Every figure ships PNG+PDF+SVG **and** a companion CSV — no number lives
only inside an image. Per-sample predictions, per-epoch gradient contributions, configs, metadata,
and a model registry are all written to disk.

---

## What this adds over rev3

rev3 fixed the three defects that made rev2's RQ1 test uninformative (collapsed CORAL thresholds,
40×-undersized student, CSD with no gradient). That core is **kept unchanged**. This notebook adds:

1. 5 seeds on core conditions (was 3)
2. Complete categorical metrics: Accuracy, Balanced Accuracy, macro/weighted P/R/F1, per-grade
   P/R/F1/specificity/support
3. Confusion matrices (raw + normalized) with **automatic prediction-collapse warnings**
4. **QAT** alongside PTQ — RQ2 becomes a three-way FP32/PTQ/QAT comparison
5. Quantization with a **matched scope**: PTQ and QAT both eager backbone-only, so RQ2 compares the
   procedure and not the operator set. PT2E (`torch.export` + `prepare_pt2e`) is run as a
   **supplementary** deployment-path demonstration, reported separately and excluded from RQ2.
   Export artifacts are `checkpoint.pt` / `model_object.pt` / `model.pt2` / `model.onnx`
   (TorchScript is deprecated and is no longer the deployment path)
6. Full efficiency suite: params, serialized size, compression ratio, mean/median/p95/p99 latency,
   throughput, speedup, memory — under a standardized benchmark protocol
7. Performance-retention metrics (INT8 vs FP32)
8. Statistics: hierarchical paired cluster bootstrap over MATCHED seeds + permutation p-values
9. DeepDRiD **external confirmatory validation**, frozen
10. Deployment artifacts + `predict_dr()` inference wrapper + parity checks + model registry
11. Ten publication-grade figures, each with a companion data CSV
12. Gates 0–9 with a final consolidated gate report

## 01 — Environment & Reproducibility (Gate 0)

In [1]:
!nvidia-smi

Sun Aug  9 05:24:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Colab ships a CUDA-enabled torch -- we deliberately do NOT reinstall it. torchao is needed for
# the modern PT2E quantization path (PTQ + QAT); if it is unavailable the notebook falls back to
# eager-mode quantization and SAYS SO explicitly rather than pretending the modern path ran.
# Quantization APIs are highly version-sensitive and the PyTorch ecosystem is actively migrating
# to torchao/PT2E, so versions are PINNED. Loosen only deliberately, and re-run preflight if you do.
!pip install -q "albumentations==1.4.21" "scikit-learn==1.5.2" "pandas==2.2.2" "tqdm==4.66.5"                "pyyaml==6.0.2" "psutil==6.0.0" "onnx==1.17.0" "onnxruntime==1.19.2" "scipy==1.14.1"
!pip install -q torchao || echo "torchao unavailable -- PT2E path will report as unavailable (not silently faked)"

import torch, torchvision, numpy, sklearn, platform, subprocess, json, os
print("torch       :", torch.__version__)
print("torchvision :", torchvision.__version__)
print("numpy       :", numpy.__version__)
print("sklearn     :", sklearn.__version__)
try:
    import torchao; print("torchao     :", torchao.__version__)
except Exception as e:
    print("torchao     : NOT AVAILABLE ->", e)
print("python      :", platform.python_version())
print("CUDA avail  :", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
print("quant engines:", torch.backends.quantized.supported_engines)
assert torch.cuda.is_available(), "No GPU -- Runtime > Change runtime type > GPU."

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.9/227.9 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 95.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.5/290.5 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.2/13.2 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 9.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not curre

## 02–03 — Locked configuration & paths

In [4]:
import os, json, hashlib, platform, subprocess, math, random, time, copy
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F

# ---------------- EDIT THESE ----------------
DRIVE_BASE = "/content/drive/MyDrive/DR-VERGE"

# PREFLIGHT=True runs the ENTIRE pipeline end-to-end at tiny scale into a SEPARATE namespace:
#   backbones 1 epoch | teacher 1+1 | students 2 epochs | 1 seed | grids truncated to 2 points
#   PTQ 4 calibration batches | QAT + FP32-FT 1 epoch | bootstrap/permutations 300 | benchmark 10 runs
# It proves every stage EXECUTES and every required RQ column is populated. It proves NOTHING about
# accuracy -- the models are deliberately undertrained, so Gate 2 / Gate 4 are non-blocking here and
# their preflight verdicts are meaningless. Read the gate report for "did it run", not "is it good".
# Set False for the real run.
PREFLIGHT = True

# RUN_TAG isolates this run's artifacts. A fresh tag for the final run guarantees no checkpoint from
# rev2/rev3 can be silently reused: checkpoint_is_compatible() only checks key/shape, so a rev3
# checkpoint with an identical architecture but different transforms/loss/seed protocol WOULD be
# accepted. A new namespace removes that ambiguity entirely.
RUN_TAG = "preflight_v1" if PREFLIGHT else "final_locked_v1"
# --------------------------------------------

DATASET_ROOT = f"{DRIVE_BASE}/dataset"

def _resolve_drtid_root(root):
    for cand in (f"{root}/DRTiD/DRTiD", f"{root}/DRTiD"):
        if os.path.exists(f"{cand}/Ground Truths/DR_grade/a. DR_grade_Training.csv"):
            return cand
    return f"{root}/DRTiD/DRTiD"

def _resolve_deepdrid_root(root):
    for cand in (f"{root}/DeepDRiD-master/regular_fundus_images",
                 f"{root}/DeepDRiD/regular_fundus_images",
                 f"{root}/DeepDRiD-master", f"{root}/DeepDRiD"):
        if os.path.exists(f"{cand}/regular-fundus-validation/regular-fundus-validation.csv"):
            return cand
    return None

DRTID_ROOT    = _resolve_drtid_root(DATASET_ROOT)
APTOS_ROOT    = f"{DATASET_ROOT}/APTOS"
DEEPDRID_ROOT = _resolve_deepdrid_root(DATASET_ROOT)

ART          = f"{DRIVE_BASE}/artifacts_{RUN_TAG}"
SPLITS_DIR   = f"{ART}/splits"
CKPT_DIR     = f"{ART}/checkpoints"
MODELS_DIR   = f"{ART}/models"
RESULTS_DIR  = f"{ART}/results"
FIGURES_DIR  = f"{RESULTS_DIR}/figures"
TABLES_DIR   = f"{RESULTS_DIR}/tables"
METRICS_DIR  = f"{RESULTS_DIR}/metrics"
PREDS_DIR    = f"{RESULTS_DIR}/predictions"
LOGS_DIR     = f"{RESULTS_DIR}/logs"
CONFIG_DIR   = f"{ART}/configs"
for d in [ART, SPLITS_DIR, CKPT_DIR, MODELS_DIR, RESULTS_DIR, FIGURES_DIR, TABLES_DIR,
          METRICS_DIR, PREDS_DIR, LOGS_DIR, CONFIG_DIR,
          f"{CKPT_DIR}/pretrained_backbones", f"{CKPT_DIR}/teacher", f"{CKPT_DIR}/student"]:
    os.makedirs(d, exist_ok=True)

# ================= LOCKED EXPERIMENT PROTOCOL =================
SEEDS_CORE     = [42, 123, 2026, 3407, 8888]   # no-distill / logit-KD / feature-KD / CSD
SEEDS_BASELINE = [42, 123, 2026]               # single-view baselines, ablations
SEEDS_QAT      = [42, 123, 2026]               # QAT robustness (its own seeds, see QAT section)
PRIMARY_SEED   = 42
# Pre-registered inferential seed: statistics use ALL matched core seeds, but where a single seed
# must be named it is this one -- fixed in advance, never the best-performing one.
INFERENTIAL_SEED = 42

if PREFLIGHT:
    SEEDS_CORE, SEEDS_BASELINE, SEEDS_QAT = [42], [42], [42]

IMG_SIZE       = 224
NUM_CLASSES    = 5
NUM_THRESHOLDS = NUM_CLASSES - 1
POS_WEIGHT_MODE   = "sqrt"                                  # none | sqrt | full
STUDENT_CHANNELS  = (32, 64, 96, 128, 160, 192, 224)        # ~330K-param student
FUSION_TYPE       = "interaction_mlp"

# Model selection (validation only) -- tie-break chain fixed in advance
SELECTION_METRIC   = "QWK"
SELECTION_TIE_EPS  = 0.005
SELECTION_TIEBREAK = ["MacroF1", "-SevereErrorRate", "-MAE"]

# Statistics
BOOTSTRAP_B      = 10000
BOOTSTRAP_ALPHA  = 0.05
PREREGISTERED_COMPARISONS = {
    "RQ1": [("dual_csd", "dual_no_distill"), ("dual_csd", "dual_logitkd"), ("dual_csd", "dual_featkd")],
    # qat_int8 vs fp32_ft_control isolates fake-quantization adaptation from the effect of simply
    # giving the model extra fine-tuning epochs -- without it, any QAT gain is confounded.
    "RQ2": [("ptq_int8", "best_fp32"), ("qat_int8", "best_fp32"), ("qat_int8", "ptq_int8"),
            ("qat_int8", "fp32_ft_control")],
}

# Standardized CPU benchmark protocol
BENCH = {"batch_size": 1, "warmup": 50, "runs": 500, "threads": 1}
BENCH_PREFLIGHT = {"batch_size": 1, "warmup": 3, "runs": 10, "threads": 1}

# DeepDRiD field-order is NOT documented in its public CSVs (no column says which of _1/_2 is
# macula- vs disc-centred). Rather than hide that behind an assumption, external validation is
# evaluated under BOTH orderings and both are reported -- turning the unknown into a robustness check.
# PRE-REGISTERED primary ordering, fixed before any DeepDRiD label-performance is inspected.
# It matches DRTiD's documented convention (field 1 = macula-centred), which is the only prior we
# have. The reverse ordering is run as a SENSITIVITY analysis and reported as supplementary --
# whichever scores higher must NOT be promoted to the headline result after the fact.
DEEPDRID_PRIMARY_FIELD_ORDER = "_1=macula"
DEEPDRID_FIELD_ORDERS = [DEEPDRID_PRIMARY_FIELD_ORDER, "_1=disc"]

CONFIG_SNAPSHOT = dict(
    seeds_core=SEEDS_CORE, seeds_baseline=SEEDS_BASELINE, img_size=IMG_SIZE,
    num_classes=NUM_CLASSES, pos_weight_mode=POS_WEIGHT_MODE,
    student_channels=list(STUDENT_CHANNELS), fusion_type=FUSION_TYPE,
    selection_metric=SELECTION_METRIC, selection_tie_eps=SELECTION_TIE_EPS,
    selection_tiebreak=SELECTION_TIEBREAK, bootstrap_B=BOOTSTRAP_B, bench=BENCH,
)

_expected = [f"{DRTID_ROOT}/Ground Truths/DR_grade/a. DR_grade_Training.csv",
             f"{DRTID_ROOT}/Ground Truths/DR_grade/b. DR_grade_Testing.csv",
             f"{DRTID_ROOT}/Original Images",
             f"{APTOS_ROOT}/train_1.csv", f"{APTOS_ROOT}/valid.csv",
             f"{APTOS_ROOT}/train_images/train_images", f"{APTOS_ROOT}/val_images/val_images"]
_missing = [p for p in _expected if not os.path.exists(p)]
if _missing:
    raise FileNotFoundError("Dataset missing:\n" + "\n".join(_missing))

print("DRTID_ROOT    :", DRTID_ROOT)
print("APTOS_ROOT    :", APTOS_ROOT)
print("DEEPDRID_ROOT :", DEEPDRID_ROOT or "NOT FOUND -- external validation will be SKIPPED (reported, not hidden)")
print("artifacts     :", ART)

DRTID_ROOT    : /content/drive/MyDrive/DR-VERGE/dataset/DRTiD
APTOS_ROOT    : /content/drive/MyDrive/DR-VERGE/dataset/APTOS
DEEPDRID_ROOT : /content/drive/MyDrive/DR-VERGE/dataset/DeepDRiD-master/regular_fundus_images
artifacts     : /content/drive/MyDrive/DR-VERGE/artifacts_preflight_v1


In [5]:
# ---- Gate 0: environment lock + provenance ----
def _pip_freeze():
    try:
        return subprocess.check_output(["pip", "freeze"], text=True)
    except Exception as e:
        return f"(pip freeze failed: {e})"

def _git_commit():
    try:
        return subprocess.check_output(["git", "rev-parse", "HEAD"], text=True,
                                        stderr=subprocess.DEVNULL).strip()
    except Exception:
        return "unavailable (not a git checkout in this runtime)"

import torchvision, sklearn
try:
    import torchao; _torchao_v = torchao.__version__
except Exception:
    _torchao_v = None

ENVIRONMENT = {
    "torch": torch.__version__, "torchvision": torchvision.__version__,
    "torchao": _torchao_v, "numpy": np.__version__, "sklearn": sklearn.__version__,
    "python": platform.python_version(), "platform": platform.platform(),
    "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "quantized_engines": list(torch.backends.quantized.supported_engines),
    "git_commit": _git_commit(), "timestamp": pd.Timestamp.now().isoformat(),
}
with open(f"{CONFIG_DIR}/environment.json", "w") as f:
    json.dump(ENVIRONMENT, f, indent=2)
with open(f"{CONFIG_DIR}/pip_freeze.txt", "w") as f:
    f.write(_pip_freeze())
with open(f"{CONFIG_DIR}/config_locked.json", "w") as f:
    json.dump(CONFIG_SNAPSHOT, f, indent=2)

GATES = {}
class GateFailure(RuntimeError):
    pass

def record_gate(name, passed, detail="", blocking=False):
    """A gate that only prints a warning is a log line, not a gate. blocking=True raises and stops
    the run, because anything computed past a failed upstream gate is not interpretable."""
    GATES[name] = {"passed": bool(passed), "detail": detail, "blocking": bool(blocking)}
    print(f"{'PASS' if passed else 'FAIL'} | {name}" + (f" | {detail}" if detail else ""))
    if blocking and not passed:
        raise GateFailure(
            f"{name} FAILED and is blocking: {detail}. Fix the upstream cause before continuing."
        )
    return passed

record_gate("Gate0_Environment", True,
            f"torch={torch.__version__} torchao={_torchao_v} engines={ENVIRONMENT['quantized_engines']}")
print(json.dumps(ENVIRONMENT, indent=2))

PASS | Gate0_Environment | torch=2.11.0+cu128 torchao=0.10.0 engines=['qnnpack', 'onednn', 'x86', 'fbgemm']
{
  "torch": "2.11.0+cu128",
  "torchvision": "0.26.0+cu128",
  "torchao": "0.10.0",
  "numpy": "2.0.2",
  "sklearn": "1.5.2",
  "python": "3.12.13",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "cuda": "12.8",
  "gpu": "Tesla T4",
  "quantized_engines": [
    "qnnpack",
    "onednn",
    "x86",
    "fbgemm"
  ],
  "git_commit": "unavailable (not a git checkout in this runtime)",
  "timestamp": "2026-08-09T05:26:35.488098"
}


## 04 — Reproducibility utilities

In [6]:
def set_seed(seed: int, deterministic: bool = False):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    s = torch.initial_seed() % 2**32
    np.random.seed(s); random.seed(s)

def make_generator(seed):
    g = torch.Generator(); g.manual_seed(seed); return g

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for blk in iter(lambda: f.read(chunk), b""):
            h.update(blk)
    return h.hexdigest()

def robust_torch_load(path, map_location=None, retries=6, delay=1.0):
    """Drive's FUSE mount can lag behind its own writes -- retry rather than crash a long run."""
    last = None
    for i in range(retries):
        try:
            return torch.load(path, map_location=map_location, weights_only=False)
        except (FileNotFoundError, OSError) as e:
            last = e
            if i < retries - 1:
                print(f"  robust_torch_load retry {i+1}/{retries} for {path}")
                time.sleep(delay); delay *= 1.5
    raise last

def robust_torch_save(obj, path, retries=6, delay=1.0):
    last = None
    parent = os.path.dirname(path)
    for i in range(retries):
        try:
            if parent: os.makedirs(parent, exist_ok=True)
            torch.save(obj, path); return
        except (RuntimeError, OSError) as e:
            last = e
            if i < retries - 1:
                print(f"  robust_torch_save retry {i+1}/{retries} for {path}: {e}")
                time.sleep(delay); delay *= 1.5
    raise last

def checkpoint_is_compatible(ckpt_path, model, unwrap_key="model_state"):
    """Side-effect-free key/shape check -- never partially mutates `model`."""
    if not os.path.exists(ckpt_path): return False
    try:
        raw = robust_torch_load(ckpt_path, map_location="cpu")
        state = raw[unwrap_key] if (unwrap_key and isinstance(raw, dict) and unwrap_key in raw) else raw
        cur = model.state_dict()
        if set(state.keys()) != set(cur.keys()):
            miss = list(set(cur) - set(state))[:4]; unexp = list(set(state) - set(cur))[:4]
            raise RuntimeError(f"key mismatch missing={miss} unexpected={unexp}")
        for k in state:
            if state[k].shape != cur[k].shape:
                raise RuntimeError(f"shape mismatch '{k}': {tuple(state[k].shape)} vs {tuple(cur[k].shape)}")
        return True
    except Exception as e:
        print(f"  {os.path.basename(ckpt_path)} incompatible with current architecture ({e}) -- retraining.")
        return False

def save_json(obj, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f: json.dump(obj, f, indent=2, default=str)

print("Reproducibility utilities defined.")

device: cuda
Reproducibility utilities defined.


## 05 — DRTiD integrity, splits & exploratory statistics (Gate 1)

DRTiD ships an **official** train/test split, used as-is. We only carve train/val out of the
official training rows. `_1` = Macula, `_2` = Optic disc — confirmed against the CrossFiT
reference loader (DRTiD's own benchmark authors' code).

**Scope note (verified, not assumed):** every `ID` in DRTiD's ground truth appears exactly once and
none carries both an `L` and `R` row, so `ID` is a per-**eye** identifier with no patient linkage
exposed. Splits and bootstrap clustering group by `ID` because it is the finest key the data
provides — that is eye-wise, *not* verified patient-wise. Reported as a limitation, not papered over.

In [7]:
from sklearn.model_selection import train_test_split

def make_drtid_splits(seed=42, val_fraction=0.2, force=False):
    out = {k: f"{SPLITS_DIR}/drtid_{k}.csv" for k in ("train", "val", "test")}
    images_dir = f"{DRTID_ROOT}/Original Images"
    off_train = pd.read_csv(f"{DRTID_ROOT}/Ground Truths/DR_grade/a. DR_grade_Training.csv")
    off_test  = pd.read_csv(f"{DRTID_ROOT}/Ground Truths/DR_grade/b. DR_grade_Testing.csv")

    overlap = set(off_train["ID"]) & set(off_test["ID"])
    assert not overlap, f"Gate 1 FAILED: official train/test share IDs: {sorted(overlap)[:10]}"

    def std(df):
        # NOTE: DRTiD's public ground truth exposes `ID` only. Every ID occurs exactly once and none
        # carries both an L and R row, and the official CrossFiT loader does not treat it as a patient
        # key (it reads Grade/Macula/Optic disc and leaves ID commented out). We therefore call it
        # record_id, NOT patient_id, and all clustering built on it is EYE/RECORD-level -- never
        # described as patient-level in the paper.
        return pd.DataFrame({
            "record_id": df["ID"],
            "macula_path": df["Macula"].apply(lambda s: f"{images_dir}/{s}.jpg"),
            "disc_path":   df["Optic disc"].apply(lambda s: f"{images_dir}/{s}.jpg"),
            "grade":       df["Grade"],
            "laterality":  df["LR"],
        })

    if not force and all(os.path.exists(p) for p in out.values()):
        print("Splits already exist on Drive -- reusing (guarantees identical splits across sessions).")
    else:
        # STRATIFIED by grade. Grade 4 is only ~3.9% of the official training rows, so an unstratified
        # random split makes the validation grade composition highly seed-sensitive -- and validation
        # is what every model-selection decision rests on.
        tr_ids, va_ids = train_test_split(off_train["ID"].values, test_size=val_fraction,
                                          random_state=seed, stratify=off_train["Grade"].values)
        std(off_train[off_train["ID"].isin(tr_ids)]).to_csv(out["train"], index=False)
        std(off_train[off_train["ID"].isin(va_ids)]).to_csv(out["val"], index=False)
        std(off_test).to_csv(out["test"], index=False)

    dfs = {k: pd.read_csv(v) for k, v in out.items()}
    assert not (set(dfs["train"].record_id) & set(dfs["val"].record_id)), "Gate 1 FAILED: train/val overlap"
    assert not (set(dfs["val"].record_id) & set(dfs["test"].record_id)),  "Gate 1 FAILED: val/test overlap"
    assert not (set(dfs["train"].record_id) & set(dfs["test"].record_id)), "Gate 1 FAILED: train/test overlap"

    rows, ok = [], True
    for name, df in dfs.items():
        missing = [p for c in ("macula_path", "disc_path") for p in df[c] if not os.path.exists(p)]
        if missing:
            ok = False; print(f"  MISSING {len(missing)} images in {name}, e.g. {missing[:3]}")
        dist = df["grade"].value_counts().sort_index()
        absent = sorted(set(range(NUM_CLASSES)) - set(dist.index))
        if absent:
            ok = False; print(f"  {name}: grades {absent} ABSENT")
        rows.append({"split": name, "n_records_eyes": len(df), "n_images": 2 * len(df),
                     **{f"grade_{g}": int(dist.get(g, 0)) for g in range(NUM_CLASSES)}})
    stats = pd.DataFrame(rows)
    stats.to_csv(f"{TABLES_DIR}/table_00_dataset_statistics.csv", index=False)
    print(stats.to_string(index=False))

    manifest = {k: {"path": v, "sha256": sha256_file(v), "rows": len(dfs[k])} for k, v in out.items()}
    save_json(manifest, f"{CONFIG_DIR}/split_manifest.json")
    record_gate("Gate1_Dataset", ok, f"train/val/test = {len(dfs['train'])}/{len(dfs['val'])}/{len(dfs['test'])} eyes; "
                                     f"no ID overlap; all grades present; all images resolve")
    return out["train"], out["val"], out["test"]

DRTID_TRAIN_CSV, DRTID_VAL_CSV, DRTID_TEST_CSV = make_drtid_splits()

split  n_records_eyes  n_images  grade_0  grade_1  grade_2  grade_3  grade_4
train             800      1600      386       72      208      104       30
  val             200       400       96       18       52       26        8
 test             550      1100      265       50      146       69       20
PASS | Gate1_Dataset | train/val/test = 800/200/550 eyes; no ID overlap; all grades present; all images resolve


## 06 — Preprocessing & augmentation

In [8]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
from torch.utils.data import Dataset, DataLoader

# DRTiD channel stats from the CrossFiT authors' own loader -- keeps preprocessing aligned with
# the benchmark this work is positioned against.
DRTID_MEAN, DRTID_STD = [0.372487, 0.217266, 0.119367], [0.281526, 0.179457, 0.109162]
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def build_transforms(train, mean, std, geometric=True, photometric=True):
    # Horizontal flip deliberately OMITTED: it risks changing macula/disc laterality semantics, and
    # the CrossFiT reference implementation has its flip code commented out for the same reason.
    ops = [A.Resize(IMG_SIZE, IMG_SIZE)]
    if train and geometric:
        ops += [A.Rotate(limit=15, p=0.7)]
    if train and photometric:
        ops += [A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5)]
    ops += [A.Normalize(mean=mean, std=std), ToTensorV2()]
    return A.Compose(ops)

class PairedDualViewTransform:
    """Applies the SAME geometric transform to both fields, with independent photometric jitter.

    The two fields of one eye share an acquisition geometry, and CrossFiT's own loader applies its
    geometric ops jointly to the pair while keeping colour jitter separate. Rotating macula and disc
    by different random angles injects a spurious geometric discrepancy into exactly the quantity CSD
    is trying to learn (the difference between the two views), so geometry is shared here.
    """
    def __init__(self, mean, std, train=True):
        self.train = train
        self.geo = A.ReplayCompose([A.Resize(IMG_SIZE, IMG_SIZE)] +
                                   ([A.Rotate(limit=15, p=0.7)] if train else []))
        self.photo = A.Compose(([A.RandomBrightnessContrast(brightness_limit=0.15,
                                                            contrast_limit=0.15, p=0.5)] if train else []) +
                               [A.Normalize(mean=mean, std=std), ToTensorV2()])

    def __call__(self, img_macula, img_disc):
        g = self.geo(image=img_macula)                       # sample geometry once...
        a = g["image"]
        b = A.ReplayCompose.replay(g["replay"], image=img_disc)["image"]   # ...and replay it
        return self.photo(image=a)["image"], self.photo(image=b)["image"]

train_transform = build_transforms(True,  DRTID_MEAN, DRTID_STD)
eval_transform  = build_transforms(False, DRTID_MEAN, DRTID_STD)
paired_train_transform = PairedDualViewTransform(DRTID_MEAN, DRTID_STD, train=True)
aptos_train_transform = build_transforms(True,  IMAGENET_MEAN, IMAGENET_STD)
aptos_eval_transform  = build_transforms(False, IMAGENET_MEAN, IMAGENET_STD)

PREPROCESSING_META = {"input_size": [IMG_SIZE, IMG_SIZE], "normalization_mean": DRTID_MEAN,
                      "normalization_std": DRTID_STD, "horizontal_flip": False,
                      "views": ["macula", "optic_disc"], "ordinal_threshold": 0.5}

def _rgb(path): return np.array(Image.open(path).convert("RGB"))

class DRTiDDualViewDataset(Dataset):
    def __init__(self, split_csv, transform=None, paired_transform=None):
        self.df = pd.read_csv(split_csv)
        self.transform = transform if transform is not None else eval_transform
        self.paired_transform = paired_transform
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        if self.paired_transform is not None:
            a, b = self.paired_transform(_rgb(r["macula_path"]), _rgb(r["disc_path"]))
            return {"macula": a, "disc": b,
                    "label": torch.tensor(int(r["grade"]), dtype=torch.long),
                    "cluster_id": int(r["record_id"])}
        return {"macula": self.transform(image=_rgb(r["macula_path"]))["image"],
                "disc":   self.transform(image=_rgb(r["disc_path"]))["image"],
                "label":  torch.tensor(int(r["grade"]), dtype=torch.long),
                "cluster_id": int(r["record_id"])}   # eye/record level -- see Gate 1 note

class APTOSSingleViewDataset(Dataset):
    def __init__(self, csv_path, root_dir, transform=None):
        self.df = pd.read_csv(csv_path); self.root_dir = root_dir
        self.transform = transform if transform is not None else aptos_eval_transform
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = _rgb("{}/{}.png".format(self.root_dir, r["id_code"]))
        return {"image": self.transform(image=img)["image"],
                "label": torch.tensor(int(r["diagnosis"]), dtype=torch.long)}

def make_loader(ds, batch_size, shuffle, seed=None, workers=2):
    kw = {}
    if seed is not None:
        kw = {"worker_init_fn": seed_worker, "generator": make_generator(seed)}
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=workers, **kw)

# ---- Persistent evaluation loaders (RUNTIME BLOCKER FIX) ----
# Validation is read by Gate 2, Gate 4, every grid search and model selection. The earlier code
# created a throwaway validation loader inside the teacher cell and deleted it at the end of that
# cell; the logit-KD grid then called run_grid(), which still referenced that name, and the run died
# with NameError before a single grid point finished. One loader, defined here, never deleted.
VAL_DS     = DRTiDDualViewDataset(DRTID_VAL_CSV, eval_transform)
VAL_LOADER = make_loader(VAL_DS, 16, False)

print(f"Preprocessing defined. Persistent VAL_LOADER over {len(VAL_DS)} validation eyes.")

Preprocessing defined. Persistent VAL_LOADER over 200 validation eyes.


/usr/local/lib/python3.12/dist-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.21). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


## 07–09 — Model architecture, CORAL initialization & unit tests

**CORAL thresholds are initialized from the empirical marginal** `b_k = logit(P(Y>k))`. rev2
initialized all four thresholds within 0.15 logits of each other while DRTiD needs a 3.24-logit
spread; because CORAL gives each sample one scalar score compared against all thresholds, collapsed
thresholds make intermediate grades unreachable — measured rev2 sensitivity for Grades 1–3 was
0.00–0.04 across every condition including the teacher. Kept from rev3, with assertions.

In [9]:
import torchvision.models as tv

def compute_pos_weights(train_csv, num_thresholds=NUM_THRESHOLDS, grade_col="grade", mode=POS_WEIGHT_MODE):
    """pos_weight_k = N_neg/N_pos. Raw ratio is 24.8x at k=3 on DRTiD (31/800 eyes are Grade 4),
    which drove rev2's collapse onto the extreme grades. 'sqrt' keeps the correction's direction
    without its degeneracy."""
    g = pd.read_csv(train_csv)[grade_col].values
    w = []
    for k in range(num_thresholds):
        pos, neg = int((g > k).sum()), int((g <= k).sum())
        if pos == 0 or neg == 0:
            raise ValueError(f"degenerate threshold k={k}: pos={pos} neg={neg}")
        r = neg / pos
        w.append({"full": r, "sqrt": math.sqrt(r), "none": 1.0}[mode])
    return torch.tensor(w, dtype=torch.float32)

def compute_init_thresholds(train_csv, num_thresholds=NUM_THRESHOLDS, grade_col="grade", eps=1e-3):
    g = pd.read_csv(train_csv)[grade_col].values
    return [math.log(min(max(float((g > k).mean()), eps), 1 - eps) /
                     (1 - min(max(float((g > k).mean()), eps), 1 - eps))) for k in range(num_thresholds)]


class CORALHead(nn.Module):
    """Monotone cumulative outputs P(y>k) by construction (ordered non-negative softplus steps)."""
    def __init__(self, in_dim, num_classes=NUM_CLASSES, init_thresholds=None):
        super().__init__()
        self.num_thresholds = num_classes - 1
        self.fc = nn.Linear(in_dim, 1, bias=False)
        if init_thresholds is None:
            init_thresholds = [-0.7 * i for i in range(self.num_thresholds)]
        t = torch.tensor(list(init_thresholds), dtype=torch.float32)
        if t.numel() != self.num_thresholds:
            raise ValueError(f"need {self.num_thresholds} thresholds, got {t.numel()}")
        gaps = (t[:-1] - t[1:]).clamp_min(1e-4)
        self.base_bias  = nn.Parameter(t[0].clone())
        self.bias_steps = nn.Parameter(torch.log(torch.expm1(gaps)).clone())

    def _ordered_biases(self):
        steps = F.softplus(self.bias_steps)
        cum = torch.cat([torch.zeros(1, device=steps.device), torch.cumsum(steps, dim=0)])
        return self.base_bias - cum

    def forward(self, z):
        logits = self.fc(z) + self._ordered_biases().unsqueeze(0)
        return logits, torch.sigmoid(logits)


class InteractionFusion(nn.Module):
    """Concat + |diff| + product through a small MLP (judge.md Flag 2: a bare linear fusion can
    only form a weighted sum and cannot represent cross-view interaction). LayerNorm rather than
    BatchNorm removes batch-size sensitivity at the small batch sizes used here.

    All submodules are defined unconditionally: TorchScript/export statically analyses every branch,
    and rev2's Gate 5 failed with "has no attribute 'norm'" because submodules were created only
    inside one branch of an if."""
    def __init__(self, feat_dim, fusion_type=FUSION_TYPE, hidden_dim=None):
        super().__init__()
        if fusion_type not in ("linear", "interaction_mlp"):
            raise ValueError(fusion_type)
        self.fusion_type = fusion_type
        hidden_dim = hidden_dim or feat_dim
        self.norm     = nn.LayerNorm(feat_dim * 2)
        self.norm_in  = nn.LayerNorm(feat_dim * 4)
        self.proj     = nn.Linear(feat_dim * 4, hidden_dim)
        self.act      = nn.ReLU(inplace=True)
        self.norm_out = nn.LayerNorm(hidden_dim)
        self.out_dim  = feat_dim * 2 if fusion_type == "linear" else hidden_dim

    def forward(self, z_m, z_d):
        if self.fusion_type == "linear":
            return self.norm(torch.cat([z_m, z_d], dim=1))
        combined = self.norm_in(torch.cat([z_m, z_d, torch.abs(z_m - z_d), z_m * z_d], dim=1))
        return self.norm_out(self.act(self.proj(combined)))


class DepthwiseSeparableBlock(nn.Module):
    """ReLU (not ReLU6): eager-mode fuse_modules has no fuser for Conv-BN-ReLU6."""
    def __init__(self, i, o, stride=1):
        super().__init__()
        self.dw = nn.Conv2d(i, i, 3, stride=stride, padding=1, groups=i, bias=False)
        self.bn1 = nn.BatchNorm2d(i); self.act1 = nn.ReLU(inplace=True)
        self.pw = nn.Conv2d(i, o, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(o); self.act2 = nn.ReLU(inplace=True)
    def forward(self, x):
        return self.act2(self.bn2(self.pw(self.act1(self.bn1(self.dw(x))))))
    def fuse(self, qat=False):
        fn = torch.ao.quantization.fuse_modules_qat if qat else torch.ao.quantization.fuse_modules
        fn(self, [["dw", "bn1", "act1"], ["pw", "bn2", "act2"]], inplace=True)


class LightweightBackbone(nn.Module):
    """~125K-param feature extractor. rev2's was 8,176 params (its fusion MLP was 75% of the whole
    34K model), ~40x below the technical doc's 0.3-0.4M target, which capacity-capped every
    dual-view condition at the same QWK and made RQ1 untestable."""
    def __init__(self, channels=None):
        super().__init__()
        ch = tuple(channels or STUDENT_CHANNELS)
        self.stem_conv = nn.Conv2d(3, ch[0], 3, stride=2, padding=1, bias=False)
        self.stem_bn = nn.BatchNorm2d(ch[0]); self.stem_act = nn.ReLU(inplace=True)
        strides = [2 if i % 2 == 0 else 1 for i in range(len(ch) - 1)]
        self.blocks = nn.ModuleList([DepthwiseSeparableBlock(ch[i], ch[i+1], strides[i])
                                     for i in range(len(ch) - 1)])
        self.gap = nn.AdaptiveAvgPool2d(1); self.out_dim = ch[-1]
    def forward(self, x):
        x = self.stem_act(self.stem_bn(self.stem_conv(x)))
        for b in self.blocks: x = b(x)
        return self.gap(x).flatten(1)
    def fuse_model(self, qat=False):
        fn = torch.ao.quantization.fuse_modules_qat if qat else torch.ao.quantization.fuse_modules
        fn(self, [["stem_conv", "stem_bn", "stem_act"]], inplace=True)
        for b in self.blocks: b.fuse(qat=qat)


class _DualViewBase(nn.Module):
    def forward(self, macula, disc):
        z_m, z_d = self.backbone(macula), self.backbone(disc)
        z_f = self.fusion(z_m, z_d)
        ld, pd_ = self.main_head(z_f)
        lm, pm = self.macula_head(z_m)
        ldd, pdd = self.disc_head(z_d)
        return {"p_dual": pd_, "logit_dual": ld, "p_macula": pm, "logit_macula": lm,
                "p_disc": pdd, "logit_disc": ldd, "z_fused": z_f}
    def forward_single(self, x, which="macula"):
        z = self.backbone(x)
        logit, p = (self.macula_head if which == "macula" else self.disc_head)(z)
        return {"logit": logit, "p": p}
    def counterfactual_forward(self, macula, disc):
        """Same-head counterfactual (judge.md Flag 1/3): dual / macula-only / disc-only all go
        through the SAME main_head, so their difference cannot be head discrepancy."""
        z_m, z_d = self.backbone(macula), self.backbone(disc)
        zero = torch.zeros_like(z_m)
        _, p_dual = self.main_head(self.fusion(z_m, z_d))
        _, p_m    = self.main_head(self.fusion(z_m, zero))
        _, p_d    = self.main_head(self.fusion(zero, z_d))
        return {"p_dual": p_dual, "p_macula_cf": p_m, "p_disc_cf": p_d}


class DualViewResNetTeacher(_DualViewBase):
    def __init__(self, num_classes=NUM_CLASSES, feat_dim=2048, fusion_type=FUSION_TYPE, init_thresholds=None):
        super().__init__()
        bb = tv.resnet50(weights=tv.ResNet50_Weights.IMAGENET1K_V2); bb.fc = nn.Identity()
        self.backbone = bb
        self.fusion = InteractionFusion(feat_dim, fusion_type)
        self.main_head   = CORALHead(self.fusion.out_dim, num_classes, init_thresholds)
        self.macula_head = CORALHead(feat_dim, num_classes, init_thresholds)
        self.disc_head   = CORALHead(feat_dim, num_classes, init_thresholds)


class DualViewLightStudent(_DualViewBase):
    def __init__(self, num_classes=NUM_CLASSES, backbone=None, fusion_type=FUSION_TYPE, init_thresholds=None):
        super().__init__()
        self.backbone = backbone or LightweightBackbone()
        fd = self.backbone.out_dim
        self.fusion = InteractionFusion(fd, fusion_type)
        self.main_head   = CORALHead(self.fusion.out_dim, num_classes, init_thresholds)
        self.macula_head = CORALHead(fd, num_classes, init_thresholds)
        self.disc_head   = CORALHead(fd, num_classes, init_thresholds)
    def fuse_model(self, qat=False):
        if hasattr(self.backbone, "fuse_model"): self.backbone.fuse_model(qat=qat)

INIT_THRESHOLDS = compute_init_thresholds(DRTID_TRAIN_CSV)
POS_WEIGHT = compute_pos_weights(DRTID_TRAIN_CSV)
print("CORAL init thresholds :", [round(t, 4) for t in INIT_THRESHOLDS])
print("  implied P(y>k)      :", [round(float(torch.sigmoid(torch.tensor(t))), 4) for t in INIT_THRESHOLDS])
print(f"pos_weight ({POS_WEIGHT_MODE:4s})     :", [round(float(w), 3) for w in POS_WEIGHT])

CORAL init thresholds : [0.07, -0.2921, -1.6034, -3.2452]
  implied P(y>k)      : [0.5175, 0.4275, 0.1675, 0.0375]
pos_weight (sqrt)     : [0.966, 1.157, 2.229, 5.066]


In [10]:
# ---- Unit tests on the ordinal head (fail loudly, before any training) ----
def test_coral_head():
    h = CORALHead(16, NUM_CLASSES, INIT_THRESHOLDS)
    b = h._ordered_biases().detach()
    assert torch.all(b[:-1] >= b[1:]), "thresholds must be non-increasing"
    spread = float(b[0] - b[-1])
    assert spread > 1.5, f"threshold spread {spread:.3f} too small -- predictions will collapse to extremes"
    _, p = h(torch.randn(32, 16))
    assert torch.all(p[:, :-1] >= p[:, 1:] - 1e-6), "P(y>k) must be non-increasing in k"
    emp = [float(torch.sigmoid(torch.tensor(t))) for t in INIT_THRESHOLDS]
    got = [float(torch.sigmoid(x)) for x in b]
    assert max(abs(a - c) for a, c in zip(emp, got)) < 1e-5, "init must reproduce empirical marginals"
    print(f"  CORAL unit tests PASSED (spread={spread:.3f} logits, monotone, matches marginals)")
    return spread

_spread = test_coral_head()

def test_fusion_interaction():
    f = InteractionFusion(8, "interaction_mlp").eval()
    a, b = torch.randn(4, 8), torch.randn(4, 8)
    assert not torch.allclose(f(a, b), f(b, a)), "fusion must not be order-invariant (it models interaction)"
    print("  InteractionFusion unit test PASSED (view-order sensitive => genuine interaction)")

test_fusion_interaction()
record_gate("Gate_CORAL_UnitTests", True, f"threshold spread {_spread:.3f} logits; monotone; matches marginals")

  CORAL unit tests PASSED (spread=3.315 logits, monotone, matches marginals)
  InteractionFusion unit test PASSED (view-order sensitive => genuine interaction)
PASS | Gate_CORAL_UnitTests | threshold spread 3.315 logits; monotone; matches marginals


True

## 10 — Loss definitions

`L = L_task + λ·L_aux + α·L_logitKD + β·L_CSD (+ γ·L_featKD)`

**What Δ is, and is not.** `Δ` is computed through three separate heads, so besides genuine
complementarity it can also carry head discrepancy and calibration discrepancy. It is therefore an
**operational proxy of the dual-view decision shift** — the wording the paper must use. The same-head
counterfactual (`counterfactual_forward`, ablation `abl_csd_counterfactual`) routes dual / macula /
disc through the *same* `main_head` and bounds how much of Δ could be head discrepancy.

**CSD (normalized).** `Δ = p_dual − (p_macula+p_disc)/2` for teacher and student; both are divided
by `s = mean(|Δ^T|)` (detached) before a Huber loss. Because the divisor is detached and identical
on both sides, the optimum is unchanged but the gradient becomes usable: rev2 logged `L_CSD≈0.014`
against `L_task≈0.82` (<0.5% of the objective, essentially no gradient), which is why its RQ1 test
could not have detected any CSD effect.

In [11]:
def coral_loss(logits, labels, num_thresholds=NUM_THRESHOLDS, pos_weight=None):
    levels = torch.arange(num_thresholds, device=logits.device).unsqueeze(0)
    y_k = (labels.unsqueeze(1) > levels).float()
    return F.binary_cross_entropy_with_logits(logits, y_k, pos_weight=pos_weight)

def aux_loss(out, labels, pos_weight=None):
    return (coral_loss(out["logit_macula"], labels, pos_weight=pos_weight) +
            coral_loss(out["logit_disc"],   labels, pos_weight=pos_weight))

def logit_kd_loss(logit_t, logit_s, tau=2.0):
    """No tau^2 factor (judge.md Flag 17): alpha and tau are therefore coupled -- do not claim
    independent temperature tuning in the paper."""
    return F.binary_cross_entropy(torch.sigmoid(logit_s / tau), torch.sigmoid(logit_t.detach() / tau))

def _delta(p_dual, p_m, p_d):
    return p_dual - (p_m + p_d) / 2

def csd_loss(p_dual_t, p_m_t, p_d_t, p_dual_s, p_m_s, p_d_s,
             variant="smoothl1_norm", tau_csd=0.5, huber_beta=1.0, eps=1e-6, scale=None):
    """Complementarity-Shift Distillation.

    NORMALIZATION (corrected): the scale `s` is a FIXED GLOBAL constant estimated once from the
    frozen teacher over the training split, not `mean(|delta_T|)` of the current batch. A per-batch
    divisor would amplify batches whose teacher shift happens to be small and shrink batches whose
    shift is large -- that is not a pure rescaling, it silently re-weights samples relative to each
    other. A single fixed scalar fixes the gradient magnitude problem while leaving the relative
    magnitude structure across samples and batches untouched.

    Variants
      smoothl1_norm                -- DEFAULT, globally-scaled signed Huber on delta
      smoothl1                     -- unscaled (rev2 formulation), ablation only
      magnitude_weighted_direction -- magnitude-weighted cosine + scaled magnitude term
      kl_softmax                   -- v1 formulation, negative control (destroys magnitude info)
    """
    dt = _delta(p_dual_t.detach(), p_m_t.detach(), p_d_t.detach())
    ds = _delta(p_dual_s, p_m_s, p_d_s)
    s_glob = float(scale) if scale is not None else float(globals().get("CSD_GLOBAL_SCALE", 1.0))
    s_glob = max(s_glob, 1e-3)

    if variant == "smoothl1_norm":
        return F.smooth_l1_loss(ds / s_glob, dt / s_glob, beta=huber_beta)
    if variant == "smoothl1":
        return F.smooth_l1_loss(ds, dt, beta=huber_beta)
    if variant == "magnitude_weighted_direction":
        mag = dt.norm(dim=1)
        w = (mag / mag.median().clamp_min(eps)).clamp(max=1.0)
        l_dir = ((1 - F.cosine_similarity(ds, dt, dim=1, eps=eps)) * w).sum() / w.sum().clamp_min(eps)
        return 0.5 * l_dir + 0.5 * F.smooth_l1_loss(ds / s_glob, dt / s_glob, beta=huber_beta)
    if variant == "kl_softmax":
        return F.kl_div(F.log_softmax(ds / tau_csd, dim=1), F.softmax(dt / tau_csd, dim=1), reduction="batchmean")
    raise ValueError(f"unknown csd_variant: {variant}")


@torch.no_grad()
def compute_global_delta_scale(teacher, loader, device, counterfactual=False):
    """E_train[|delta_T|] from the FROZEN teacher -- computed once, then held fixed for all training."""
    teacher.eval(); tot, n = 0.0, 0
    for b in loader:
        m, d = b["macula"].to(device), b["disc"].to(device)
        if counterfactual:
            o = teacher.counterfactual_forward(m, d)
            dt = _delta(o["p_dual"], o["p_macula_cf"], o["p_disc_cf"])
        else:
            o = teacher(m, d)
            dt = _delta(o["p_dual"], o["p_macula"], o["p_disc"])
        tot += float(dt.abs().sum()); n += dt.numel()
    return max(tot / max(n, 1), 1e-3)


def feature_kd_loss(z_t, z_s, projector):
    """Representation-level control.

    CORRECTED DIRECTION: the projector maps STUDENT -> TEACHER space and the teacher features are
    detached, so the regression target is FIXED. Projecting teacher->student with a trainable
    projector (the earlier form) lets the target drift as the projector learns, which makes this a
    weaker control than it appears -- and this is the primary control for CSD's novelty claim, so it
    has to be clean.
    """
    return F.mse_loss(projector(z_s), z_t.detach())


def get_student_output(student, macula, disc, view_mode):
    if view_mode == "dual":        return student(macula, disc)
    if view_mode == "macula_only": return student.forward_single(macula, "macula")
    if view_mode == "disc_only":   return student.forward_single(disc, "disc")
    raise ValueError(view_mode)

def ordinal_violation_rate(p):
    return float((p[:, 1:] - p[:, :-1] > 0).float().mean())

def combined_student_loss(teacher_out, student_out, labels, view_mode, alpha=0.0, beta=0.0,
                          lambda_aux=0.5, tau_kd=2.0, csd_variant="smoothl1_norm", tau_csd=0.5,
                          pos_weight=None, use_counterfactual_csd=False, teacher_cf_out=None,
                          student_cf_out=None, gamma_feat=0.0, feat_projector=None, huber_beta=1.0,
                          csd_scale=None):
    task_logit = student_out["logit_dual"] if view_mode == "dual" else student_out["logit"]
    l_task = coral_loss(task_logit, labels, pos_weight=pos_weight)
    total, log, comps = l_task, {"L_task": l_task.item()}, {"task": l_task}

    if view_mode == "dual":
        l_aux = aux_loss(student_out, labels, pos_weight=pos_weight)
        total = total + lambda_aux * l_aux
        log["L_aux"] = l_aux.item(); log["W_aux"] = float(lambda_aux * l_aux)
        comps["aux"] = lambda_aux * l_aux
        if alpha > 0:
            l_kd = logit_kd_loss(teacher_out["logit_dual"], student_out["logit_dual"], tau_kd)
            total = total + alpha * l_kd
            log["L_logit_KD"] = l_kd.item(); log["W_logit_KD"] = float(alpha * l_kd)
            comps["logit_kd"] = alpha * l_kd
        if beta > 0:
            if use_counterfactual_csd:
                l_csd = csd_loss(teacher_cf_out["p_dual"], teacher_cf_out["p_macula_cf"], teacher_cf_out["p_disc_cf"],
                                 student_cf_out["p_dual"], student_cf_out["p_macula_cf"], student_cf_out["p_disc_cf"],
                                 variant=csd_variant, tau_csd=tau_csd, huber_beta=huber_beta,
                                 scale=csd_scale if csd_scale is not None else globals().get("CSD_GLOBAL_SCALE_CF"))
            else:
                l_csd = csd_loss(teacher_out["p_dual"], teacher_out["p_macula"], teacher_out["p_disc"],
                                 student_out["p_dual"], student_out["p_macula"], student_out["p_disc"],
                                 variant=csd_variant, tau_csd=tau_csd, huber_beta=huber_beta,
                                 scale=csd_scale)
            total = total + beta * l_csd
            log["L_CSD"] = l_csd.item(); log["W_CSD"] = float(beta * l_csd)
            comps["csd"] = beta * l_csd
        if gamma_feat > 0 and feat_projector is not None:
            l_f = feature_kd_loss(teacher_out["z_fused"], student_out["z_fused"], feat_projector)
            total = total + gamma_feat * l_f
            log["L_feat_KD"] = l_f.item(); log["W_feat_KD"] = float(gamma_feat * l_f)
            comps["feat_kd"] = gamma_feat * l_f

    log["L_total"] = total.item()
    return total, log, comps

def component_grad_norms(components, params):
    """Per-component gradient norms. Loss VALUES alone cannot establish that a term influences
    learning -- rev2 logged values only, which is why its dead CSD term stayed invisible until the
    logs were re-read by hand after the entire run had finished."""
    out, params = {}, [p for p in params if p.requires_grad]
    for name, t in components.items():
        if t is None or not t.requires_grad: continue
        g = torch.autograd.grad(t, params, retain_graph=True, allow_unused=True)
        out[f"gnorm_{name}"] = sum(float(x.pow(2).sum()) for x in g if x is not None) ** 0.5
    if out.get("gnorm_task", 0) > 0 and "gnorm_csd" in out:
        out["gnorm_ratio_csd_over_task"] = out["gnorm_csd"] / out["gnorm_task"]
    return out

print("Losses defined.")

Losses defined.


## 11 — Complete metrics library

**QWK is the single primary metric** (DR grades are ordinal; a 4-grade error is not the same as a
1-grade error). Everything else is reported as secondary/supplementary — comprehensive coverage,
but explicitly not "everything is primary", which would read as metric fishing.

- *Ordinal*: QWK, MAE, Severe-Error Rate, Ordinal Violation Rate
- *Categorical*: Accuracy, Balanced Accuracy, macro/weighted Precision·Recall·F1
- *Per grade*: Precision, Recall (sensitivity), F1, Specificity (one-vs-rest), Support
- *Calibration*: Brier, ECE
- *Mechanism*: ShiftMAE, CosAgree, BenefitCorr, internal & external dual-view gain

In [12]:
from sklearn.metrics import (cohen_kappa_score, f1_score, precision_score, recall_score,
                             accuracy_score, balanced_accuracy_score, confusion_matrix)

def fast_qwk(y_true, y_pred, K=NUM_CLASSES):
    """Vectorized QWK -- the bootstrap calls this ~10,000x per comparison."""
    y_true = np.asarray(y_true, int); y_pred = np.asarray(y_pred, int)
    O = np.zeros((K, K)); np.add.at(O, (y_true, y_pred), 1)
    w = (np.arange(K)[:, None] - np.arange(K)[None, :]) ** 2 / (K - 1) ** 2
    ht, hp = np.bincount(y_true, minlength=K), np.bincount(y_pred, minlength=K)
    E = np.outer(ht, hp) / max(len(y_true), 1)
    den = (w * E).sum()
    return 1.0 - (w * O).sum() / den if den > 0 else 0.0

def compute_all_metrics(y_true, y_pred, p_cum=None, K=NUM_CLASSES):
    y_true = np.asarray(y_true, int); y_pred = np.asarray(y_pred, int)
    labels = list(range(K))
    m = {
        "QWK": fast_qwk(y_true, y_pred, K),
        "MAE": float(np.mean(np.abs(y_true - y_pred))),
        "SevereErrorRate": float(np.mean(np.abs(y_true - y_pred) >= 2)),
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "BalancedAccuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }
    for avg in ("macro", "weighted"):
        tag = "Macro" if avg == "macro" else "Weighted"
        m[f"{tag}Precision"] = float(precision_score(y_true, y_pred, average=avg, labels=labels, zero_division=0))
        m[f"{tag}Recall"]    = float(recall_score(y_true, y_pred, average=avg, labels=labels, zero_division=0))
        m[f"{tag}F1"]        = float(f1_score(y_true, y_pred, average=avg, labels=labels, zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    prec = precision_score(y_true, y_pred, average=None, labels=labels, zero_division=0)
    rec  = recall_score(y_true, y_pred, average=None, labels=labels, zero_division=0)
    f1   = f1_score(y_true, y_pred, average=None, labels=labels, zero_division=0)
    total = cm.sum()
    for g in labels:
        tp = cm[g, g]; fn = cm[g, :].sum() - tp; fp = cm[:, g].sum() - tp; tn = total - tp - fn - fp
        m[f"Precision_Grade{g}"]   = float(prec[g])
        m[f"Recall_Grade{g}"]      = float(rec[g])          # sensitivity
        m[f"Sensitivity_Grade{g}"] = float(rec[g])          # alias kept for continuity with rev2/rev3
        m[f"F1_Grade{g}"]          = float(f1[g])
        m[f"Specificity_Grade{g}"] = float(tn / (tn + fp)) if (tn + fp) > 0 else float("nan")
        m[f"Support_Grade{g}"]     = int(cm[g, :].sum())
        m[f"Predicted_Grade{g}"]   = int(cm[:, g].sum())
    if p_cum is not None:
        m["OrdinalViolationRate"] = ordinal_violation_rate(p_cum)
        m.update(compute_calibration(p_cum, y_true))
    return m

def compute_calibration(p_cum, y_true, n_bins=10):
    """Pooled over the K-1 cumulative thresholds. CSD is framed as distilling a shift in
    CONFIDENCE, so calibration is load-bearing for the claim (judge.md Flag 4)."""
    p = p_cum.detach().cpu().float()
    y = torch.as_tensor(np.asarray(y_true, int)).long()
    levels = torch.arange(p.shape[1]).unsqueeze(0)
    t = (y.unsqueeze(1) > levels).float()
    pf, tf = p.flatten(), t.flatten()
    brier = float(((pf - tf) ** 2).mean())
    ece, n, edges = 0.0, pf.numel(), torch.linspace(0, 1, n_bins + 1)
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        msk = (pf > lo) & (pf <= hi) if i > 0 else (pf >= lo) & (pf <= hi)
        if msk.sum() == 0: continue
        ece += float(msk.sum()) / n * abs(float(pf[msk].mean()) - float(tf[msk].mean()))
    # Named precisely: these pool the K-1 CUMULATIVE THRESHOLDS, which is not conventional
    # multiclass ECE/Brier. Per-threshold values are reported alongside.
    out = {"OrdinalThreshold_Brier": brier, "OrdinalThreshold_ECE": ece,
           "Brier": brier, "ECE": ece}   # aliases kept so existing table/figure code still resolves
    # Per-threshold diagnostics, named for what they actually are. The earlier `ECE_threshold{k}`
    # was mean|p - y| -- a mean absolute probability error, NOT an Expected Calibration Error, which
    # requires binning. Both quantities are now reported, each under its correct name.
    for k in range(p.shape[1]):
        pk, tk = p[:, k], t[:, k]
        out[f"Threshold{k}_MAEProb"] = float((pk - tk).abs().mean())
        e_k, n_k = 0.0, pk.numel()
        for i in range(n_bins):
            lo, hi = edges[i], edges[i + 1]
            msk = (pk > lo) & (pk <= hi) if i > 0 else (pk >= lo) & (pk <= hi)
            if msk.sum() == 0: continue
            e_k += float(msk.sum()) / n_k * abs(float(pk[msk].mean()) - float(tk[msk].mean()))
        out[f"Threshold{k}_ECE"] = e_k
    return out

def reliability_curve(p_cum, y_true, n_bins=10):
    p = p_cum.detach().cpu().float()
    y = torch.as_tensor(np.asarray(y_true, int)).long()
    levels = torch.arange(p.shape[1]).unsqueeze(0)
    t = (y.unsqueeze(1) > levels).float()
    pf, tf = p.flatten(), t.flatten()
    edges = torch.linspace(0, 1, n_bins + 1); rows = []
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        msk = (pf > lo) & (pf <= hi) if i > 0 else (pf >= lo) & (pf <= hi)
        rows.append({"bin_lo": float(lo), "bin_hi": float(hi), "count": int(msk.sum()),
                     "mean_predicted": float(pf[msk].mean()) if msk.sum() else float("nan"),
                     "observed_frequency": float(tf[msk].mean()) if msk.sum() else float("nan")})
    return pd.DataFrame(rows)

def check_prediction_collapse(y_true, y_pred, K=NUM_CLASSES, recall_floor=0.05):
    """rev2's core pathology only became visible through per-grade recall -- Grades 1-3 were
    essentially never predicted while overall QWK still looked plausible. This makes that failure
    mode impossible to miss again."""
    y_true = np.asarray(y_true, int); y_pred = np.asarray(y_pred, int)
    warns = []
    for g in range(K):
        if (y_pred == g).sum() == 0:
            warns.append(f"Grade {g} NEVER predicted")
        support = (y_true == g).sum()
        if support > 0:
            r = (y_pred[y_true == g] == g).mean()
            if r < recall_floor:
                warns.append(f"Grade {g} recall {r:.3f} < {recall_floor}")
    n_distinct = len(np.unique(y_pred))
    if n_distinct <= 2:
        warns.append(f"prediction distribution COLLAPSED to {n_distinct} distinct grade(s)")
    return warns

print("Metrics library defined.")

Metrics library defined.


In [13]:
@torch.no_grad()
def get_predictions(model, loader, device, view_mode="dual", return_clusters=False):
    model.eval()
    preds, targets, ps, cids = [], [], [], []
    for batch in loader:
        macula, disc = batch["macula"].to(device), batch["disc"].to(device)
        out = get_student_output(model, macula, disc, view_mode)
        p = out["p_dual" if view_mode == "dual" else "p"]
        preds.extend((p > 0.5).sum(dim=1).cpu().tolist())
        targets.extend(batch["label"].tolist())
        ps.append(p.cpu())
        if return_clusters: cids.extend(batch["cluster_id"].tolist())
    y_true, y_pred, p_cum = np.array(targets), np.array(preds), torch.cat(ps, 0)
    return (y_true, y_pred, p_cum, np.array(cids)) if return_clusters else (y_true, y_pred, p_cum)

@torch.no_grad()
def quick_val_qwk(model, loader, device, view_mode="dual"):
    y, yp, _ = get_predictions(model, loader, device, view_mode)
    return fast_qwk(y, yp)

@torch.no_grad()
def compute_dual_view_gain(model, loader, device):
    """INTERNAL gain: dual head vs this model's OWN auxiliary heads (shared, jointly-trained
    backbone). Must not be conflated with external gain -- judge.md Flag 8."""
    y, pd_, _ = get_predictions(model, loader, device, "dual")
    _, pm, _  = get_predictions(model, loader, device, "macula_only")
    _, pdd, _ = get_predictions(model, loader, device, "disc_only")
    qd, qm, qdd = fast_qwk(y, pd_), fast_qwk(y, pm), fast_qwk(y, pdd)
    return {"QWK_dual": qd, "QWK_aux_macula": qm, "QWK_aux_disc": qdd,
            "DualViewGain_G_internal": qd - max(qm, qdd)}

def compute_external_gain(qwk_dual, qwk_indep_macula, qwk_indep_disc):
    """EXTERNAL gain: vs INDEPENDENTLY trained single-view students."""
    return qwk_dual - max(qwk_indep_macula, qwk_indep_disc)

def _sample_ordinal_nll(p_cum, y, K=NUM_THRESHOLDS):
    levels = torch.arange(K, device=p_cum.device).unsqueeze(0)
    y_k = (y.unsqueeze(1) > levels).float()
    p = p_cum.clamp(1e-6, 1 - 1e-6)
    return -(y_k * torch.log(p) + (1 - y_k) * torch.log(1 - p)).sum(dim=1)

from scipy import stats as sps

@torch.no_grad()
def compute_shift_fidelity(teacher, student, loader, device):
    """Did CSD transfer the PATTERN, independently of whether QWK moved? (judge.md Flag 10)

    BenefitCorr is the strongest of the three: it correlates the teacher's and student's per-sample
    fusion benefit B_i = NLL(p_agg) - NLL(p_dual). If complementarity really transferred, the
    student should benefit from dual-view on the SAME samples the teacher does."""
    teacher.eval(); student.eval()
    smae, cos, bt, bs = [], [], [], []
    for batch in loader:
        m, d = batch["macula"].to(device), batch["disc"].to(device)
        y = batch["label"].to(device)
        to, so = teacher(m, d), student(m, d)
        dt = _delta(to["p_dual"], to["p_macula"], to["p_disc"])
        ds = _delta(so["p_dual"], so["p_macula"], so["p_disc"])
        smae.append((ds - dt).abs().sum(1).cpu())
        cos.append(F.cosine_similarity(ds, dt, dim=1, eps=1e-6).cpu())
        for out, sink in ((to, bt), (so, bs)):
            p_agg = (out["p_macula"] + out["p_disc"]) / 2
            sink.append((_sample_ordinal_nll(p_agg, y) - _sample_ordinal_nll(out["p_dual"], y)).cpu())
    smae, cos = torch.cat(smae), torch.cat(cos)
    a, b = torch.cat(bt).numpy(), torch.cat(bs).numpy()
    # Named precisely: ShiftL1 is the mean L1 NORM of the shift difference; ShiftMAE is the mean
    # absolute error PER THRESHOLD. The earlier code reported the L1 norm under the name "ShiftMAE".
    if a.std() > 1e-8 and b.std() > 1e-8:
        pear = float(sps.pearsonr(a, b)[0]); spear = float(sps.spearmanr(a, b).statistic)
    else:
        pear = spear = float("nan")
    return {"ShiftL1": float(smae.mean()), "ShiftMAE": float(smae.mean() / NUM_THRESHOLDS),
            "CosAgree": float(cos.mean()), "BenefitCorr": pear, "BenefitCorrSpearman": spear,
            "TeacherBenefitPositiveFrac": float((a > 0).mean()),
            "StudentBenefitPositiveFrac": float((b > 0).mean())}

print("Evaluation helpers defined.")

Evaluation helpers defined.


## 12 — Efficiency benchmark suite (standardized protocol)

Latency is always measured on a **CPU copy** of the model — there is no code path that can time a
CUDA model and label it CPU. Protocol is fixed and recorded with every measurement:
`batch=1, warmup=50, runs=500, threads=1`.

Size comparisons use **equivalent deployment artifacts** (FP32 export vs INT8 export), never an
FP32 training checkpoint against an INT8 state_dict.

In [14]:
try:
    import psutil; _HAS_PSUTIL = True
except Exception:
    _HAS_PSUTIL = False

def param_count(model):
    return int(sum(p.numel() for p in model.parameters()))

def file_size_mb(path):
    return os.path.getsize(path) / (1024 ** 2) if path and os.path.exists(path) else float("nan")

def benchmark_latency(model, macula, disc, view_mode="dual", bench=BENCH, on_cpu=True, label=""):
    """Returns mean/median/SD/p95/p99/throughput plus the protocol used."""
    torch.set_num_threads(bench["threads"])
    if on_cpu:
        m = copy.deepcopy(model).to("cpu").eval()
        mac, dsc = macula[:bench["batch_size"]].cpu(), disc[:bench["batch_size"]].cpu()
    else:
        m = model.eval(); mac, dsc = macula[:bench["batch_size"]], disc[:bench["batch_size"]]

    if view_mode == "dual":
        fn = lambda: m(mac, dsc)
    else:
        which = "macula" if "macula" in view_mode else "disc"
        img = mac if which == "macula" else dsc
        fn = lambda: m.forward_single(img, which=which)

    with torch.no_grad():
        for _ in range(bench["warmup"]): fn()
        ts = []
        for _ in range(bench["runs"]):
            t0 = time.perf_counter(); fn(); ts.append((time.perf_counter() - t0) * 1000)
    ts = np.array(ts)
    med = float(np.median(ts))
    return {"Latency_mean_ms": float(ts.mean()), "Latency_median_ms": med,
            "Latency_sd_ms": float(ts.std()), "Latency_p95_ms": float(np.percentile(ts, 95)),
            "Latency_p99_ms": float(np.percentile(ts, 99)),
            # One inference call processes one EYE = one (macula, disc) PAIR = TWO fundus images.
            # "img_per_s" understated the image rate by 2x and misnamed the unit that matters.
            "Throughput_pairs_per_s": float(1000.0 / med) if med > 0 else float("nan"),
            "Throughput_images_per_s": float(2000.0 / med) if med > 0 else float("nan"),
            "bench_device": "cpu" if on_cpu else "cuda", "bench_threads": bench["threads"],
            "bench_warmup": bench["warmup"], "bench_runs": bench["runs"],
            "bench_batch_size": bench["batch_size"]}

def measure_memory(build_fn, macula, disc):
    """Peak resident memory around model construction + one inference (CPU-side, deployment-relevant)."""
    if not _HAS_PSUTIL: return {"PeakRSS_MB": float("nan"), "InferenceRSSDelta_MB": float("nan")}
    proc = psutil.Process(os.getpid())
    base = proc.memory_info().rss / 1024 ** 2
    m = build_fn()
    loaded = proc.memory_info().rss / 1024 ** 2
    with torch.no_grad(): m(macula[:1].cpu(), disc[:1].cpu())
    after = proc.memory_info().rss / 1024 ** 2
    del m
    return {"ModelLoadRSS_MB": max(loaded - base, 0.0), "InferenceRSSDelta_MB": max(after - loaded, 0.0),
            "PeakRSS_MB": after}

def efficiency_derived(size_mb, ref_size_mb, latency_ms, ref_latency_ms):
    out = {}
    if ref_size_mb and size_mb and not math.isnan(size_mb) and not math.isnan(ref_size_mb) and size_mb > 0:
        out["CompressionRatio_vs_ref"] = ref_size_mb / size_mb
        out["SizeReduction_pct"] = (1 - size_mb / ref_size_mb) * 100
    if ref_latency_ms and latency_ms and latency_ms > 0:
        out["Speedup_vs_ref"] = ref_latency_ms / latency_ms
    return out

def retention_metrics(m_comp, m_ref, keys=("QWK", "Accuracy", "MacroF1", "MacroRecall")):
    """Retention for score-type metrics; for error-type metrics report the DELTA instead, since a
    ratio of errors is not interpretable in the same direction."""
    out = {}
    for k in keys:
        if k in m_comp and k in m_ref and m_ref[k] not in (0, None) and not math.isnan(m_ref[k]):
            out[f"{k}_retention_pct"] = 100.0 * m_comp[k] / m_ref[k]
    for k in ("MAE", "SevereErrorRate", "ECE", "Brier"):
        if k in m_comp and k in m_ref:
            out[f"delta_{k}"] = m_comp[k] - m_ref[k]
    return out

BENCH_EFF = BENCH_PREFLIGHT if PREFLIGHT else BENCH
CPU_INFO = platform.processor() or "unknown"
try:
    CPU_INFO = subprocess.check_output("lscpu | grep 'Model name' | head -1", shell=True, text=True).split(":")[-1].strip() or CPU_INFO
except Exception:
    pass
print("Benchmark CPU:", CPU_INFO, "| protocol:", BENCH)

Benchmark CPU: Intel(R) Xeon(R) CPU @ 2.00GHz | protocol: {'batch_size': 1, 'warmup': 50, 'runs': 500, 'threads': 1}


## 13 — FP32 smoke tests (must pass before any training)

In [15]:
from tqdm.auto import tqdm

def smoke_test():
    ds = DRTiDDualViewDataset(DRTID_TRAIN_CSV, eval_transform)
    batch = next(iter(make_loader(ds, 8, True, workers=0)))
    m, d, y = batch["macula"].to(DEVICE), batch["disc"].to(DEVICE), batch["label"].to(DEVICE)

    teacher = DualViewResNetTeacher(init_thresholds=INIT_THRESHOLDS).to(DEVICE)
    student = DualViewLightStudent(init_thresholds=INIT_THRESHOLDS).to(DEVICE)

    n_s, n_sb, n_t = param_count(student), param_count(student.backbone), param_count(teacher)
    print(f"Student {n_s:,} params (backbone {n_sb:,}) | Teacher {n_t:,} | compression {n_t/n_s:.0f}x")
    assert n_s > 150_000, f"student too small ({n_s:,}) -- rev2's 34K student was capacity-capped"

    t_out = teacher(m, d)
    ovr = ordinal_violation_rate(t_out["p_dual"])
    assert ovr == 0.0, f"CORAL monotonicity broken: OVR={ovr}"

    _ = teacher.forward_single(m, "macula")
    cf = teacher.counterfactual_forward(m, d)

    for vm in ["dual", "macula_only", "disc_only"]:
        s_out = get_student_output(student, m, d, vm)
        loss, log, comps = combined_student_loss(t_out, s_out, y, vm, alpha=0.5, beta=1.0,
                                                 pos_weight=POS_WEIGHT.to(DEVICE))
        loss.backward(); student.zero_grad()
        print(f"  [{vm}] loss={loss.item():.4f}")

    s_cf, s_dual = student.counterfactual_forward(m, d), student(m, d)
    l_cf, _, _ = combined_student_loss(t_out, s_dual, y, "dual", alpha=0.5, beta=1.0,
                                       use_counterfactual_csd=True, teacher_cf_out=cf,
                                       student_cf_out=s_cf, pos_weight=POS_WEIGHT.to(DEVICE))
    l_cf.backward(); student.zero_grad()
    print(f"  [dual + counterfactual CSD] loss={l_cf.item():.4f}")

    # Gate 4 precursor: CSD must produce a real gradient, not decoration.
    s_dual = student(m, d)
    _, _, comps = combined_student_loss(t_out, s_dual, y, "dual", alpha=0.5, beta=1.0,
                                        pos_weight=POS_WEIGHT.to(DEVICE))
    gn = component_grad_norms(comps, list(student.fusion.parameters()) + list(student.main_head.parameters()))
    student.zero_grad()
    ratio = gn.get("gnorm_ratio_csd_over_task", 0.0)
    print("  gradient norms:", {k: round(v, 4) for k, v in gn.items()})
    assert ratio > 0.01, f"CSD gradient ratio {ratio:.5f} negligible -- rev2's failure mode"
    print(f"  CSD/task gradient ratio at beta=1.0 (probe value): {ratio:.3f}")
    if ratio > 3.0:
        print("  NOTE: at beta=1.0 CSD would dominate the task loss. That is the OPPOSITE of rev2's")
        print("        failure and equally harmful, which is exactly why the Section 22 grid searches")
        print("        beta in [0.1, 0.5] and lets validation choose rather than fixing beta by hand.")

    student.eval(); student.fuse_model()
    print("  fuse_model() OK")
    print("\nSMOKE TEST PASSED.")
    return {"student_params": n_s, "teacher_params": n_t, "csd_grad_ratio": ratio}

SMOKE = smoke_test()
record_gate("Gate_SmokeTest_FP32", True,
            f"student={SMOKE['student_params']:,} teacher={SMOKE['teacher_params']:,} "
            f"csd/task grad ratio={SMOKE['csd_grad_ratio']:.3f}")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 178MB/s]


Student 329,484 params (backbone 124,736) | Teacher 40,322,124 | compression 122x


/tmp/ipykernel_1065/3973821283.py:103: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  log["L_aux"] = l_aux.item(); log["W_aux"] = float(lambda_aux * l_aux)


  [dual] loss=2.5066
  [macula_only] loss=1.0204
  [disc_only] loss=1.0217
  [dual + counterfactual CSD] loss=2.5098
  gradient norms: {'gnorm_task': 8.4412, 'gnorm_aux': 0.0, 'gnorm_logit_kd': 0.233, 'gnorm_csd': 0.2934, 'gnorm_ratio_csd_over_task': 0.0348}
  CSD/task gradient ratio at beta=1.0 (probe value): 0.035
  fuse_model() OK

SMOKE TEST PASSED.
PASS | Gate_SmokeTest_FP32 | student=329,484 teacher=40,322,124 csd/task grad ratio=0.035


True

## 14 — APTOS backbone pretraining

In [16]:
def build_backbone(kind):
    if kind == "resnet50":
        m = tv.resnet50(weights=tv.ResNet50_Weights.IMAGENET1K_V2); m.fc = nn.Identity(); return m, 2048
    if kind == "lightweight":
        m = LightweightBackbone(); return m, m.out_dim
    raise ValueError(kind)

# APTOS has its own label distribution, so its ordinal head must start from APTOS marginals --
# initializing it with DRTiD thresholds would place the pretraining head at the wrong operating point.
APTOS_INIT_THRESHOLDS = compute_init_thresholds(f"{APTOS_ROOT}/train_1.csv", grade_col="diagnosis")
print("APTOS init thresholds:", [round(t, 4) for t in APTOS_INIT_THRESHOLDS])
print("DRTiD init thresholds:", [round(t, 4) for t in INIT_THRESHOLDS])

def pretrain_backbone(kind, epochs, lr, batch_size, seed=PRIMARY_SEED, force=False):
    out = f"{CKPT_DIR}/pretrained_backbones/aptos_{kind}_backbone.pt"
    epochs = 1 if PREFLIGHT else epochs          # rehearsal: prove the path runs, not convergence
    set_seed(seed)                      # BEFORE construction so random init is actually controlled
    backbone, feat_dim = build_backbone(kind)
    if not force and checkpoint_is_compatible(out, backbone, unwrap_key=None):
        print(f"{kind}: compatible checkpoint exists, skipping."); return out
    pw = compute_pos_weights(f"{APTOS_ROOT}/train_1.csv", grade_col="diagnosis").to(DEVICE)
    tl = make_loader(APTOSSingleViewDataset(f"{APTOS_ROOT}/train_1.csv", f"{APTOS_ROOT}/train_images/train_images",
                                            aptos_train_transform), batch_size, True, seed)
    vl = make_loader(APTOSSingleViewDataset(f"{APTOS_ROOT}/valid.csv", f"{APTOS_ROOT}/val_images/val_images",
                                            aptos_eval_transform), batch_size, False)
    head = CORALHead(feat_dim, NUM_CLASSES, APTOS_INIT_THRESHOLDS)
    backbone, head = backbone.to(DEVICE), head.to(DEVICE)
    opt = torch.optim.AdamW(list(backbone.parameters()) + list(head.parameters()), lr=lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=lr * 0.02)

    best, hist = -1.0, []
    for ep in range(epochs):
        backbone.train(); head.train()
        for b in tqdm(tl, desc=f"[pretrain-{kind}] ep{ep}", leave=False):
            img, y = b["image"].to(DEVICE), b["label"].to(DEVICE)
            loss = coral_loss(head(backbone(img))[0], y, pos_weight=pw)
            opt.zero_grad(); loss.backward(); opt.step()
        sch.step()
        backbone.eval(); head.eval()
        pr, tg = [], []
        with torch.no_grad():
            for b in vl:
                _, p = head(backbone(b["image"].to(DEVICE)))
                pr.extend((p > 0.5).sum(1).cpu().tolist()); tg.extend(b["label"].tolist())
        q = fast_qwk(tg, pr); hist.append({"epoch": ep, "val_qwk": q})
        print(f"  ep{ep}: val_QWK={q:.4f}")
        if q > best:
            best = q; robust_torch_save(backbone.state_dict(), out)
    pd.DataFrame(hist).to_csv(f"{LOGS_DIR}/pretrain_{kind}_history.csv", index=False)
    assert best > 0.0, f"pretrain {kind} QWK<=0 -- worse than majority baseline"
    print(f"{kind} pretrain done. best val QWK={best:.4f}")
    return out

RESNET50_BACKBONE_CKPT   = pretrain_backbone("resnet50",   epochs=20, lr=1e-4, batch_size=32)
LIGHTWEIGHT_BACKBONE_CKPT = pretrain_backbone("lightweight", epochs=30, lr=1e-3, batch_size=32)

APTOS init thresholds: [0.0423, -0.3714, -1.8797, -2.4442]
DRTiD init thresholds: [0.07, -0.2921, -1.6034, -3.2452]


[pretrain-resnet50] ep0:   0%|          | 0/92 [00:00<?, ?it/s]

  ep0: val_QWK=0.8276
resnet50 pretrain done. best val QWK=0.8276


[pretrain-lightweight] ep0:   0%|          | 0/92 [00:00<?, ?it/s]

  ep0: val_QWK=0.7329
lightweight pretrain done. best val QWK=0.7329


## 15 — Teacher training & Gate 2

In [18]:
def train_teacher(freeze_epochs=5, finetune_epochs=20, patience=8, lambda_aux=0.3,
                  seed=PRIMARY_SEED, batch_size=16, freeze_lr=3e-4, finetune_lr=1e-5, force=False):
    out = f"{CKPT_DIR}/teacher/teacher_final.pt"
    if PREFLIGHT:
        freeze_epochs, finetune_epochs, patience = 1, 1, 1
    set_seed(seed)                      # BEFORE construction
    model = DualViewResNetTeacher(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS)
    if not force and checkpoint_is_compatible(out, model, "model_state"):
        print("Teacher checkpoint compatible, skipping."); return out
    model = model.to(DEVICE)
    pw = POS_WEIGHT.to(DEVICE)
    tl = make_loader(DRTiDDualViewDataset(DRTID_TRAIN_CSV, paired_transform=paired_train_transform), batch_size, True, seed)
    vl = make_loader(DRTiDDualViewDataset(DRTID_VAL_CSV, eval_transform), batch_size, False)
    model.backbone.load_state_dict(robust_torch_load(RESNET50_BACKBONE_CKPT, map_location=DEVICE))

    best, hist, holder = -1.0, [], {"state": None}
    def run(epochs, lr, best):
        opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs, 1), eta_min=lr * 0.02)
        bad = 0
        for _ in range(epochs):
            ge = len(hist); model.train()
            for b in tqdm(tl, desc=f"[teacher] ep{ge}", leave=False):
                m, d, y = b["macula"].to(DEVICE), b["disc"].to(DEVICE), b["label"].to(DEVICE)
                o = model(m, d)
                loss = coral_loss(o["logit_dual"], y, pos_weight=pw) + lambda_aux * aux_loss(o, y, pos_weight=pw)
                opt.zero_grad(); loss.backward(); opt.step()
            sch.step()
            q = quick_val_qwk(model, vl, DEVICE, "dual")
            hist.append({"epoch": ge, "val_qwk": q, "lr": sch.get_last_lr()[0]})
            print(f"  ep{ge}: val_QWK={q:.4f}")
            if q > best:
                best, bad = q, 0
                holder["state"] = copy.deepcopy(model.state_dict())
                robust_torch_save({"model_state": holder["state"], "epoch": ge, "val_qwk": q}, out)
            else:
                bad += 1
                if bad >= patience: print(f"  early stop @ep{ge}"); break
        return best

    for p in model.backbone.parameters(): p.requires_grad = False
    best = run(freeze_epochs, freeze_lr, best)
    # Reload best freeze-phase weights from MEMORY, not Drive -- a write-then-immediate-read of the
    # same path can hit Drive's FUSE sync lag and raise FileNotFoundError mid-run.
    if holder["state"] is not None: model.load_state_dict(holder["state"])
    for p in model.backbone.parameters(): p.requires_grad = True
    best = run(finetune_epochs, finetune_lr, best)

    pd.DataFrame(hist).to_csv(f"{LOGS_DIR}/teacher_history.csv", index=False)
    print(f"Teacher done. best val QWK={best:.4f}")
    return out

TEACHER_CKPT = train_teacher()

_t = DualViewResNetTeacher(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS).to(DEVICE)
_t.load_state_dict(robust_torch_load(TEACHER_CKPT, map_location=DEVICE)["model_state"])
_gain = compute_dual_view_gain(_t, VAL_LOADER, DEVICE)
GATE2_PASSED = _gain["QWK_dual"] > max(_gain["QWK_aux_macula"], _gain["QWK_aux_disc"])
print("Gate 2 (validation):", {k: round(v, 4) for k, v in _gain.items()})
if not GATE2_PASSED:
    print("*** Teacher shows no dual-view advantage. CSD's Delta is only meaningful if it does.  ***")
    print("*** Levers: lower lambda_aux, lower freeze_lr, more epochs, or a different seed.      ***")
# BLOCKING in the real run: every CSD result downstream is uninterpretable without this.
# Non-blocking during PREFLIGHT, where the teacher is deliberately undertrained.
record_gate("Gate2_Teacher", GATE2_PASSED,
            f"QWK_dual={_gain['QWK_dual']:.4f} vs max(aux)={max(_gain['QWK_aux_macula'], _gain['QWK_aux_disc']):.4f} "
            f"G_internal={_gain['DualViewGain_G_internal']:+.4f}",
            blocking=not PREFLIGHT)

# ---- Fixed global CSD normalization, from the FROZEN teacher over TRAIN (P0-6) ----
_scale_loader = make_loader(DRTiDDualViewDataset(DRTID_TRAIN_CSV, eval_transform), 16, False)
CSD_GLOBAL_SCALE    = compute_global_delta_scale(_t, _scale_loader, DEVICE, counterfactual=False)
CSD_GLOBAL_SCALE_CF = compute_global_delta_scale(_t, _scale_loader, DEVICE, counterfactual=True)
print(f"CSD global scale  E_train[|delta_T|]        = {CSD_GLOBAL_SCALE:.6f}")
print(f"CSD global scale (counterfactual delta)     = {CSD_GLOBAL_SCALE_CF:.6f}")
save_json({"csd_global_scale": CSD_GLOBAL_SCALE, "csd_global_scale_cf": CSD_GLOBAL_SCALE_CF},
          f"{CONFIG_DIR}/csd_global_scale.json")
del _t, _scale_loader

[teacher] ep0:   0%|          | 0/50 [00:00<?, ?it/s]

  ep0: val_QWK=0.5068


[teacher] ep1:   0%|          | 0/50 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^Exception ignored in: ^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>^
^Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
^^    self._shutdown_workers()^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
^    ^if w.is_alive():^
^ ^ ^ ^ ^ ^ 

  ep1: val_QWK=0.5514
Teacher done. best val QWK=0.5514
Gate 2 (validation): {'QWK_dual': np.float64(0.5514), 'QWK_aux_macula': np.float64(0.3848), 'QWK_aux_disc': np.float64(0.3103), 'DualViewGain_G_internal': np.float64(0.1666)}
PASS | Gate2_Teacher | QWK_dual=0.5514 vs max(aux)=0.3848 G_internal=+0.1666
CSD global scale  E_train[|delta_T|]        = 0.124327
CSD global scale (counterfactual delta)     = 0.061041


## 16 — Generic student trainer

One function drives every student condition, so training logic cannot drift between conditions.
Logs per-epoch loss values, **weighted contributions**, and **per-component gradient norms** to
`gradient_contributions_<condition>_<seed>.csv`.

In [19]:
_teacher_cache = None
def get_teacher():
    global _teacher_cache
    if _teacher_cache is None:
        t = DualViewResNetTeacher(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS).to(DEVICE)
        t.load_state_dict(robust_torch_load(TEACHER_CKPT, map_location=DEVICE)["model_state"])
        t.eval()
        for p in t.parameters(): p.requires_grad = False
        _teacher_cache = t
    return _teacher_cache

def train_student_condition(run_name, seed, view_mode, alpha=0.0, beta=0.0, lambda_aux=0.5,
                            csd_variant="smoothl1_norm", tau_kd=2.0, tau_csd=0.5,
                            use_counterfactual_csd=False, epochs=40, patience=8, lr=1e-3,
                            batch_size=16, gamma_feat=0.0, huber_beta=1.0, weight_decay=1e-4,
                            force=False):
    if PREFLIGHT:
        # 2 epochs (not 1) so best-epoch selection and the patience path are still exercised.
        epochs, patience = 2, 2
    ck_dir = f"{CKPT_DIR}/student/{run_name}"; os.makedirs(ck_dir, exist_ok=True)
    out = f"{ck_dir}/best_seed{seed}.pt"
    set_seed(seed)                      # BEFORE construction
    probe = DualViewLightStudent(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS)
    if not force and checkpoint_is_compatible(out, probe, "model_state"):
        print(f"{run_name}|seed{seed}: compatible checkpoint exists, skipping.")
        return out, None
    del probe

    cfg = dict(run_name=run_name, seed=seed, view_mode=view_mode, alpha=alpha, beta=beta,
               lambda_aux=lambda_aux, csd_variant=csd_variant, tau_kd=tau_kd, tau_csd=tau_csd,
               use_counterfactual_csd=use_counterfactual_csd, epochs=epochs, patience=patience,
               lr=lr, batch_size=batch_size, gamma_feat=gamma_feat, huber_beta=huber_beta,
               weight_decay=weight_decay, pos_weight_mode=POS_WEIGHT_MODE)
    save_json(cfg, f"{CONFIG_DIR}/{run_name}_seed{seed}.json")

    pw = POS_WEIGHT.to(DEVICE)
    tl = make_loader(DRTiDDualViewDataset(DRTID_TRAIN_CSV, paired_transform=paired_train_transform), batch_size, True, seed)
    vl = make_loader(DRTiDDualViewDataset(DRTID_VAL_CSV, eval_transform), batch_size, False)

    teacher = get_teacher()
    student = DualViewLightStudent(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS).to(DEVICE)
    student.backbone.load_state_dict(robust_torch_load(LIGHTWEIGHT_BACKBONE_CKPT, map_location=DEVICE))

    feat_proj, trainable = None, list(student.parameters())
    if gamma_feat > 0:
        feat_proj = nn.Linear(student.fusion.out_dim, teacher.fusion.out_dim).to(DEVICE)
        trainable += list(feat_proj.parameters())

    opt = torch.optim.AdamW(trainable, lr=lr, weight_decay=weight_decay)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=lr * 0.02)
    probe_params = list(student.fusion.parameters()) + list(student.main_head.parameters())
    best, bad, hist = -1.0, 0, []

    for ep in range(epochs):
        student.train(); logs, gnorms = [], {}
        for bi, b in enumerate(tqdm(tl, desc=f"[{run_name}|s{seed}] ep{ep}", leave=False)):
            m, d, y = b["macula"].to(DEVICE), b["disc"].to(DEVICE), b["label"].to(DEVICE)
            with torch.no_grad():
                t_out = teacher(m, d)
                t_cf = teacher.counterfactual_forward(m, d) if use_counterfactual_csd else None
            s_out = get_student_output(student, m, d, view_mode)
            s_cf = student.counterfactual_forward(m, d) if (use_counterfactual_csd and view_mode == "dual") else None

            loss, log, comps = combined_student_loss(
                t_out, s_out, y, view_mode, alpha=alpha, beta=beta, lambda_aux=lambda_aux,
                tau_kd=tau_kd, csd_variant=csd_variant, tau_csd=tau_csd, pos_weight=pw,
                use_counterfactual_csd=use_counterfactual_csd, teacher_cf_out=t_cf,
                student_cf_out=s_cf, gamma_feat=gamma_feat, feat_projector=feat_proj,
                huber_beta=huber_beta)
            if bi == 0 and view_mode == "dual":
                gnorms = component_grad_norms(comps, probe_params); student.zero_grad(set_to_none=True)
            opt.zero_grad(); loss.backward(); opt.step()
            logs.append(log)
        sch.step()

        q = quick_val_qwk(student, vl, DEVICE, view_mode)
        row = {k: float(np.mean([l[k] for l in logs if k in l])) for k in logs[0]}
        row.update({"epoch": ep, "val_qwk": q, "lr": sch.get_last_lr()[0], **gnorms})
        hist.append(row)
        ls = {k: round(v, 4) for k, v in row.items() if k.startswith("L_")}
        gs = {k.replace("gnorm_", ""): round(v, 3) for k, v in gnorms.items()}
        print(f"  ep{ep}: val_QWK={q:.4f} losses={ls}" + (f" grad={gs}" if gs else ""))

        if q > best:
            best, bad = q, 0
            robust_torch_save({"model_state": student.state_dict(), "epoch": ep, "val_qwk": q,
                               "seed": seed, "config": cfg}, out)
        else:
            bad += 1
            if bad >= patience: print(f"  early stop @ep{ep}"); break

    hdf = pd.DataFrame(hist)
    hdf.to_csv(f"{LOGS_DIR}/{run_name}_seed{seed}_history.csv", index=False)
    gcols = [c for c in hdf.columns if c.startswith("gnorm_") or c.startswith("W_") or c.startswith("L_")]
    if gcols:
        hdf[["epoch"] + gcols].to_csv(f"{LOGS_DIR}/gradient_contributions_{run_name}_{seed}.csv", index=False)
    print(f"[{run_name}|s{seed}] best val QWK={best:.4f}")
    return out, hist

print("Student trainer defined.")

Student trainer defined.


## 17–20 — Baselines with matched hyperparameter budgets

**Hyperparameter fairness.** Every distillation method gets its own small, pre-registered validation
grid on the inferential seed, then the winner is run across all core seeds. Giving CSD a grid while
fixing `alpha=0.5, tau=2` for logit-KD and `gamma=1.0` for feature-KD would make any CSD win
attributable to tuning budget rather than to the method — the first thing a reviewer would ask.

Grids are declared here, before any result is seen, and selection uses validation QWK only.

In [20]:
# Single-view baselines. Also the reference for EXTERNAL dual-view gain, since they are the only
# models that never see two views.
for s_ in SEEDS_BASELINE:
    train_student_condition("macula_only", seed=s_, view_mode="macula_only")
    train_student_condition("disc_only",   seed=s_, view_mode="disc_only")

[macula_only|s42] ep0:   0%|          | 0/50 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process


  ep0: val_QWK=0.0941 losses={'L_task': 0.7076, 'L_total': 0.7076}


[macula_only|s42] ep1:   0%|          | 0/50 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

...

[disc_only|s42] ep0:   0%|          | 0/50 [00:00<?, ?it/s]

  ep0: val_QWK=0.0448 losses={'L_task': 0.7052, 'L_total': 0.7052}


[disc_only|s42] ep1:   0%|          | 0/50 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^Exception ignored in: ^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>^
^^Traceback (most recent call last):
^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
^^    ^self._shutdown_workers()^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
^^    ^if w.is_alive():^
^ ^ ^ ^ ^ ^^  ^

  ep1: val_QWK=-0.0444 losses={'L_task': 0.6938, 'L_total': 0.6938}
[disc_only|s42] best val QWK=0.0448


In [21]:
for s_ in SEEDS_CORE:
    train_student_condition("dual_no_distill", seed=s_, view_mode="dual", alpha=0.0, beta=0.0, lambda_aux=0.5)

[dual_no_distill|s42] ep0:   0%|          | 0/50 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

  ep0: val_QWK=0.0394 losses={'L_task': 0.7759, 'L_aux': 1.4143, 'L_total': 1.483} grad={'task': 6.472, 'aux': 0.0}


[dual_no_distill|s42] ep1:   0%|          | 0/50 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^Exception ignored in: ^^^
<function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>AssertionError
: Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
can only test a child process
    Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>
self._shutdown_workers(

  ep1: val_QWK=0.0215 losses={'L_task': 0.6898, 'L_aux': 1.3795, 'L_total': 1.3795} grad={'task': 2.621, 'aux': 0.0}
[dual_no_distill|s42] best val QWK=0.0394


In [22]:
# ---- Pre-registered grids (fixed before any result is inspected) ----
GRID_LOGITKD  = [{"alpha": 0.25, "tau_kd": 2.0}, {"alpha": 0.5, "tau_kd": 2.0},
                 {"alpha": 0.5,  "tau_kd": 4.0}, {"alpha": 1.0, "tau_kd": 2.0}]
GRID_FEATKD   = [{"gamma_feat": 0.1}, {"gamma_feat": 0.5}, {"gamma_feat": 1.0}, {"gamma_feat": 2.0}]
GRID_CSD      = [{"csd_variant": "smoothl1_norm", "alpha": 0.5,  "beta": 0.1},
                 {"csd_variant": "smoothl1_norm", "alpha": 0.5,  "beta": 0.2},
                 {"csd_variant": "smoothl1_norm", "alpha": 0.5,  "beta": 0.5},
                 {"csd_variant": "smoothl1_norm", "alpha": 0.25, "beta": 0.2},
                 {"csd_variant": "magnitude_weighted_direction", "alpha": 0.5, "beta": 0.2}]
if PREFLIGHT:
    GRID_LOGITKD, GRID_FEATKD, GRID_CSD = GRID_LOGITKD[:2], GRID_FEATKD[:2], GRID_CSD[:2]
save_json({"logitkd": GRID_LOGITKD, "featkd": GRID_FEATKD, "csd": GRID_CSD},
          f"{CONFIG_DIR}/preregistered_grids.json")

def run_grid(tag, grid, base_kwargs):
    """Trains each config on the inferential seed and selects on VALIDATION only."""
    rows = []
    for combo in grid:
        name = tag + "_" + "_".join(f"{k}{v}" for k, v in sorted(combo.items()))
        ck, _ = train_student_condition(name, seed=INFERENTIAL_SEED, view_mode="dual", **{**base_kwargs, **combo})
        st = robust_torch_load(ck, map_location=DEVICE)
        mdl = DualViewLightStudent(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS).to(DEVICE)
        mdl.load_state_dict(st["model_state"])
        y, yp, pc = get_predictions(mdl, VAL_LOADER, DEVICE, "dual")
        m = compute_all_metrics(y, yp, pc)
        rows.append({**combo, "run_name": name, "val_QWK": st["val_qwk"], "val_MacroF1": m["MacroF1"],
                     "val_SevereErrorRate": m["SevereErrorRate"], "val_MAE": m["MAE"]})
        del mdl
    df = pd.DataFrame(rows).sort_values(["val_QWK", "val_MacroF1"], ascending=[False, False]).reset_index(drop=True)
    df.to_csv(f"{TABLES_DIR}/table_01_grid_{tag}.csv", index=False)
    print(f"--- {tag} grid (validation only) ---"); print(df.to_string(index=False))
    return df

GRID_DF_LOGITKD = run_grid("grid_logitkd", GRID_LOGITKD, {"beta": 0.0, "lambda_aux": 0.5})
BEST_KD = {"alpha": float(GRID_DF_LOGITKD.iloc[0]["alpha"]), "tau_kd": float(GRID_DF_LOGITKD.iloc[0]["tau_kd"])}
print("selected logit-KD:", BEST_KD)

[grid_logitkd_alpha0.25_tau_kd2.0|s42] ep0:   0%|          | 0/50 [00:00<?, ?it/s]

  ep0: val_QWK=0.0570 losses={'L_task': 0.7711, 'L_aux': 1.4146, 'L_logit_KD': 0.659, 'L_total': 1.6432} grad={'task': 6.472, 'aux': 0.0, 'logit_kd': 0.517}


[grid_logitkd_alpha0.25_tau_kd2.0|s42] ep1:   0%|          | 0/50 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

  ep1: val_QWK=0.0106 losses={'L_task': 0.6884, 'L_aux': 1.3767, 'L_logit_KD': 0.6312, 'L_total': 1.5346} grad={'task': 2.69, 'aux': 0.0, 'logit_kd': 0.03}
[grid_logitkd_alpha0.25_tau_kd2.0|s42] best val QWK=0.0570


[grid_logitkd_alpha0.5_tau_kd2.0|s42] ep0:   0%|          | 0/50 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
Exception ignored in:   <function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>
  Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
     self._shutdown_workers() 
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
^    ^if w.is_alive():^
 ^ ^ ^^^ ^^   ^^^
^^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^    ^assert self._parent_pid == os.getpid(), 'can only test a child process'^^
^   ^^^ 
   File "/usr/lib/p

  ep0: val_QWK=0.0118 losses={'L_task': 0.7653, 'L_aux': 1.4147, 'L_logit_KD': 0.6572, 'L_total': 1.8013} grad={'task': 6.472, 'aux': 0.0, 'logit_kd': 1.034}


[grid_logitkd_alpha0.5_tau_kd2.0|s42] ep1:   0%|          | 0/50 [00:00<?, ?it/s]

  ep1: val_QWK=-0.0021 losses={'L_task': 0.6889, 'L_aux': 1.3781, 'L_logit_KD': 0.6308, 'L_total': 1.6934} grad={'task': 2.742, 'aux': 0.0, 'logit_kd': 0.058}
[grid_logitkd_alpha0.5_tau_kd2.0|s42] best val QWK=0.0118
--- grid_logitkd grid (validation only) ---
 alpha  tau_kd                         run_name  val_QWK  val_MacroF1  val_SevereErrorRate  val_MAE
  0.25     2.0 grid_logitkd_alpha0.25_tau_kd2.0 0.056958     0.201459                 0.41    1.140
  0.50     2.0  grid_logitkd_alpha0.5_tau_kd2.0 0.011808     0.178368                 0.42    1.195
selected logit-KD: {'alpha': 0.25, 'tau_kd': 2.0}


In [23]:
for s_ in SEEDS_CORE:
    train_student_condition("dual_logitkd", seed=s_, view_mode="dual", beta=0.0, lambda_aux=0.5, **BEST_KD)

[dual_logitkd|s42] ep0:   0%|          | 0/50 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^Exception ignored in: ^^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>
^^Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
^^^    ^self._shutdown_workers()

AssertionError  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
:     c

  ep0: val_QWK=0.0331 losses={'L_task': 0.7714, 'L_aux': 1.4142, 'L_logit_KD': 0.659, 'L_total': 1.6432} grad={'task': 6.472, 'aux': 0.0, 'logit_kd': 0.517}


[dual_logitkd|s42] ep1:   0%|          | 0/50 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: 
can only test a child processException ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

  ep1: val_QWK=-0.0088 losses={'L_task': 0.688, 'L_aux': 1.3764, 'L_logit_KD': 0.6311, 'L_total': 1.534} grad={'task': 2.594, 'aux': 0.0, 'logit_kd': 0.03}
[dual_logitkd|s42] best val QWK=0.0331


In [24]:
GRID_DF_FEATKD = run_grid("grid_featkd", GRID_FEATKD, {"beta": 0.0, "lambda_aux": 0.5, **BEST_KD})
BEST_FEAT = {"gamma_feat": float(GRID_DF_FEATKD.iloc[0]["gamma_feat"])}
print("selected feature-KD:", BEST_FEAT)

for s_ in SEEDS_CORE:
    train_student_condition("dual_featkd", seed=s_, view_mode="dual", beta=0.0, lambda_aux=0.5,
                            **BEST_KD, **BEST_FEAT)

[grid_featkd_gamma_feat0.1|s42] ep0:   0%|          | 0/50 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    Exception ignored in: self._shutdown_workers()<function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
Traceback (most recent call last):
      File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
if w.is_alive():    self._shutdown_workers()

   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
      if w.is_alive(): 
      ^ ^ ^ ^^ ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^    ^assert self._parent_pid == os.getpid(), 'can only test a child process'^
^ ^ ^
    File "/usr/lib/py

  ep0: val_QWK=0.0212 losses={'L_task': 0.7687, 'L_aux': 1.4156, 'L_logit_KD': 0.6576, 'L_feat_KD': 0.426, 'L_total': 1.6835} grad={'task': 6.472, 'aux': 0.0, 'logit_kd': 0.517, 'feat_kd': 0.088}


[grid_featkd_gamma_feat0.1|s42] ep1:   0%|          | 0/50 [00:00<?, ?it/s]

  ep1: val_QWK=0.0045 losses={'L_task': 0.6894, 'L_aux': 1.3808, 'L_logit_KD': 0.6306, 'L_feat_KD': 0.2259, 'L_total': 1.5601} grad={'task': 2.733, 'aux': 0.0, 'logit_kd': 0.033, 'feat_kd': 0.006}
[grid_featkd_gamma_feat0.1|s42] best val QWK=0.0212


...

[dual_featkd|s42] ep0:   0%|          | 0/50 [00:00<?, ?it/s]

  ep0: val_QWK=0.0513 losses={'L_task': 0.7654, 'L_aux': 1.4158, 'L_logit_KD': 0.6573, 'L_feat_KD': 0.4074, 'L_total': 1.8413} grad={'task': 6.472, 'aux': 0.0, 'logit_kd': 0.517, 'feat_kd': 0.441}


[dual_featkd|s42] ep1:   0%|          | 0/50 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

  ep1: val_QWK=0.0116 losses={'L_task': 0.6892, 'L_aux': 1.3803, 'L_logit_KD': 0.6301, 'L_feat_KD': 0.2205, 'L_total': 1.6471} grad={'task': 2.97, 'aux': 0.0, 'logit_kd': 0.031, 'feat_kd': 0.032}
[dual_featkd|s42] best val QWK=0.0513


## 21–23 — Gate 4 (CSD signal), grid search, final CSD training

In [25]:
@torch.no_grad()
def gate4_csd_signal(teacher, loader, device):
    """Is there a non-trivial complementarity shift to distil at all?"""
    teacher.eval(); norms = []
    for b in loader:
        o = teacher(b["macula"].to(device), b["disc"].to(device))
        norms.append(_delta(o["p_dual"], o["p_macula"], o["p_disc"]).abs().sum(1).cpu())
    n = torch.cat(norms)
    stats = {"mean_L1": float(n.mean()), "median_L1": float(n.median()),
             "q25": float(n.quantile(.25)), "q75": float(n.quantile(.75)),
             "frac_gt_0.02": float((n > 0.02).float().mean())}
    pd.DataFrame({"delta_L1_norm": n.numpy()}).to_csv(f"{METRICS_DIR}/gate4_teacher_delta_distribution.csv", index=False)
    ok = stats["mean_L1"] >= 1e-3
    record_gate("Gate4_CSD_Signal", ok, ", ".join(f"{k}={v:.4f}" for k, v in stats.items()),
                blocking=not PREFLIGHT)
    return stats

DELTA_STATS = gate4_csd_signal(get_teacher(), VAL_LOADER, DEVICE)

PASS | Gate4_CSD_Signal | mean_L1=0.4227, median_L1=0.3235, q25=0.1460, q75=0.5724, frac_gt_0.02=0.9900


In [26]:
# CSD gets exactly the same treatment as the other methods: pre-registered grid, inferential seed,
# validation-only selection. Beta range is set from the MEASURED gradient ratio (beta=1.0 puts CSD
# several times above the task gradient), so the grid brackets subordinate-to-balanced.
GRID_DF_CSD = run_grid("grid_csd", GRID_CSD, {"lambda_aux": 0.5})
BEST_CSD_VARIANT = GRID_DF_CSD.iloc[0]["csd_variant"]
BEST_ALPHA       = float(GRID_DF_CSD.iloc[0]["alpha"])
BEST_BETA        = float(GRID_DF_CSD.iloc[0]["beta"])
grid_df = GRID_DF_CSD
print(f"selected CSD: variant={BEST_CSD_VARIANT} alpha={BEST_ALPHA} beta={BEST_BETA}")
save_json({"csd_variant": BEST_CSD_VARIANT, "alpha": BEST_ALPHA, "beta": BEST_BETA,
           "logitkd": BEST_KD, "featkd": BEST_FEAT}, f"{CONFIG_DIR}/selected_hyperparameters.json")

[grid_csd_alpha0.5_beta0.1_csd_variantsmoothl1_norm|s42] ep0:   0%|          | 0/50 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

  ep0: val_QWK=0.0231 losses={'L_task': 0.7739, 'L_aux': 1.4178, 'L_logit_KD': 0.659, 'L_CSD': 0.8428, 'L_total': 1.8966} grad={'task': 6.472, 'aux': 0.0, 'logit_kd': 1.034, 'csd': 2.665, 'ratio_csd_over_task': 0.412}


[grid_csd_alpha0.5_beta0.1_csd_variantsmoothl1_norm|s42] ep1:   0%|          | 0/50 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

...

[grid_csd_alpha0.5_beta0.2_csd_variantsmoothl1_norm|s42] ep0:   0%|          | 0/50 [00:00<?, ?it/s]

  ep0: val_QWK=0.0301 losses={'L_task': 0.778, 'L_aux': 1.4254, 'L_logit_KD': 0.6609, 'L_CSD': 0.8281, 'L_total': 1.9868} grad={'task': 6.472, 'aux': 0.0, 'logit_kd': 1.034, 'csd': 5.329, 'ratio_csd_over_task': 0.823}


[grid_csd_alpha0.5_beta0.2_csd_variantsmoothl1_norm|s42] ep1:   0%|          | 0/50 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

  ep1: val_QWK=0.0441 losses={'L_task': 0.6913, 'L_aux': 1.3822, 'L_logit_KD': 0.63, 'L_CSD': 0.5982, 'L_total': 1.817} grad={'task': 3.001, 'aux': 0.0, 'logit_kd': 0.053, 'csd': 0.321, 'ratio_csd_over_task': 0.107}
[grid_csd_alpha0.5_beta0.2_csd_variantsmoothl1_norm|s42] best val QWK=0.0441
--- grid_csd grid (validation only) ---
  csd_variant  alpha  beta                                           run_name  val_QWK  val_MacroF1  val_SevereErrorRate  val_MAE
smoothl1_norm    0.5   0.2 grid_csd_alpha0.5_beta0.2_csd_variantsmoothl1_norm 0.044110     0.115285                0.295    1.145
smoothl1_norm    0.5   0.1 grid_csd_alpha0.5_beta0.1_csd_variantsmoothl1_norm 0.023111     0.190921                0.400    1.150
selected CSD: variant=smoothl1_norm alpha=0.5 beta=0.2


In [27]:
for s_ in SEEDS_CORE:
    train_student_condition("dual_csd", seed=s_, view_mode="dual", alpha=BEST_ALPHA, beta=BEST_BETA,
                            lambda_aux=0.5, csd_variant=BEST_CSD_VARIANT,
                            tau_kd=BEST_KD["tau_kd"], tau_csd=0.5)

[dual_csd|s42] ep0:   0%|          | 0/50 [00:00<?, ?it/s]

  ep0: val_QWK=0.0266 losses={'L_task': 0.7786, 'L_aux': 1.4262, 'L_logit_KD': 0.6611, 'L_CSD': 0.8301, 'L_total': 1.9882} grad={'task': 6.472, 'aux': 0.0, 'logit_kd': 1.034, 'csd': 5.329, 'ratio_csd_over_task': 0.823}


[dual_csd|s42] ep1:   0%|          | 0/50 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

  ep1: val_QWK=0.0361 losses={'L_task': 0.6924, 'L_aux': 1.3837, 'L_logit_KD': 0.6301, 'L_CSD': 0.5969, 'L_total': 1.8187} grad={'task': 3.01, 'aux': 0.0, 'logit_kd': 0.052, 'csd': 0.3, 'ratio_csd_over_task': 0.1}
[dual_csd|s42] best val QWK=0.0361


In [28]:
# ---- CSD ablations (spec 13): formulation controls, at the selected alpha/beta ----
for s_ in SEEDS_BASELINE:
    train_student_condition("abl_csd_raw_smoothl1", seed=s_, view_mode="dual", alpha=BEST_ALPHA,
                            beta=0.7, lambda_aux=0.5, csd_variant="smoothl1")          # rev2 formulation
for s_ in SEEDS_BASELINE:
    train_student_condition("abl_csd_kl_softmax", seed=s_, view_mode="dual", alpha=BEST_ALPHA,
                            beta=0.5, lambda_aux=0.5, csd_variant="kl_softmax")        # v1 negative control
train_student_condition("abl_csd_counterfactual", seed=PRIMARY_SEED, view_mode="dual",
                        alpha=BEST_ALPHA, beta=BEST_BETA, lambda_aux=0.5,
                        csd_variant=BEST_CSD_VARIANT, use_counterfactual_csd=True)     # judge.md Flag 1/3

[abl_csd_raw_smoothl1|s42] ep0:   0%|          | 0/50 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

  ep0: val_QWK=0.0288 losses={'L_task': 0.7663, 'L_aux': 1.4126, 'L_logit_KD': 0.6572, 'L_CSD': 0.0232, 'L_total': 1.8174} grad={'task': 6.472, 'aux': 0.0, 'logit_kd': 1.034, 'csd': 0.863, 'ratio_csd_over_task': 0.133}


[abl_csd_raw_smoothl1|s42] ep1:   0%|          | 0/50 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^Exception ignored in: ^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>^
Traceback (most recent call last):
^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
^^^^^^    ^^^self._shutdown_workers()^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
^^    ^^if w.is_alive():^
^ ^ ^

...

  ep0: val_QWK=-0.0080 losses={'L_task': 0.8065, 'L_aux': 1.4269, 'L_logit_KD': 0.6709, 'L_CSD': 1.3736, 'L_total': 2.1301} grad={'task': 6.472, 'aux': 0.0, 'logit_kd': 1.034, 'csd': 14.893, 'ratio_csd_over_task': 2.301}


[abl_csd_counterfactual|s42] ep1:   0%|          | 0/50 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

  ep1: val_QWK=0.0180 losses={'L_task': 0.6928, 'L_aux': 1.392, 'L_logit_KD': 0.6328, 'L_CSD': 0.5788, 'L_total': 1.8209} grad={'task': 2.176, 'aux': 0.0, 'logit_kd': 0.133, 'csd': 1.655, 'ratio_csd_over_task': 0.761}
[abl_csd_counterfactual|s42] best val QWK=0.0180


('/content/drive/MyDrive/DR-VERGE/artifacts_preflight_v1/checkpoints/student/abl_csd_counterfactual/best_seed42.pt',
 [{'L_task': 0.8065138387680054,
   'L_aux': 1.4268918013572693,
   'W_aux': 0.7134459006786347,
   'L_logit_KD': 0.6708504498004914,
   'W_logit_KD': 0.3354252249002457,
   'L_CSD': 1.3736079344153405,
   'W_CSD': 0.27472158923745155,
   'L_total': 2.130106568336487,
   'epoch': 0,
   'val_qwk': np.float64(-0.00804608210661062),
   'lr': 0.00051,
   'gnorm_task': 6.471666602059163,
   'gnorm_aux': 0.0,
   'gnorm_logit_kd': 1.0340615238139093,
   'gnorm_csd': 14.893120006597194,
   'gnorm_ratio_csd_over_task': 2.301280477251174},
  {'L_task': 0.6927844119071961,
   'L_aux': 1.3919514620304108,
   'W_aux': 0.6959757310152054,
   'L_logit_KD': 0.6327997708320617,
   'W_logit_KD': 0.31639988541603087,
   'L_CSD': 0.5788439586758614,
   'W_CSD': 0.11576879367232323,
   'L_total': 1.8209288144111633,
   'epoch': 1,
   'val_qwk': np.float64(0.01796807798910438),
   'lr': 2e-05

## 24–25 — Model selection (validation only) & best FP32 deployment candidate

`M* = argmax QWK_val` over {no-distill, logit-KD, feature-KD, CSD}. Ties within 0.005 resolve by
Macro-F1 → lower severe-error → lower MAE → simpler method. **The test set is not consulted.**

In [29]:
CORE_CONDITIONS = ["dual_no_distill", "dual_logitkd", "dual_featkd", "dual_csd"]

def collect_val_scores():
    rows = []
    for cond in CORE_CONDITIONS:
        for s in SEEDS_CORE:
            ck = f"{CKPT_DIR}/student/{cond}/best_seed{s}.pt"
            if not os.path.exists(ck): continue
            st = robust_torch_load(ck, map_location="cpu")
            mdl = DualViewLightStudent(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS).to(DEVICE)
            mdl.load_state_dict(st["model_state"])
            y, yp, pc = get_predictions(mdl, VAL_LOADER, DEVICE, "dual")
            m = compute_all_metrics(y, yp, pc)
            rows.append({"condition": cond, "seed": s, "checkpoint": ck, **{k: m[k] for k in
                         ("QWK", "MacroF1", "SevereErrorRate", "MAE", "Accuracy")}})
            del mdl
    return pd.DataFrame(rows)

VAL_SCORES = collect_val_scores()
VAL_SCORES.to_csv(f"{TABLES_DIR}/table_02_validation_scores.csv", index=False)
print(VAL_SCORES.groupby("condition")[["QWK", "MacroF1", "SevereErrorRate", "MAE"]].agg(["mean", "std"]).round(4).to_string())

def select_best(df):
    d = df.sort_values("QWK", ascending=False).reset_index(drop=True)
    top = d.iloc[0]["QWK"]
    tied = d[d["QWK"] >= top - SELECTION_TIE_EPS]
    if len(tied) > 1:
        print(f"  {len(tied)} candidates within {SELECTION_TIE_EPS} QWK -- applying tie-break chain")
        tied = tied.sort_values(["MacroF1", "SevereErrorRate", "MAE"], ascending=[False, True, True])
    return tied.iloc[0]

BEST_ROW = select_best(VAL_SCORES)
BEST_CONDITION, BEST_SEED = BEST_ROW["condition"], int(BEST_ROW["seed"])
BEST_FP32_CKPT = BEST_ROW["checkpoint"]
print(f"\nM* (validation-selected) = {BEST_CONDITION} seed {BEST_SEED} | val QWK={BEST_ROW['QWK']:.4f}")

# The best CSD model is tracked separately: even if M* is not CSD, the paper still needs the best
# CSD artifact for the RQ1 mechanism analysis.
_csd = VAL_SCORES[VAL_SCORES.condition == "dual_csd"]
BEST_CSD_SEED = int(select_best(_csd)["seed"]) if len(_csd) else PRIMARY_SEED
BEST_CSD_CKPT = f"{CKPT_DIR}/student/dual_csd/best_seed{BEST_CSD_SEED}.pt"
print(f"Best CSD artifact        = dual_csd seed {BEST_CSD_SEED}")
save_json({"best_condition": BEST_CONDITION, "best_seed": BEST_SEED, "best_ckpt": BEST_FP32_CKPT,
           "best_val_qwk": float(BEST_ROW["QWK"]), "best_csd_seed": BEST_CSD_SEED,
           "selection_rule": f"argmax {SELECTION_METRIC}_val, tie<{SELECTION_TIE_EPS} -> {SELECTION_TIEBREAK}"},
          f"{CONFIG_DIR}/model_selection.json")

def load_student(ckpt):
    m = DualViewLightStudent(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS).to(DEVICE)
    m.load_state_dict(robust_torch_load(ckpt, map_location=DEVICE)["model_state"])
    return m.eval()

# One fixed example pair reused for export tracing, latency benchmarking and parity checks.
_ex_batch = next(iter(make_loader(DRTiDDualViewDataset(DRTID_VAL_CSV, eval_transform), 8, False, workers=0)))
_sample_pair = (_ex_batch["macula"], _ex_batch["disc"])

                    QWK     MacroF1     SevereErrorRate        MAE    
                   mean std    mean std            mean std   mean std
condition                                                             
dual_csd         0.0361 NaN  0.1255 NaN           0.320 NaN  1.150 NaN
dual_featkd      0.0513 NaN  0.1732 NaN           0.380 NaN  1.155 NaN
dual_logitkd     0.0331 NaN  0.1802 NaN           0.405 NaN  1.180 NaN
dual_no_distill  0.0394 NaN  0.1698 NaN           0.400 NaN  1.185 NaN

M* (validation-selected) = dual_featkd seed 42 | val QWK=0.0513
Best CSD artifact        = dual_csd seed 42


## 26–29 — Quantization: PTQ INT8, QAT INT8, and matched controls

**Matched scope comes first.** RQ2 asks *PTQ vs QAT*. That question is only answerable if both
quantize the same operators, so **both use eager backbone-only quantization**: the same
`QuantizableBackbone` wrapper, the same fused Conv-BN-ReLU set, the same `fbgemm` qconfig family.
They differ in exactly one thing — static calibration versus quantization-aware fine-tuning.

**Why PT2E is not the RQ2 path.** `prepare_pt2e` with `Quantizer.set_global()` quantizes every
eligible operator in the exported graph, which is a *different* operator set from eager
backbone-only. Comparing PT2E PTQ against eager QAT would confound "PTQ vs QAT" with "different
quantization scope". PT2E is still exercised — as a **supplementary deployment-path demonstration**
(`ptq_int8_pt2e`), reported in its own row and excluded from every RQ2 statistic. The paper must
describe it exactly this way:

```
RQ2 pair      : eager static PTQ (backbone-only)  vs  eager QAT (backbone-only)   [identical scope]
supplementary : PT2E static PTQ, reported separately, never mixed into the comparison
```

**Scope is stated honestly.** Only the CNN backbone is quantized: `torch.cat`, LayerNorm and CORAL's
cumsum/softplus/sigmoid have no eager INT8 kernels. The artifact is an **INT8-quantized backbone with
FP32 fusion and ordinal heads** (mixed-precision deployment).

**`quantization_coverage_pct` is an integrity check, not a headline.** For eager it counts converted
modules; for a PT2E graph it counts quantization *nodes*, which do not map one-to-one onto
conv/linear operators. Gates require `quantized_ops > 0`; the percentage never becomes a claim.

**QAT gets two controls.**

* `fp32_ft_control` (**primary**) — the *identical* fused, QAT-prepared graph with fake quantization
  and observers switched off. Same architecture, same fusion, same optimizer, same epochs, same
  early stopping; the only remaining difference is whether fake quantization is active. The earlier
  control fine-tuned an ordinary unfused model, so it differed in graph structure too and could not
  isolate the variable it was named after.
* `fp32_ft_plain` (secondary) — an ordinary FP32 fine-tune, kept because it is the control a reader
  expects to see.

In [30]:
from torch.ao.quantization import (prepare, convert, prepare_qat, get_default_qconfig,
                                   get_default_qat_qconfig, QuantStub, DeQuantStub,
                                   disable_fake_quant, disable_observer)

# LOCKED QUANTIZATION SCOPE. The RQ2 pair (PTQ vs QAT) must quantize the same operators, so both
# take the eager backbone-only path. PT2E runs separately, as a supplementary demonstration that the
# modern export path works, and never enters an RQ2 statistic.
QUANT_SCOPE = "eager_backbone_only"
QUANT_COVERAGE_CAVEAT = ("quantization_coverage_pct is an INTEGRITY CHECK, not a headline metric: "
                         "for PT2E it counts quantization graph NODES, which do not map one-to-one "
                         "onto conv/linear operators. Gates require quantized_ops > 0 only.")

QUANT_ENGINE = "fbgemm" if "fbgemm" in torch.backends.quantized.supported_engines                else torch.backends.quantized.supported_engines[0]
torch.backends.quantized.engine = QUANT_ENGINE
print("quantization engine:", QUANT_ENGINE)

PT2E_AVAILABLE, PT2E_IMPORT_ERROR = False, None
try:
    # `prepare_qat_pt2e` is imported as part of the availability probe only. PT2E QAT is deliberately
    # NOT used: QAT runs eager so that it shares an identical quantization scope with eager PTQ, which
    # is what makes the RQ2 comparison fair. Nothing in this notebook claims a PT2E QAT result.
    from torch.ao.quantization.quantize_pt2e import prepare_pt2e, convert_pt2e, prepare_qat_pt2e
    try:
        from torch.ao.quantization.quantizer.x86_inductor_quantizer import (
            X86InductorQuantizer as _Quantizer, get_default_x86_inductor_quantization_config as _qcfg)
    except Exception:
        from torch.ao.quantization.quantizer.xnnpack_quantizer import (
            XNNPACKQuantizer as _Quantizer, get_symmetric_quantization_config as _qcfg)
    PT2E_AVAILABLE = True
except Exception as _e:
    PT2E_IMPORT_ERROR = repr(_e)
print("PT2E available:", PT2E_AVAILABLE, "" if PT2E_AVAILABLE else f"({PT2E_IMPORT_ERROR})")

class QuantizableBackbone(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.quant = QuantStub(); self.backbone = backbone; self.dequant = DeQuantStub()
    def forward(self, x):
        return self.dequant(self.backbone(self.quant(x)))

def count_quantized_graph_nodes(model):
    """PT2E returns a GraphModule whose INT8 ops are call_function nodes
    (torch.ops.quantized_decomposed.*), not nn.Module instances -- counting modules alone reports 0
    for a perfectly good PT2E conversion, which would fail Gate 6 on a working model."""
    g = getattr(model, "graph", None)
    if g is None: return 0
    n = 0
    for node in g.nodes:
        t = str(getattr(node, "target", ""))
        if "quantize_per_tensor" in t or "quantize_per_channel" in t or "quantized_decomposed" in t:
            n += 1
    return n

def count_quantized_modules(model):
    """Eager mode: fused quantized modules live under torch.ao.nn.intrinsic.quantized.* -- the class
    NAME (e.g. ConvReLU2d) need not contain 'Quantized', so check the module PATH."""
    return sum(1 for m in model.modules() if "quantized" in type(m).__module__.lower())

def count_quantized_ops(model):
    """Mode-agnostic: eager modules OR PT2E graph nodes."""
    return max(count_quantized_modules(model), count_quantized_graph_nodes(model))

def is_quantized_tree(model):
    return count_quantized_ops(model) > 0

def count_active_fake_quant(model):
    """FakeQuantize modules whose fake-quant is actually ENABLED (a disabled one is a pass-through)."""
    n = 0
    for m in model.modules():
        if hasattr(m, "fake_quant_enabled"):
            try:
                n += int(m.fake_quant_enabled.sum().item() > 0)
            except Exception:
                n += 1
    return n

def count_eligible_ops(model):
    return sum(1 for m in model.modules() if isinstance(m, (nn.Conv2d, nn.Linear)))

def quantization_coverage(quantized_model, fp32_reference):
    """Fraction of eligible conv/linear ops actually converted. Works for eager and PT2E."""
    elig = count_eligible_ops(fp32_reference)
    got = count_quantized_ops(quantized_model)
    return {"eligible_ops": elig, "quantized_ops": got,
            "quantization_coverage_pct": 100.0 * min(got, elig) / max(elig, 1),
            "quantized_op_count_raw": got}

def make_calib_loader(batch_size=8):
    return make_loader(DRTiDDualViewDataset(DRTID_TRAIN_CSV, eval_transform), batch_size, True, workers=0)

def _fresh_student_from(ckpt):
    m = DualViewLightStudent(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS)
    m.load_state_dict(robust_torch_load(ckpt, map_location="cpu")["model_state"])
    return m.to("cpu").eval()

def _calibrate(prepared, calib_batches):
    cl = make_calib_loader()
    n = 0
    with torch.no_grad():
        for i, b in enumerate(cl):
            prepared(b["macula"], b["disc"]); n = i + 1
            if n >= calib_batches: break
    return n

def run_ptq_eager(fp32_ckpt, calib_batches=64):
    """PRIMARY PTQ path -- eager, backbone-only, byte-for-byte the same scope QAT uses."""
    calib_batches = 4 if PREFLIGHT else calib_batches
    ref = _fresh_student_from(fp32_ckpt)
    model = _fresh_student_from(fp32_ckpt)
    model.backbone.fuse_model(qat=False)
    model.backbone = QuantizableBackbone(model.backbone)
    model.backbone.qconfig = get_default_qconfig(QUANT_ENGINE)
    prepared = prepare(model, inplace=False)
    n = _calibrate(prepared, calib_batches)
    q = convert(prepared, inplace=False)
    if count_quantized_ops(q) == 0:
        raise RuntimeError("eager PTQ convert produced no quantized modules")
    return q, {"path": "eager_static_ptq", "scope": QUANT_SCOPE, "engine": QUANT_ENGINE,
               "calib_batches": n, "is_graph_module": False, "used_in_RQ2": True,
               **quantization_coverage(q, ref)}

def run_ptq_pt2e(fp32_ckpt, calib_batches=64):
    """SUPPLEMENTARY PT2E path (torch.export + prepare_pt2e/convert_pt2e).

    Reported on its own row and never compared against eager QAT: set_global() quantizes every
    eligible operator in the exported graph, a different scope from eager backbone-only, and mixing
    the two would confound "PTQ vs QAT" with "different scope".
    """
    if not PT2E_AVAILABLE:
        raise RuntimeError(f"PT2E unavailable: {PT2E_IMPORT_ERROR}")
    calib_batches = 4 if PREFLIGHT else calib_batches
    ref = _fresh_student_from(fp32_ckpt)
    model = _fresh_student_from(fp32_ckpt)
    # Export with a DYNAMIC batch dimension: exporting from a batch-1 example specializes the graph
    # to batch 1 and calibration then trips "Guard failed: macula.size()[0] == 1".
    ex = (_sample_pair[0][:2].cpu(), _sample_pair[1][:2].cpu())
    try:
        from torch.export import Dim
        _bd = Dim("batch", min=1, max=256)
        exported = torch.export.export(model, ex, dynamic_shapes=({0: _bd}, {0: _bd})).module()
    except Exception:
        exported = torch.export.export(model, ex).module()
    prepared = prepare_pt2e(exported, _Quantizer().set_global(_qcfg()))
    n = _calibrate(prepared, calib_batches)
    q = convert_pt2e(prepared)
    cov = quantization_coverage(q, ref)
    if cov["quantized_ops"] == 0:
        raise RuntimeError("PT2E convert produced no quantized ops")
    return q, {"path": "pt2e_static_ptq", "scope": "pt2e_global_exported_graph",
               "engine": QUANT_ENGINE, "calib_batches": n, "is_graph_module": True,
               "used_in_RQ2": False, **cov}

quantization engine: fbgemm
PT2E available: False (ModuleNotFoundError("No module named 'torch.ao.quantization.quantize_pt2e'"))


In [31]:
# ---- PRIMARY PTQ: eager, backbone-only, matched to QAT ----
PTQ_MODEL, PTQ_INFO, PTQ_OK = None, {}, False
try:
    PTQ_MODEL, PTQ_INFO = run_ptq_eager(BEST_FP32_CKPT)
    with torch.no_grad():
        _b = next(iter(make_loader(DRTiDDualViewDataset(DRTID_VAL_CSV, eval_transform), 4, False, workers=0)))
        _o = PTQ_MODEL(_b["macula"], _b["disc"])
        _p = _o["p_dual"] if isinstance(_o, dict) else _o
    assert _p.shape[1] == NUM_THRESHOLDS, f"PTQ output has {_p.shape[1]} thresholds, expected {NUM_THRESHOLDS}"
    PTQ_OK = is_quantized_tree(PTQ_MODEL)
    record_gate("Gate6_PTQ_Integrity", PTQ_OK,
                f"path={PTQ_INFO['path']} scope={PTQ_INFO['scope']} "
                f"quantized_ops={PTQ_INFO['quantized_ops']}/{PTQ_INFO['eligible_ops']} eligible "
                f"(the percentage is an integrity check, not a headline)",
                blocking=not PREFLIGHT)
except Exception as e:
    print(f"PTQ FAILED: {e!r}")
    record_gate("Gate6_PTQ_Integrity", False, f"FAILED: {e!r}", blocking=not PREFLIGHT)

# ---- SUPPLEMENTARY PT2E PTQ: modern export path, reported, never mixed into RQ2 ----
PTQ_PT2E_MODEL, PTQ_PT2E_INFO, PTQ_PT2E_OK = None, {}, False
try:
    PTQ_PT2E_MODEL, PTQ_PT2E_INFO = run_ptq_pt2e(BEST_FP32_CKPT)
    PTQ_PT2E_OK = is_quantized_tree(PTQ_PT2E_MODEL)
    print(f"  PT2E supplementary PTQ OK: {PTQ_PT2E_INFO['quantized_ops']} quantization graph nodes")
except Exception as e:
    PTQ_PT2E_INFO = {"path": "pt2e_static_ptq", "pt2e_available": PT2E_AVAILABLE,
                     "error": repr(e), "used_in_RQ2": False}
    print(f"  PT2E supplementary PTQ unavailable ({e!r}). The RQ2 pair is unaffected -- it never used PT2E.")
record_gate("Gate6b_PT2E_Supplementary", PTQ_PT2E_OK,
            "modern torch.export quantization path demonstrated" if PTQ_PT2E_OK else
            f"not demonstrated ({PTQ_PT2E_INFO.get('error', 'unavailable')}); supplementary only, non-blocking")

/tmp/ipykernel_1065/376275280.py:113: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  prepared = prepare(model, inplace=False)
/usr/local/lib/python3.12/dist-packages/torch/ao/quantization/observer.py:1039: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.


PASS | Gate6_PTQ_Integrity | path=eager_static_ptq scope=eager_backbone_only quantized_ops=15/17 eligible (the percentage is an integrity check, not a headline)
  PT2E supplementary PTQ unavailable (RuntimeError('PT2E unavailable: ModuleNotFoundError("No module named \'torch.ao.quantization.quantize_pt2e\'")')). The RQ2 pair is unaffected -- it never used PT2E.
FAIL | Gate6b_PT2E_Supplementary | not demonstrated (RuntimeError('PT2E unavailable: ModuleNotFoundError("No module named \'torch.ao.quantization.quantize_pt2e\'")')); supplementary only, non-blocking


False

In [32]:
def _finetune_loop(model, tag, epochs, lr, batch_size, patience, seed, fake_quant=False):
    """Shared fine-tuning loop for QAT and its FP32 control, so the ONLY difference between them is
    whether fake quantization is active."""
    set_seed(seed)
    pw = POS_WEIGHT.to(DEVICE)
    tl = make_loader(DRTiDDualViewDataset(DRTID_TRAIN_CSV, paired_transform=paired_train_transform), batch_size, True, seed)
    vl = make_loader(DRTiDDualViewDataset(DRTID_VAL_CSV, eval_transform), batch_size, False)
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    best, bad, best_state, hist = -1.0, 0, None, []
    for ep in range(epochs):
        model.train()
        for b in tqdm(tl, desc=f"[{tag}] ep{ep}", leave=False):
            m, d, y = b["macula"].to(DEVICE), b["disc"].to(DEVICE), b["label"].to(DEVICE)
            o = model(m, d)
            loss = coral_loss(o["logit_dual"], y, pos_weight=pw) + 0.5 * aux_loss(o, y, pos_weight=pw)
            opt.zero_grad(); loss.backward(); opt.step()
        q = quick_val_qwk(model, vl, DEVICE, "dual")
        hist.append({"epoch": ep, "val_qwk": q}); print(f"  {tag} ep{ep}: val_QWK={q:.4f}")
        if q > best:
            best, bad = q, 0; best_state = copy.deepcopy(model.state_dict())
        else:
            bad += 1
            if bad >= patience: print(f"  early stop @ep{ep}"); break
    if best_state is not None: model.load_state_dict(best_state)
    pd.DataFrame(hist).to_csv(f"{LOGS_DIR}/{tag}_history.csv", index=False)
    return model, best, len(hist)

def count_fake_quant(model):
    return sum(1 for m in model.modules() if "fakequantize" in type(m).__name__.lower()
               or "FakeQuant" in type(m).__name__)

def _build_qat_prepared(fp32_ckpt, load_fp32=True):
    model = DualViewLightStudent(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS)
    if load_fp32:
        model.load_state_dict(robust_torch_load(fp32_ckpt, map_location="cpu")["model_state"])
    model = model.to("cpu").train()
    model.backbone.fuse_model(qat=True)
    model.backbone = QuantizableBackbone(model.backbone)
    model.backbone.qconfig = get_default_qat_qconfig(QUANT_ENGINE)
    prepare_qat(model, inplace=True)
    return model

def run_qat(fp32_ckpt, epochs=10, lr=3e-5, batch_size=16, patience=4, seed=PRIMARY_SEED, force=False):
    """QAT from the best FP32 weights; validation-selected; then converted.
    Resumable: the fine-tuned pre-conversion weights are cached so a re-run skips the epochs."""
    epochs = 1 if PREFLIGHT else epochs
    ck = f"{CKPT_DIR}/student/qat/qat_prepared_seed{seed}.pt"
    os.makedirs(os.path.dirname(ck), exist_ok=True)

    if not force and os.path.exists(ck):
        try:
            saved = robust_torch_load(ck, map_location="cpu")
            model = _build_qat_prepared(fp32_ckpt, load_fp32=False)
            model.load_state_dict(saved["model_state"])
            q = convert(model.to("cpu").eval(), inplace=False)
            info = dict(saved.get("info", {}))
            info.update({"resumed_from_checkpoint": True, "scope": QUANT_SCOPE, "used_in_RQ2": True})
            info.update(quantization_coverage(q, _fresh_student_from(fp32_ckpt)))
            print(f"QAT: reused cached fine-tuned weights (val QWK={info.get('best_val_qwk', float('nan')):.4f})")
            return q, info
        except Exception as e:
            print(f"QAT cache unusable ({e!r}) -- re-running fine-tuning.")

    model = _build_qat_prepared(fp32_ckpt, load_fp32=True)
    # ASSERT fake quantization is genuinely active DURING training -- counting quantized modules
    # after conversion proves nothing about what happened while training.
    n_fq = count_fake_quant(model)
    assert n_fq > 0, "QAT prepare produced no FakeQuantize modules -- fake quantization is NOT active"
    print(f"  QAT: {n_fq} FakeQuantize modules active before fine-tuning starts")

    model, best, n_ep = _finetune_loop(model, f"QAT_seed{seed}", epochs, lr, batch_size, patience, seed)
    info = {"path": "eager_qat", "scope": QUANT_SCOPE, "engine": QUANT_ENGINE,
            "best_val_qwk": best, "epochs_run": n_ep, "lr": lr, "qat_training_seed": seed,
            "fake_quant_modules_during_training": n_fq, "used_in_RQ2": True,
            "resumed_from_checkpoint": False}
    robust_torch_save({"model_state": {k: v.cpu() for k, v in model.state_dict().items()},
                       "info": info, "epochs_run": n_ep}, ck)
    q = convert(model.to("cpu").eval(), inplace=False)
    info.update(quantization_coverage(q, _fresh_student_from(fp32_ckpt)))
    return q, info

def run_fp32_ft_control(fp32_ckpt, epochs=10, lr=3e-5, batch_size=16, patience=4,
                        seed=PRIMARY_SEED, force=False):
    """PRIMARY QAT control: the SAME fused, QAT-prepared graph with fake quantization and observers
    DISABLED.

    The earlier control fine-tuned an ordinary unfused FP32 student, so it differed from QAT in
    fusion AND graph structure as well as in fake quantization -- it could not isolate the single
    variable it was named after. Returns the live model, because a prepared graph's state_dict
    cannot be loaded back into a plain DualViewLightStudent.
    """
    epochs = 1 if PREFLIGHT else epochs
    out = f"{CKPT_DIR}/student/fp32_ft_control/prepared_seed{seed}.pt"
    os.makedirs(os.path.dirname(out), exist_ok=True)

    def _fresh_control():
        m = _build_qat_prepared(fp32_ckpt, load_fp32=True)
        m.apply(disable_fake_quant); m.apply(disable_observer)
        n_active = count_active_fake_quant(m)
        assert n_active == 0, f"control still has {n_active} ACTIVE fake-quant modules -- not a control"
        return m

    if not force and os.path.exists(out):
        try:
            saved = robust_torch_load(out, map_location="cpu")
            m = _fresh_control()
            m.load_state_dict(saved["model_state"])
            m.apply(disable_fake_quant); m.apply(disable_observer)
            v = float(saved.get("val_qwk", float("nan")))
            print(f"fp32_ft_control seed{seed}: reused cached weights (val QWK={v:.4f})")
            return m.eval(), out, v
        except Exception as e:
            print(f"fp32_ft_control cache unusable ({e!r}) -- re-running fine-tuning.")

    model = _fresh_control()
    model, best, n_ep = _finetune_loop(model, f"FP32FT_seed{seed}", epochs, lr, batch_size, patience, seed)
    model.apply(disable_fake_quant); model.apply(disable_observer)   # belt and braces after training
    assert count_active_fake_quant(model) == 0, "fake quantization re-enabled during control training"
    robust_torch_save({"model_state": {k: v.cpu() for k, v in model.state_dict().items()},
                       "val_qwk": best, "seed": seed, "epochs_run": n_ep,
                       "role": "qat_matched_fp32_control_fake_quant_disabled"}, out)
    return model.eval(), out, best

def run_fp32_ft_plain(fp32_ckpt, epochs=10, lr=3e-5, batch_size=16, patience=4,
                      seed=PRIMARY_SEED, force=False):
    """SECONDARY control: ordinary FP32 fine-tuning of the unfused student, same budget."""
    epochs = 1 if PREFLIGHT else epochs
    out = f"{CKPT_DIR}/student/fp32_ft_plain/best_seed{seed}.pt"
    os.makedirs(os.path.dirname(out), exist_ok=True)
    probe = DualViewLightStudent(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS)
    if not force and checkpoint_is_compatible(out, probe, "model_state"):
        print(f"fp32_ft_plain seed{seed}: cached, skipping."); return out
    del probe
    model = _fresh_student_from(fp32_ckpt)
    model, best, n_ep = _finetune_loop(model, f"FP32FTplain_seed{seed}", epochs, lr, batch_size,
                                       patience, seed)
    robust_torch_save({"model_state": model.state_dict(), "val_qwk": best, "seed": seed,
                       "epochs_run": n_ep, "role": "secondary_plain_fp32_finetuning_control"}, out)
    return out

In [33]:
# ---- QAT controls ----
# PRIMARY: the identical QAT-prepared graph with fake quantization OFF (kept in memory -- its
# state_dict belongs to the prepared graph, not to a plain DualViewLightStudent).
FP32_FT_MODELS, FP32_FT_CKPTS, FP32_FT_VALQWK = {}, {}, {}
for sd in SEEDS_QAT:
    _m, _p, _v = run_fp32_ft_control(BEST_FP32_CKPT, seed=sd)
    FP32_FT_MODELS[sd], FP32_FT_CKPTS[sd], FP32_FT_VALQWK[sd] = _m, _p, _v
# SECONDARY: ordinary FP32 fine-tune, primary seed only.
FP32_FT_PLAIN_CKPT = run_fp32_ft_plain(BEST_FP32_CKPT, seed=PRIMARY_SEED)

QAT_MODELS, QAT_INFOS, QAT_OK = {}, {}, False
for sd in SEEDS_QAT:
    try:
        qm, qi = run_qat(BEST_FP32_CKPT, seed=sd)
        QAT_MODELS[sd], QAT_INFOS[sd] = qm, qi
    except Exception as e:
        print(f"QAT seed {sd} FAILED: {e!r}")

def _qat_val(s):
    try:
        v = float(QAT_INFOS[s].get("best_val_qwk", float("nan")))
    except Exception:
        v = float("nan")
    return v if v == v else float("-inf")

QAT_DEPLOY_SEED = None
if QAT_MODELS:
    # Deployment seed is chosen on DRTiD VALIDATION -- never on test or external data. The earlier
    # code took sorted(QAT_MODELS)[0] but then labelled that model with BEST_SEED in the external
    # table, so the reported seed could name a model that was never trained.
    QAT_DEPLOY_SEED = max(QAT_MODELS, key=_qat_val)
    QAT_MODEL, QAT_INFO = QAT_MODELS[QAT_DEPLOY_SEED], QAT_INFOS[QAT_DEPLOY_SEED]
    # Precedence fixed. `A and B or C` binds as `(A and B) or C`, so a cached run could pass this
    # gate with an UNQUANTIZED tree. Every configured seed must now have produced a quantized model.
    QAT_OK = (len(QAT_MODELS) == len(SEEDS_QAT)
              and all(is_quantized_tree(m) for m in QAT_MODELS.values())
              and all((QAT_INFOS[s].get("fake_quant_modules_during_training", 0) or 0) > 0
                      or QAT_INFOS[s].get("resumed_from_checkpoint", False) for s in QAT_MODELS))
    record_gate("Gate7_QAT_Integrity", QAT_OK,
                f"{len(QAT_MODELS)}/{len(SEEDS_QAT)} seed(s) quantized; deploy seed={QAT_DEPLOY_SEED} "
                f"(val QWK={_qat_val(QAT_DEPLOY_SEED):.4f}); fake-quant during training="
                f"{QAT_INFO.get('fake_quant_modules_during_training', 'cached')}",
                blocking=not PREFLIGHT)
else:
    QAT_MODEL, QAT_INFO = None, {}
    record_gate("Gate7_QAT_Integrity", False, "no QAT model produced", blocking=not PREFLIGHT)

# RQ2 fairness: the compared pair must quantize the SAME operator set.
_scope_ok = bool(PTQ_OK and QAT_OK
                 and PTQ_INFO.get("scope") == QAT_INFO.get("scope")
                 and PTQ_INFO.get("quantized_ops") == QAT_INFO.get("quantized_ops"))
record_gate("Gate6c_QuantScopeMatched", _scope_ok,
            f"PTQ scope={PTQ_INFO.get('scope')} ops={PTQ_INFO.get('quantized_ops')} | "
            f"QAT scope={QAT_INFO.get('scope')} ops={QAT_INFO.get('quantized_ops')} "
            f"-- RQ2 is only apples-to-apples when these are identical")

save_json({"locked_scope": QUANT_SCOPE,
           "rq2_pair": "eager_static_ptq vs eager_qat (identical backbone-only scope)",
           "coverage_caveat": QUANT_COVERAGE_CAVEAT,
           "scope_match_verified": _scope_ok,
           "ptq": PTQ_INFO, "ptq_pt2e_supplementary": PTQ_PT2E_INFO,
           "qat": {str(k): v for k, v in QAT_INFOS.items()},
           "qat_deploy_seed": QAT_DEPLOY_SEED,
           "qat_deploy_seed_selected_on": "DRTiD validation QWK",
           "fp32_ft_control": {"kind": "qat_prepared_graph_with_fake_quant_disabled",
                               "val_qwk": {str(k): v for k, v in FP32_FT_VALQWK.items()}},
           "fp32_ft_plain_control": {"kind": "ordinary_fp32_finetune", "seed": PRIMARY_SEED},
           "engine": QUANT_ENGINE, "pt2e_available": PT2E_AVAILABLE,
           "pt2e_import_error": PT2E_IMPORT_ERROR,
           "artifact": "INT8-quantized backbone with FP32 fusion and ordinal heads (mixed precision)"},
          f"{CONFIG_DIR}/quantization_info.json")

/tmp/ipykernel_1065/4047005942.py:41: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  prepare_qat(model, inplace=True)
/usr/local/lib/python3.12/dist-packages/torch/ao/quantization/observer.py:534: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  super()

[FP32FT_seed42] ep0:   0%|          | 0/50 [00:00<?, ?it/s]

  FP32FT_seed42 ep0: val_QWK=0.0269


[FP32FTplain_seed42] ep0:   0%|          | 0/50 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  Exception ignored in:  <function _MultiProcessingDataLoaderIter.__del__ at 0x7ea170291580>
  Traceback (most recent call last):
    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
^    ^^self._shutdown_workers()^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
^    ^if w.is_alive():^
^^ ^ ^ ^ 
   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
      assert self._parent_pid == os.getpid(), 'can only test a child process'^
^ ^ ^ ^ ^   ^^ ^ ^ ^ ^^
^  File "/u

  FP32FTplain_seed42 ep0: val_QWK=0.0269
  QAT: 27 FakeQuantize modules active before fine-tuning starts


/tmp/ipykernel_1065/4047005942.py:41: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  prepare_qat(model, inplace=True)
/usr/local/lib/python3.12/dist-packages/torch/ao/quantization/observer.py:534: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  super()

[QAT_seed42] ep0:   0%|          | 0/50 [00:00<?, ?it/s]

  QAT_seed42 ep0: val_QWK=0.0410
PASS | Gate7_QAT_Integrity | 1/1 seed(s) quantized; deploy seed=42 (val QWK=0.0410); fake-quant during training=27
PASS | Gate6c_QuantScopeMatched | PTQ scope=eager_backbone_only ops=15 | QAT scope=eager_backbone_only ops=15 -- RQ2 is only apples-to-apples when these are identical


/tmp/ipykernel_1065/4047005942.py:79: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  q = convert(model.to("cpu").eval(), inplace=False)


## 30–33 — Full internal test evaluation

**Honest scope statement.** Within this locked run the DRTiD official test set is not consulted for
any selection decision — every choice (hyperparameters, `M*`, deployment candidate) is made on
validation. But this project has a history: outcomes from an earlier development run (rev2) were
inspected and did motivate the revisions that produced this pipeline. So the correct wording for the
paper is *"the official test set was not used for model selection within the final locked run;
outcomes from earlier development runs had previously been inspected"* — **not** "touched once ever".
**DeepDRiD is the genuinely frozen confirmatory evaluation.**

Every condition × seed is evaluated on the held-out DRTiD test set. **Per-sample predictions are
written to `predictions/<condition>_<seed>.csv`** so any future metric can be recomputed without
re-running inference or touching the models again.

In [34]:
TEST_DS = DRTiDDualViewDataset(DRTID_TEST_CSV, eval_transform)
TEST_LOADER = make_loader(TEST_DS, 16, False)
_sample = next(iter(TEST_LOADER))
TEACHER = get_teacher()

CLUSTER_LEVEL = {"DRTiD": "record_eye", "DeepDRiD": "patient"}

def save_predictions(condition, seed, y_true, y_pred, p_cum, cluster_ids, quantization="FP32",
                     dataset="DRTiD", split="test", latency_ms=np.nan):
    df = pd.DataFrame({"dataset": dataset, "split": split,
                       "cluster_id": cluster_ids,
                       "cluster_level": CLUSTER_LEVEL.get(dataset, "unknown"),
                       "sample_id": [f"{dataset}_{p}" for p in cluster_ids],
                       "true_grade": y_true, "pred_grade": y_pred,
                       "condition": condition, "seed": seed, "quantization": quantization,
                       "latency_ms": latency_ms})
    for k in range(p_cum.shape[1]):
        df[f"p_threshold_{k}"] = p_cum[:, k].cpu().numpy()
    path = f"{PREDS_DIR}/{dataset}_{split}_{condition}_seed{seed}_{quantization}.csv"
    df.to_csv(path, index=False)
    return path

def save_confusion(condition, seed, y_true, y_pred, tag=""):
    labels = list(range(NUM_CLASSES))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cmn = cm.astype(float) / np.maximum(cm.sum(1, keepdims=True), 1)
    idx = [f"true_{g}" for g in labels]; col = [f"pred_{g}" for g in labels]
    suffix = f"{condition}_seed{seed}{tag}"
    pd.DataFrame(cm, index=idx, columns=col).to_csv(f"{METRICS_DIR}/confusion_matrix_raw_{suffix}.csv")
    pd.DataFrame(cmn, index=idx, columns=col).to_csv(f"{METRICS_DIR}/confusion_matrix_normalized_{suffix}.csv")
    return cm, cmn

PRED_STORE = {}   # (condition, seed, quantization) -> dict(y_true, y_pred, cluster_ids)
rows, collapse_report = [], []

def serialized_state_dict_size_mb(model, tag):
    """Apples-to-apples model size: ALWAYS a freshly serialized state_dict, FP32 and INT8 alike.

    Previously FP32 rows reported `file_size_mb(training_checkpoint)` -- a file that also carries
    epoch, val_qwk, seed and config -- while INT8 rows reported a pure state_dict. The headline
    compression ratio was therefore computed between two different kinds of file.
    """
    path = f"{CKPT_DIR}/size_probe/{tag}_state_dict.pt"
    os.makedirs(os.path.dirname(path), exist_ok=True)
    try:
        robust_torch_save(model.state_dict(), path)
        return path, file_size_mb(path)
    except Exception as e:
        print(f"  could not serialize {tag} state_dict: {e!r}")
        return None, float("nan")

_int8_state_dict_size_mb = serialized_state_dict_size_mb   # legacy alias

@torch.no_grad()
def _predict_cpu_all(model, loader):
    """ONE forward pass yields the dual head AND both auxiliary heads.

    The earlier version called `model.forward_single(...)` for the auxiliary views. A model produced
    by `torch.export` is a GraphModule carrying only the traced `forward()` -- it has no
    `forward_single` attribute at all, so the internal dual-view gain would have raised
    AttributeError the moment the PT2E path succeeded. `forward()` already returns p_macula and
    p_disc, so reading them from the same pass is both export-safe and three times cheaper.
    """
    yt, cids, pd_, pm_, pdd_ = [], [], [], [], []
    aux_ok = True
    for b in loader:
        o = model(b["macula"].cpu(), b["disc"].cpu())
        if isinstance(o, dict):
            pd_.append(o["p_dual"])
            if "p_macula" in o and "p_disc" in o:
                pm_.append(o["p_macula"]); pdd_.append(o["p_disc"])
            else:
                aux_ok = False
        else:
            pd_.append(o); aux_ok = False
        yt.extend(b["label"].tolist()); cids.extend(b["cluster_id"].tolist())
    p_dual = torch.cat(pd_, 0)
    out = {"y_true": np.array(yt), "cluster_ids": np.array(cids), "p_dual": p_dual,
           "y_pred_dual": (p_dual > 0.5).sum(1).numpy()}
    if aux_ok:
        pm, pdd = torch.cat(pm_, 0), torch.cat(pdd_, 0)
        out.update({"p_macula": pm, "p_disc": pdd,
                    "y_pred_macula": (pm > 0.5).sum(1).numpy(),
                    "y_pred_disc": (pdd > 0.5).sum(1).numpy()})
    return out

def _predict_cpu(model, loader, view_mode="dual"):
    """Thin compatibility wrapper over _predict_cpu_all."""
    r = _predict_cpu_all(model, loader)
    key = {"dual": "dual", "macula_only": "macula", "disc_only": "disc"}[view_mode]
    if f"y_pred_{key}" not in r:
        raise RuntimeError(f"model exposes no head output for view_mode={view_mode}")
    return r["y_true"], r[f"y_pred_{key}"], r[f"p_{key}"], r["cluster_ids"]

def evaluate_condition(model, condition, seed, view_mode="dual", ckpt=None, quantization="FP32",
                       loader=None, on_cpu_model=False, with_shift=True, with_latency=True,
                       fp32_reference_model=None, size_tag=None, with_memory=False):
    loader = loader or TEST_LOADER
    cpu_pred = None
    if on_cpu_model:
        cpu_pred = _predict_cpu_all(model, loader)
        y_true, y_pred, p_cum, cids = (cpu_pred["y_true"], cpu_pred["y_pred_dual"],
                                       cpu_pred["p_dual"], cpu_pred["cluster_ids"])
    else:
        y_true, y_pred, p_cum, cids = get_predictions(model, loader, DEVICE, view_mode, return_clusters=True)

    m = compute_all_metrics(y_true, y_pred, p_cum)
    row = {"condition": condition, "seed": seed, "quantization": quantization,
           "view_mode": view_mode, **m}

    if view_mode == "dual":
        if on_cpu_model:
            # Quantized models are CPU-only, but they still need the dual-view gain -- this is the
            # exact RQ2 sub-question ("does INT8 preserve the dual-view advantage?"). Taken from the
            # SAME forward pass, so it works for an exported GraphModule too.
            if "y_pred_macula" in cpu_pred:
                qd = fast_qwk(y_true, y_pred)
                qm = fast_qwk(y_true, cpu_pred["y_pred_macula"])
                qdd = fast_qwk(y_true, cpu_pred["y_pred_disc"])
                row.update({"QWK_dual": qd, "QWK_aux_macula": qm, "QWK_aux_disc": qdd,
                            "DualViewGain_G_internal": qd - max(qm, qdd)})
            else:
                print(f"  {condition}: no auxiliary head outputs -- internal dual-view gain unavailable")
                row.update({"QWK_dual": fast_qwk(y_true, y_pred), "QWK_aux_macula": float("nan"),
                            "QWK_aux_disc": float("nan"), "DualViewGain_G_internal": float("nan")})
        else:
            row.update(compute_dual_view_gain(model, loader, DEVICE))
            if with_shift:
                row.update(compute_shift_fidelity(TEACHER, model, loader, DEVICE))

    # Quantization does NOT change the architectural parameter count -- packed INT8 weights simply
    # stop appearing as ordinary Parameters. Report the FP32 architecture's count and let SIZE carry
    # the compression story.
    ref_for_params = fp32_reference_model if fp32_reference_model is not None else model
    row["ParamCount"] = param_count(ref_for_params)
    row["ParamCount_is_architectural"] = True

    # SIZE: always a freshly serialized state_dict, for every condition and every precision, so
    # FP32-vs-INT8 compression is measured between two artifacts of the same kind.
    _sdp, _sds = serialized_state_dict_size_mb(model, size_tag or f"{condition}_seed{seed}_{quantization}")
    row["CheckpointSize_MB"] = _sds          # the comparable number used by every ratio
    row["StateDictSize_MB"] = _sds
    row["state_dict_path"] = _sdp or ""
    row["TrainingCheckpointSize_MB"] = (file_size_mb(ckpt) if (ckpt and os.path.exists(ckpt))
                                        else float("nan"))

    if with_latency:
        row.update(benchmark_latency(model, _sample["macula"], _sample["disc"], view_mode,
                                     bench=BENCH_EFF, on_cpu=True, label=condition))
    if with_memory:
        row.update(measure_memory(lambda: copy.deepcopy(model).to("cpu").eval(),
                                  _sample["macula"], _sample["disc"]))

    warns = check_prediction_collapse(y_true, y_pred)
    if warns:
        collapse_report.append({"condition": condition, "seed": seed, "quantization": quantization,
                                "warnings": "; ".join(warns)})
        print(f"  !! {condition}|s{seed}|{quantization}: " + "; ".join(warns))
    save_predictions(condition, seed, y_true, y_pred, p_cum, cids, quantization)
    save_confusion(condition, seed, y_true, y_pred, tag=f"_{quantization}")
    PRED_STORE[(condition, seed, quantization)] = {"y_true": y_true, "y_pred": y_pred, "cluster_ids": cids}
    rows.append(row)
    return row

# ---- Teacher ----
evaluate_condition(TEACHER, "teacher", "-", "dual", TEACHER_CKPT, with_shift=False)

# ---- Single-view baselines (also the EXTERNAL gain reference) ----
for cond, vm in [("macula_only", "macula_only"), ("disc_only", "disc_only")]:
    for s in SEEDS_BASELINE:
        ck = f"{CKPT_DIR}/student/{cond}/best_seed{s}.pt"
        if not os.path.exists(ck): continue
        mdl = load_student(ck); evaluate_condition(mdl, cond, s, vm, ck, with_shift=False); del mdl

# ---- Core dual-view conditions ----
for cond in CORE_CONDITIONS:
    for s in SEEDS_CORE:
        ck = f"{CKPT_DIR}/student/{cond}/best_seed{s}.pt"
        if not os.path.exists(ck): continue
        mdl = load_student(ck); evaluate_condition(mdl, cond, s, "dual", ck); del mdl

# ---- Ablations ----
for cond, seeds in [("abl_csd_raw_smoothl1", SEEDS_BASELINE), ("abl_csd_kl_softmax", SEEDS_BASELINE),
                    ("abl_csd_counterfactual", [PRIMARY_SEED])]:
    for s in seeds:
        ck = f"{CKPT_DIR}/student/{cond}/best_seed{s}.pt"
        if not os.path.exists(ck): continue
        mdl = load_student(ck); evaluate_condition(mdl, cond, s, "dual", ck); del mdl

RAW = pd.DataFrame(rows)
RAW.to_csv(f"{METRICS_DIR}/all_conditions_raw.csv", index=False)
print(f"\nEvaluated {len(RAW)} model-runs on the internal test set.")

  !! teacher|s-|FP32: Grade 1 recall 0.020 < 0.05
  !! macula_only|s42|FP32: Grade 3 recall 0.043 < 0.05; Grade 4 NEVER predicted; Grade 4 recall 0.000 < 0.05
  !! disc_only|s42|FP32: Grade 3 NEVER predicted; Grade 3 recall 0.000 < 0.05; Grade 4 NEVER predicted; Grade 4 recall 0.000 < 0.05
  !! dual_no_distill|s42|FP32: Grade 4 NEVER predicted; Grade 4 recall 0.000 < 0.05
  !! dual_logitkd|s42|FP32: Grade 4 NEVER predicted; Grade 4 recall 0.000 < 0.05
  !! dual_featkd|s42|FP32: Grade 4 NEVER predicted; Grade 4 recall 0.000 < 0.05
  !! dual_csd|s42|FP32: Grade 3 recall 0.029 < 0.05; Grade 4 NEVER predicted; Grade 4 recall 0.000 < 0.05
  !! abl_csd_raw_smoothl1|s42|FP32: Grade 4 NEVER predicted; Grade 4 recall 0.000 < 0.05
  !! abl_csd_kl_softmax|s42|FP32: Grade 3 NEVER predicted; Grade 3 recall 0.000 < 0.05; Grade 4 NEVER predicted; Grade 4 recall 0.000 < 0.05
  !! abl_csd_counterfactual|s42|FP32: Grade 0 NEVER predicted; Grade 0 recall 0.000 < 0.05; Grade 3 NEVER predicted; Grade 3 rec

In [35]:
# ---- External dual-view gain, now that the independent single-view references exist ----
_indep_mac = RAW[(RAW.condition == "macula_only")]["QWK"].mean()
_indep_dsc = RAW[(RAW.condition == "disc_only")]["QWK"].mean()
RAW["DualViewGain_G_external"] = RAW.apply(
    lambda r: compute_external_gain(r["QWK"], _indep_mac, _indep_dsc)
    if r["view_mode"] == "dual" else np.nan, axis=1)
print(f"External gain reference: independent macula QWK={_indep_mac:.4f}, disc QWK={_indep_dsc:.4f}")

# ---- Quantized variants of M* (RQ2) ----
BEST_FP32_MODEL = load_student(BEST_FP32_CKPT)
fp32_row = evaluate_condition(BEST_FP32_MODEL, "best_fp32", BEST_SEED, "dual", BEST_FP32_CKPT,
                              quantization="FP32", with_memory=True, size_tag="best_fp32")

# PRIMARY QAT control: the matched QAT-prepared graph with fake quantization disabled. It is
# evaluated from the live object because its state_dict belongs to the prepared graph and cannot be
# loaded into a plain DualViewLightStudent.
for sd, mdl in FP32_FT_MODELS.items():
    evaluate_condition(mdl.to(DEVICE), "fp32_ft_control", sd, "dual", FP32_FT_CKPTS[sd], "FP32",
                       with_shift=False, size_tag=f"fp32_ft_control_seed{sd}")

# SECONDARY control: ordinary FP32 fine-tune of the unfused student.
if os.path.exists(FP32_FT_PLAIN_CKPT):
    _mp = load_student(FP32_FT_PLAIN_CKPT)
    evaluate_condition(_mp, "fp32_ft_plain", PRIMARY_SEED, "dual", FP32_FT_PLAIN_CKPT, "FP32",
                       with_shift=False, size_tag=f"fp32_ft_plain_seed{PRIMARY_SEED}")
    del _mp

if PTQ_OK:
    evaluate_condition(PTQ_MODEL, "ptq_int8", BEST_SEED, "dual", None, "PTQ_INT8", on_cpu_model=True,
                       fp32_reference_model=BEST_FP32_MODEL, size_tag="ptq_int8", with_memory=True)
for sd, qm in QAT_MODELS.items():
    evaluate_condition(qm, "qat_int8", sd, "dual", None, "QAT_INT8", on_cpu_model=True,
                       fp32_reference_model=BEST_FP32_MODEL, size_tag=f"qat_int8_seed{sd}",
                       with_memory=(sd == QAT_DEPLOY_SEED))

# SUPPLEMENTARY PT2E row -- reported for completeness of the deployment story, never entered into an
# RQ2 comparison, because its quantization scope differs from the eager PTQ/QAT pair.
if PTQ_PT2E_OK:
    try:
        evaluate_condition(PTQ_PT2E_MODEL, "ptq_int8_pt2e", BEST_SEED, "dual", None, "PT2E_INT8",
                           on_cpu_model=True, fp32_reference_model=BEST_FP32_MODEL,
                           size_tag="ptq_int8_pt2e")
    except Exception as e:
        print(f"  supplementary PT2E evaluation failed ({e!r}) -- the RQ2 pair is unaffected.")

RAW = pd.DataFrame(rows)
RAW["DualViewGain_G_external"] = RAW.apply(
    lambda r: compute_external_gain(r["QWK"], _indep_mac, _indep_dsc)
    if r["view_mode"] == "dual" else np.nan, axis=1)
RAW.to_csv(f"{METRICS_DIR}/all_conditions_raw.csv", index=False)

if collapse_report:
    pd.DataFrame(collapse_report).to_csv(f"{METRICS_DIR}/prediction_collapse_warnings.csv", index=False)
    print(f"\n{len(collapse_report)} collapse warning(s) written -- see prediction_collapse_warnings.csv")
else:
    print("\nNo prediction-collapse warnings: every model predicts >2 distinct grades with recall above floor.")

# The run must not be able to "succeed" while silently leaving RQ1/RQ2 incomplete.
# Conditions are checked against an EXPECTED list, not against whatever happens to be in RAW. The
# earlier version derived the list from RAW itself, so a total PTQ failure simply removed PTQ from
# the question and the gate still passed without ever answering "FP32 vs PTQ vs QAT".
EXPECTED_RQ1_CONDITIONS = list(CORE_CONDITIONS)
EXPECTED_RQ2_CONDITIONS = ["best_fp32", "fp32_ft_control", "ptq_int8", "qat_int8"]
REQUIRED_RQ1 = ["QWK", "MacroF1", "ShiftL1", "CosAgree", "BenefitCorr"]
REQUIRED_RQ2 = ["QWK", "Accuracy", "MacroF1", "CheckpointSize_MB", "Latency_median_ms",
                "DualViewGain_G_internal"]
_present_conditions = set(RAW.condition)
_missing_conditions = {"RQ1": [c for c in EXPECTED_RQ1_CONDITIONS if c not in _present_conditions],
                       "RQ2": [c for c in EXPECTED_RQ2_CONDITIONS if c not in _present_conditions]}

def _missing_columns(conds, required):
    out = {}
    for c in conds:
        if c not in _present_conditions: continue
        sub = RAW[RAW.condition == c]
        for col in required:
            if col not in sub.columns or sub[col].isna().all():
                out.setdefault(c, []).append(col)
    return out

_missing_rq1 = _missing_columns(EXPECTED_RQ1_CONDITIONS, REQUIRED_RQ1)
_missing_rq2 = _missing_columns(EXPECTED_RQ2_CONDITIONS, REQUIRED_RQ2)
_rq_complete = not (_missing_rq1 or _missing_rq2
                    or _missing_conditions["RQ1"] or _missing_conditions["RQ2"])
record_gate("Gate_RQ_Completeness", _rq_complete,
            f"missing conditions={_missing_conditions} | RQ1 missing columns={_missing_rq1 or 'none'} "
            f"| RQ2 missing columns={_missing_rq2 or 'none'}",
            blocking=not PREFLIGHT)

_all_grades_ok = not any("NEVER predicted" in c["warnings"] for c in collapse_report)
record_gate("Gate3_StudentViability", _all_grades_ok,
            f"{len(collapse_report)} condition(s) flagged; intermediate grades "
            f"{'reachable' if _all_grades_ok else 'COLLAPSED for some conditions'}")

External gain reference: independent macula QWK=0.1373, disc QWK=0.1236
  !! best_fp32|s42|FP32: Grade 4 NEVER predicted; Grade 4 recall 0.000 < 0.05
  !! fp32_ft_control|s42|FP32: Grade 4 NEVER predicted; Grade 4 recall 0.000 < 0.05
  !! fp32_ft_plain|s42|FP32: Grade 4 NEVER predicted; Grade 4 recall 0.000 < 0.05
  !! ptq_int8|s42|PTQ_INT8: Grade 4 NEVER predicted; Grade 4 recall 0.000 < 0.05
  !! qat_int8|s42|QAT_INT8: Grade 4 NEVER predicted; Grade 4 recall 0.000 < 0.05

15 collapse warning(s) written -- see prediction_collapse_warnings.csv
PASS | Gate_RQ_Completeness | missing conditions={'RQ1': [], 'RQ2': []} | RQ1 missing columns=none | RQ2 missing columns=none
FAIL | Gate3_StudentViability | 15 condition(s) flagged; intermediate grades COLLAPSED for some conditions


False

In [36]:
# ---- Aggregate across seeds (mean/SD + median/IQR) ----
NUMERIC = [c for c in RAW.columns if RAW[c].dtype.kind in "fc" and c not in ("seed",)]
agg_mean = RAW.groupby("condition")[NUMERIC].agg(["mean", "std", "median",
                                                   lambda x: x.quantile(.75) - x.quantile(.25)])
agg_mean.columns = ["_".join([a, b if b != "<lambda_0>" else "iqr"]) for a, b in agg_mean.columns]
agg_mean.to_csv(f"{METRICS_DIR}/all_conditions_aggregated.csv")

summary_cols = ["QWK", "Accuracy", "MacroPrecision", "MacroRecall", "MacroF1", "MAE", "SevereErrorRate"]
tbl = RAW.groupby("condition")[summary_cols].agg(["mean", "std"]).round(4)
print(tbl.to_string())

                           QWK     Accuracy     MacroPrecision     MacroRecall     MacroF1         MAE     SevereErrorRate    
                          mean std     mean std           mean std        mean std    mean std    mean std            mean std
condition                                                                                                                     
abl_csd_counterfactual  0.0388 NaN   0.2236 NaN         0.0742 NaN      0.2079 NaN  0.1094 NaN  1.1909 NaN          0.4091 NaN
abl_csd_kl_softmax      0.0933 NaN   0.2382 NaN         0.1978 NaN      0.2447 NaN  0.1536 NaN  1.0818 NaN          0.2800 NaN
abl_csd_raw_smoothl1    0.1086 NaN   0.3927 NaN         0.2128 NaN      0.2200 NaN  0.2098 NaN  1.1255 NaN          0.3709 NaN
best_fp32               0.0848 NaN   0.3818 NaN         0.2134 NaN      0.2163 NaN  0.2054 NaN  1.1145 NaN          0.3582 NaN
disc_only               0.1236 NaN   0.3273 NaN         0.1650 NaN      0.1833 NaN  0.1688 NaN  1.0836 NaN     

## 34 — Statistical analysis

**Hierarchical paired cluster bootstrap over MATCHED seeds.** Each replicate resamples eye-level
clusters with replacement, resamples the **seed pairs** with replacement, and averages the paired
per-seed difference

```
delta_b = (1/S) * sum_s [ M(A_s, b) − M(B_s, b) ]        with seed s the SAME on both sides
```

Two earlier defects are closed by this: (a) it no longer compares CSD's *best* seed against a
baseline's *first* seed — a mismatch that also imported selection bias, since the best seed was
chosen on validation; (b) it no longer draws the two sides' seed indices independently, which turned
a nominally paired difference into an unpaired one (CSD seed 42 against logit-KD seed 8888). Where
two conditions genuinely have **disjoint** seed lists — a 3-seed QAT set against the single
validation-selected FP32 model — pairing falls back to positional cycling and every affected row is
stamped `matched_seeds=False` plus an explanatory `note`.

**p-values come from a paired cluster permutation test**, not from a bootstrap sign proportion.
Within each cluster the two methods' predictions are exchanged at random; the null is exchangeability
of the method labels. Holm correction is applied to the primary QWK comparisons only.

Clustering is at DRTiD's **record/eye** level (its public metadata exposes no patient key) and at
DeepDRiD's **patient** level (documented). Reported as such — never as "patient-clustered" for DRTiD.

In [37]:
from scipy import stats as sps

def _metric_fns():
    return {"QWK": lambda t, p: fast_qwk(t, p),
            "Accuracy": lambda t, p: float((t == p).mean()),
            "MacroF1": lambda t, p: float(f1_score(t, p, average="macro",
                                                    labels=list(range(NUM_CLASSES)), zero_division=0)),
            "MAE": lambda t, p: float(np.mean(np.abs(t - p)))}

def _stack_seed_preds(condition, seeds, quantization="FP32"):
    """Returns (y_true, clusters, {seed: y_pred}) for the seeds actually present."""
    y_true = clusters = None
    preds = {}
    for sd in seeds:
        rec = PRED_STORE.get((condition, sd, quantization))
        if rec is None: continue
        if y_true is None:
            y_true, clusters = rec["y_true"], rec["cluster_ids"]
        else:
            if not np.array_equal(y_true, rec["y_true"]):
                raise AssertionError(f"{condition} seed {sd} has a different sample order -- pairing invalid")
        preds[sd] = rec["y_pred"]
    return y_true, clusters, preds

def _seed_pairs(pa, pb):
    """MATCHED-seed pairing: seed s of A is compared against seed s of B.

    Only when the two seed lists are disjoint -- e.g. a 3-seed QAT set against the single
    validation-selected FP32 model -- do we fall back to positional cycling, and that fallback is
    recorded on every row so it can never be mistaken for a matched comparison.
    """
    common = sorted(set(pa) & set(pb))
    if common:
        return [(sd, sd) for sd in common], True
    la, lb = sorted(pa), sorted(pb)
    n = max(len(la), len(lb))
    return [(la[i % len(la)], lb[i % len(lb)]) for i in range(n)], False

def hierarchical_paired_bootstrap(cond_a, cond_b, seeds_a, seeds_b, B=BOOTSTRAP_B,
                                   alpha=BOOTSTRAP_ALPHA, rng_seed=0, quant_a="FP32", quant_b="FP32"):
    """Hierarchical paired cluster bootstrap over MATCHED seeds.

    One replicate: resample eye-level clusters with replacement, resample the SEED PAIRS with
    replacement, and average the paired per-seed difference

        delta_b = (1/S) * sum_over_sampled_pairs [ M(A_s, b) - M(B_s, b) ]

    The earlier version drew the two conditions' seed indices independently (`ia`, `ib`), so a
    replicate could contrast CSD seed 42 against logit-KD seed 8888 and report the result as paired.
    That both loses power and misdescribes what the interval covers.
    """
    yt_a, cl, pa = _stack_seed_preds(cond_a, seeds_a, quant_a)
    yt_b, _,  pb = _stack_seed_preds(cond_b, seeds_b, quant_b)
    if yt_a is None or yt_b is None or not pa or not pb: return None
    if not np.array_equal(yt_a, yt_b):
        raise AssertionError("paired bootstrap requires identical sample order across conditions")

    pairs, matched = _seed_pairs(pa, pb)
    uniq = np.unique(cl)
    idx_by_cluster = {c: np.where(cl == c)[0] for c in uniq}
    rng = np.random.default_rng(rng_seed)
    fns = _metric_fns()
    diffs = {k: np.empty(B) for k in fns}

    for b in range(B):
        pick = rng.choice(uniq, size=len(uniq), replace=True)
        idx = np.concatenate([idx_by_cluster[c] for c in pick])
        yt = yt_a[idx]
        sel = rng.integers(0, len(pairs), size=len(pairs))     # resample PAIRS, never the two sides apart
        for k, fn in fns.items():
            diffs[k][b] = float(np.mean([fn(yt, pa[pairs[j][0]][idx]) - fn(yt, pb[pairs[j][1]][idx])
                                         for j in sel]))

    out = {}
    for k, d in diffs.items():
        lo, hi = np.percentile(d, [100 * alpha / 2, 100 * (1 - alpha / 2)])
        out[k] = {"mean_diff": float(d.mean()), "ci_low": float(lo), "ci_high": float(hi),
                  "excludes_zero": bool(lo > 0 or hi < 0),
                  "n_seeds_a": len(pa), "n_seeds_b": len(pb),
                  "n_seed_pairs": len(pairs), "matched_seeds": bool(matched)}
    return out

def paired_cluster_permutation_test(cond_a, cond_b, seeds_a, seeds_b, metric="QWK",
                                     P=BOOTSTRAP_B, rng_seed=0, quant_a="FP32", quant_b="FP32"):
    """Exchange the two methods' predictions within each cluster at random. Null = method label is
    exchangeable. Two-sided p with the standard +1 correction so p is never exactly 0. Seeds are
    paired with the SAME rule the bootstrap uses."""
    yt, cl, pa = _stack_seed_preds(cond_a, seeds_a, quant_a)
    _,  _,  pb = _stack_seed_preds(cond_b, seeds_b, quant_b)
    if yt is None or not pa or not pb: return None
    fn = _metric_fns()[metric]

    sp, matched = _seed_pairs(pa, pb)
    pairs = [(pa[a_], pb[b_]) for a_, b_ in sp]
    obs = float(np.mean([fn(yt, xa) - fn(yt, xb) for xa, xb in pairs]))

    uniq = np.unique(cl)
    masks = {c: (cl == c) for c in uniq}
    rng = np.random.default_rng(rng_seed)
    count = 0
    for _ in range(P):
        swap = rng.random(len(uniq)) < 0.5
        sel = np.zeros(len(cl), dtype=bool)
        for c, sw in zip(uniq, swap):
            if sw: sel |= masks[c]
        d = []
        for xa, xb in pairs:
            ya = np.where(sel, xb, xa)
            yb = np.where(sel, xa, xb)
            d.append(fn(yt, ya) - fn(yt, yb))
        if abs(np.mean(d)) >= abs(obs) - 1e-12:
            count += 1
    return {"observed_diff": obs, "p_perm": float((count + 1) / (P + 1)),
            "n_permutations": P, "matched_seeds": bool(matched)}

def holm_correction(pvals, names, alpha=0.05):
    pvals = list(pvals); m = len(pvals)
    order = np.argsort(pvals); adj = [None] * m; running = 0.0
    for rank, i in enumerate(order):
        running = max(running, min(1.0, (m - rank) * pvals[i])); adj[i] = running
    return {names[i]: {"p_raw": float(pvals[i]), "p_holm": float(adj[i]),
                       "significant_holm": bool(adj[i] < alpha)} for i in range(m)}

In [38]:
# Permutations are expensive; keep the preflight cheap but exercise the exact same code path.
_B = 300 if PREFLIGHT else BOOTSTRAP_B
_P = 300 if PREFLIGHT else 5000
QUANT_OF = {"ptq_int8": "PTQ_INT8", "qat_int8": "QAT_INT8", "ptq_int8_pt2e": "PT2E_INT8"}
SEEDS_OF = {"best_fp32": [BEST_SEED], "fp32_ft_control": SEEDS_QAT,
            "fp32_ft_plain": [PRIMARY_SEED],
            "ptq_int8": [BEST_SEED], "qat_int8": SEEDS_QAT, "ptq_int8_pt2e": [BEST_SEED]}

stat_rows, primary_p, primary_names = [], [], []
for rq, pairs in PREREGISTERED_COMPARISONS.items():
    for a, b in pairs:
        sa = SEEDS_OF.get(a, SEEDS_CORE); sb = SEEDS_OF.get(b, SEEDS_CORE)
        qa, qb = QUANT_OF.get(a, "FP32"), QUANT_OF.get(b, "FP32")
        try:
            res = hierarchical_paired_bootstrap(a, b, sa, sb, B=_B, quant_a=qa, quant_b=qb)
        except AssertionError as e:
            print(f"  skip {rq} {a} vs {b}: {e}"); continue
        if res is None:
            print(f"  skip {rq} {a} vs {b}: predictions unavailable"); continue
        perm = paired_cluster_permutation_test(a, b, sa, sb, "QWK", P=_P, quant_a=qa, quant_b=qb)
        for metric, r in res.items():
            row = {"RQ": rq, "comparison": f"{a}_vs_{b}", "metric": metric,
                   "cluster_level": "record_eye(DRTiD)", **r}
            if not r.get("matched_seeds", True):
                row["note"] = "seed lists disjoint -- positional pairing, NOT a matched-seed comparison"
            if metric == "QWK" and perm:
                row.update({"p_perm": perm["p_perm"], "n_permutations": perm["n_permutations"]})
                primary_p.append(perm["p_perm"]); primary_names.append(f"{rq}:{a}_vs_{b}")
            stat_rows.append(row)
        q = res["QWK"]
        print(f"  {rq:4s} {a} vs {b}: dQWK={q['mean_diff']:+.4f} [{q['ci_low']:+.4f},{q['ci_high']:+.4f}] "
              f"seeds={q['n_seeds_a']}v{q['n_seeds_b']}" + (f" p_perm={perm['p_perm']:.4f}" if perm else ""))

STATS = pd.DataFrame(stat_rows)
if primary_p:
    holm = holm_correction(primary_p, primary_names)
    key = STATS.apply(lambda r: f"{r['RQ']}:{r['comparison']}", axis=1)
    STATS["p_holm"] = [holm.get(k, {}).get("p_holm", np.nan) if m == "QWK" else np.nan
                       for k, m in zip(key, STATS["metric"])]
    STATS["significant_holm"] = [holm.get(k, {}).get("significant_holm", None) if m == "QWK" else None
                                 for k, m in zip(key, STATS["metric"])]
STATS.to_csv(f"{TABLES_DIR}/table_05_statistical_tests.csv", index=False)
print(str(len(STATS)) + " comparisons saved (bootstrap B=" + str(_B) + ", permutations=" + str(_P) + ", Holm on primary QWK).")
print("A difference whose CI includes zero is NOT a claim, regardless of the point estimate.")

  RQ1  dual_csd vs dual_no_distill: dQWK=-0.0496 [-0.1044,+0.0076] seeds=1v1 p_perm=0.2625
  RQ1  dual_csd vs dual_logitkd: dQWK=-0.0502 [-0.1033,+0.0066] seeds=1v1 p_perm=0.2558
  RQ1  dual_csd vs dual_featkd: dQWK=-0.0196 [-0.0783,+0.0488] seeds=1v1 p_perm=0.7276
  RQ2  ptq_int8 vs best_fp32: dQWK=+0.0206 [+0.0000,+0.0393] seeds=1v1 p_perm=0.0532
  RQ2  qat_int8 vs best_fp32: dQWK=+0.0472 [+0.0120,+0.0830] seeds=1v1 p_perm=0.0465
  RQ2  qat_int8 vs ptq_int8: dQWK=+0.0266 [-0.0062,+0.0626] seeds=1v1 p_perm=0.2259
  RQ2  qat_int8 vs fp32_ft_control: dQWK=+0.0084 [-0.0115,+0.0291] seeds=1v1 p_perm=0.4020
28 comparisons saved (bootstrap B=300, permutations=300, Holm on primary QWK).
A difference whose CI includes zero is NOT a claim, regardless of the point estimate.


## 35 — DeepDRiD external confirmatory validation (frozen)

Models are **frozen** before this section: no fine-tuning, no threshold tuning, no model selection,
no method change based on what happens here. A drop versus DRTiD is a domain-shift finding to
report, not something to engineer away.

**Documented ambiguity:** DeepDRiD's public CSVs contain no column stating which of `_1`/`_2` is
macula- vs disc-centred (`Field definition` is an image-quality score, not a field-type label).
Rather than bury an assumption, **both orderings are evaluated and both are reported** — turning
the unknown into a small robustness check.

**Partitions are reported separately, not silently pooled.** DeepDRiD ships an official training and
an official validation partition. Both are evaluated on their own, and pooled, giving three external
rows per model per field ordering. The **pre-registered primary external result** is
`subset=validation, field_order=_1=macula`; everything else is supplementary. Pooling is only
legitimate when the partitions do not share patients, so `Gate9b_DeepDRiD_PartitionDisjoint` measures
the patient-ID overlap and every row records `patient_disjoint_partitions` — if they do overlap, the
pooled rows are flagged as non-independent rather than quietly averaged.

In [39]:
class DeepDRiDDualViewDataset(Dataset):
    """One record per EYE, built from the ACTUAL `image_id` / `image_path` values in DeepDRiD's CSV
    rather than from a guessed `pid/pid_l1.jpg` pattern. Paths are resolved by trying the CSV's own
    relative path first and falling back to a filename search, so a different folder layout surfaces
    as "0 usable records" instead of silently mispairing images."""

    def __init__(self, root, subsets=("regular-fundus-training", "regular-fundus-validation"),
                 transform=None, field_order="_1=macula"):
        self.transform = transform or eval_transform
        self.field_order = field_order
        recs, unresolved = [], 0

        for sub in subsets:
            csv = f"{root}/{sub}/{sub}.csv"
            if not os.path.exists(csv):
                continue
            df = pd.read_csv(csv)
            img_root = f"{root}/{sub}/Images"

            # index every image on disk once, by basename -- robust to layout differences
            disk = {}
            for dp, _, fns in os.walk(img_root):
                for fn in fns:
                    if fn.lower().endswith((".jpg", ".jpeg", ".png")):
                        disk.setdefault(os.path.splitext(fn)[0], os.path.join(dp, fn))

            def resolve(image_id, image_path):
                if isinstance(image_path, str):
                    rel = image_path.replace(chr(92), "/").lstrip("/")
                    cand = os.path.join(root, rel)
                    if os.path.exists(cand):
                        return cand
                return disk.get(str(image_id))

            for pid, grp in df.groupby("patient_id"):
                for eye, col in (("l", "left_eye_DR_Level"), ("r", "right_eye_DR_Level")):
                    sub_g = grp[grp["image_id"].astype(str).str.contains(f"_{eye}", regex=False)]
                    if len(sub_g) < 2:
                        continue
                    lvl = sub_g[col].dropna() if col in sub_g else pd.Series(dtype=float)
                    if lvl.empty:
                        continue
                    grade = int(lvl.iloc[0])
                    if not (0 <= grade < NUM_CLASSES):
                        continue
                    # order by the trailing field index in image_id (…_l1 before …_l2)
                    sub_g = sub_g.assign(_k=sub_g["image_id"].astype(str).str[-1]).sort_values("_k")
                    ids = sub_g["image_id"].astype(str).tolist()[:2]
                    paths = [resolve(i, sub_g[sub_g["image_id"].astype(str) == i]["image_path"].iloc[0]
                                     if "image_path" in sub_g else None) for i in ids]
                    if any(pp is None for pp in paths):
                        unresolved += 1
                        continue
                    recs.append({"patient_id": int(pid), "eye": eye,
                                 "img_field1": paths[0], "img_field2": paths[1],
                                 "id_field1": ids[0], "id_field2": ids[1], "grade": grade})
        self.df = pd.DataFrame(recs)
        self.unresolved = unresolved
        if unresolved:
            print(f"  DeepDRiD: {unresolved} eye-record(s) skipped -- image paths could not be resolved")

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        a, b = ((r["img_field1"], r["img_field2"]) if self.field_order == "_1=macula"
                else (r["img_field2"], r["img_field1"]))
        return {"macula": self.transform(image=_rgb(a))["image"],
                "disc":   self.transform(image=_rgb(b))["image"],
                "label":  torch.tensor(int(r["grade"]), dtype=torch.long),
                "cluster_id": int(r["patient_id"])}   # DeepDRiD DOES document patient identity

# DeepDRiD ships two partitions. Reporting them separately is more informative than one pooled
# number, and pooling is only legitimate if the partitions do not share patients -- so that is
# asserted rather than assumed. The PRE-REGISTERED primary external result is the official
# validation partition under the primary field ordering; everything else is supplementary.
DEEPDRID_SUBSETS = {"validation": ("regular-fundus-validation",),
                    "training":   ("regular-fundus-training",),
                    "pooled":     ("regular-fundus-training", "regular-fundus-validation")}
DEEPDRID_PRIMARY_SUBSET = "validation"

EXTERNAL_ROWS = []
if DEEPDRID_ROOT is None:
    print("DeepDRiD not found on Drive -- external validation SKIPPED (reported, not silently omitted).")
    record_gate("Gate9_ExternalValidation", False, "DeepDRiD dataset not present in this runtime")
else:
    ext_models = [("teacher", "-", TEACHER, "FP32", False),
                  ("best_fp32", BEST_SEED, BEST_FP32_MODEL, "FP32", False)]
    if BEST_CONDITION != "dual_csd":
        ext_models.append(("best_csd_fp32", BEST_CSD_SEED, load_student(BEST_CSD_CKPT), "FP32", False))
    if PTQ_OK: ext_models.append(("ptq_int8", BEST_SEED, PTQ_MODEL, "PTQ_INT8", True))
    # The QAT row carries QAT_DEPLOY_SEED -- the seed that model was actually trained with. The
    # earlier code exported QAT_MODELS[sorted(...)[0]] but labelled it BEST_SEED, so the reported
    # seed could name a model that never existed.
    if QAT_OK: ext_models.append(("qat_int8", QAT_DEPLOY_SEED, QAT_MODEL, "QAT_INT8", True))

    _ds_cache = {}
    def _get_ext_ds(name):
        if name not in _ds_cache:
            _ds_cache[name] = DeepDRiDDualViewDataset(DEEPDRID_ROOT, subsets=DEEPDRID_SUBSETS[name],
                                                      transform=eval_transform)
        return _ds_cache[name]

    # Pooling check: are the two official partitions patient-disjoint?
    _pat = {}
    for _n in ("training", "validation"):
        _d = _get_ext_ds(_n)
        _pat[_n] = set(_d.df.patient_id.tolist()) if len(_d.df) else set()
    _overlap = _pat["training"] & _pat["validation"]
    record_gate("Gate9b_DeepDRiD_PartitionDisjoint", len(_overlap) == 0,
                f"training={len(_pat['training'])} patients, validation={len(_pat['validation'])} "
                f"patients, overlap={len(_overlap)}"
                + ("" if not _overlap else " -- POOLED rows are NOT independent and must be labelled so"))

    for sub_name in DEEPDRID_SUBSETS:
        ds = _get_ext_ds(sub_name)
        if len(ds) == 0:
            print(f"  DeepDRiD[{sub_name}]: 0 usable eye-records"); continue
        for order in DEEPDRID_FIELD_ORDERS:
            ds.field_order = order                      # read inside __getitem__, so this is enough
            ld = make_loader(ds, 16, False, workers=2)
            is_primary = (order == DEEPDRID_PRIMARY_FIELD_ORDER and sub_name == DEEPDRID_PRIMARY_SUBSET)
            print(f"\nDeepDRiD [{sub_name} | {order}]: {len(ds)} eyes, "
                  f"{ds.df.patient_id.nunique()} patients, grades {sorted(ds.df.grade.unique().tolist())}"
                  + ("   <-- PRE-REGISTERED PRIMARY" if is_primary else ""))
            for name, seed, mdl, quant, on_cpu in ext_models:
                if on_cpu:
                    r = _predict_cpu_all(mdl, ld)
                    yt, yp, pc, pid = (r["y_true"], r["y_pred_dual"], r["p_dual"], r["cluster_ids"])
                else:
                    yt, yp, pc, pid = get_predictions(mdl, ld, DEVICE, "dual", return_clusters=True)
                m = compute_all_metrics(yt, yp, pc)
                EXTERNAL_ROWS.append({"condition": name, "seed": seed, "quantization": quant,
                                      "subset": sub_name, "field_order": order,
                                      "role": "PRIMARY" if is_primary else "supplementary",
                                      "patient_disjoint_partitions": bool(not _overlap),
                                      "n_eyes": len(ds), "n_patients": int(ds.df.patient_id.nunique()), **m})
                _tag = f"external_{sub_name}_{order.replace('=', '')}"
                save_predictions(name, seed, yt, yp, pc, pid, quant, dataset="DeepDRiD", split=_tag)
                save_confusion(name, seed, yt, yp, tag=f"_DeepDRiD_{sub_name}_{order.replace('=', '')}")
                w = check_prediction_collapse(yt, yp)
                print(f"  {name:14s} QWK={m['QWK']:.4f} Acc={m['Accuracy']:.4f} MacroF1={m['MacroF1']:.4f}"
                      + (f"  !! {'; '.join(w)}" if w else ""))

    if EXTERNAL_ROWS:
        EXT = pd.DataFrame(EXTERNAL_ROWS)
        EXT.to_csv(f"{TABLES_DIR}/table_06_external_validation_deepdrid.csv", index=False)
        record_gate("Gate9_ExternalValidation", True,
                    f"{EXT.condition.nunique()} models x {EXT.subset.nunique()} partitions x "
                    f"{EXT.field_order.nunique()} field orders; primary = "
                    f"{DEEPDRID_PRIMARY_SUBSET}/{DEEPDRID_PRIMARY_FIELD_ORDER}")
    else:
        record_gate("Gate9_ExternalValidation", False, "no usable DeepDRiD records assembled")

EXT_DF = pd.DataFrame(EXTERNAL_ROWS) if EXTERNAL_ROWS else pd.DataFrame()

PASS | Gate9b_DeepDRiD_PartitionDisjoint | training=299 patients, validation=100 patients, overlap=0

DeepDRiD [validation | _1=macula]: 200 eyes, 100 patients, grades [0, 1, 2, 3, 4]   <-- PRE-REGISTERED PRIMARY
  teacher        QWK=0.7679 Acc=0.5500 MacroF1=0.4248
  best_fp32      QWK=0.2258 Acc=0.3100 MacroF1=0.1904  !! Grade 3 recall 0.000 < 0.05; Grade 4 NEVER predicted; Grade 4 recall 0.000 < 0.05
  best_csd_fp32  QWK=0.0777 Acc=0.2100 MacroF1=0.1185  !! Grade 3 NEVER predicted; Grade 3 recall 0.000 < 0.05; Grade 4 NEVER predicted; Grade 4 recall 0.000 < 0.05
  ptq_int8       QWK=0.2259 Acc=0.3050 MacroF1=0.1849  !! Grade 3 recall 0.000 < 0.05; Grade 4 NEVER predicted; Grade 4 recall 0.000 < 0.05
  qat_int8       QWK=0.1751 Acc=0.2650 MacroF1=0.1748  !! Grade 3 recall 0.029 < 0.05; Grade 4 NEVER predicted; Grade 4 recall 0.000 < 0.05

DeepDRiD [validation | _1=disc]: 200 eyes, 100 patients, grades [0, 1, 2, 3, 4]
  teacher        QWK=0.7671 Acc=0.5500 MacroF1=0.4220  !! Grade 1 r

## 36–37 — Deployment export, parity checks & inference wrapper (Gate 8)

TorchScript is deprecated and is **not** used as the deployment path. Artifacts:
`checkpoint.pt` (state_dict), `model.pt2` (`torch.export`), `model.onnx`, `metadata.json`, plus
`model_object.pt` for INT8 models only. Export failures are reported as failures — never silently
downgraded to "gate passed".

**Quantized models face the same reload test as FP32.** Every model is rebuilt by a builder and then
loaded from `checkpoint.pt`. For INT8 the builder reconstructs the *quantized skeleton*
(fuse → prepare → convert) and `load_state_dict()` restores the packed INT8 weights with their
scales and zero-points — the supported way to restore a quantized model. Whole-module pickling is
kept only as a fallback, because eager quantized modules do not reliably round-trip through pickle
(reloading one raises `AttributeError: 'ConvReLU2d' object has no attribute '_modules'`).

The artifact on disk is loaded into a fresh object, checked for numeric parity against the in-memory
reference, checked for a valid grade range, and run 100 times. `Gate8b_ArtifactReload` reports
`deployment_verified` per model; the earlier version passed `builder=None` for INT8 and recorded
`reloaded_from_disk=False`, which meant PTQ/QAT were never demonstrated to be deployable at all.

In [40]:
class DRVergeInference(nn.Module):
    """Export-friendly wrapper: two images in, cumulative threshold scores out."""
    def __init__(self, model): super().__init__(); self.model = model
    def forward(self, macula, disc): return self.model(macula, disc)["p_dual"]

def export_model(model, name, on_cpu_model=False, extra_meta=None, save_object=False):
    d = f"{MODELS_DIR}/{name}"; os.makedirs(d, exist_ok=True)
    m = (model if on_cpu_model else copy.deepcopy(model).to("cpu")).eval()
    ex_m, ex_d = _sample["macula"][:1].cpu(), _sample["disc"][:1].cpu()
    status = {"state_dict": False, "torch_export": False, "onnx": False,
              "onnx_parity_max_abs_diff": None, "export_error": None, "onnx_error": None}

    try:
        robust_torch_save(m.state_dict(), f"{d}/checkpoint.pt"); status["state_dict"] = True
    except Exception as e:
        status["export_error"] = f"state_dict: {e!r}"

    # Best-effort full pickled module, saved only for quantized models. It is a FALLBACK, not the
    # primary reload path: eager quantized modules do not reliably round-trip through plain pickle
    # (loading one raises AttributeError: 'ConvReLU2d' object has no attribute '_modules'), which is
    # why the primary path rebuilds the quantized skeleton and loads checkpoint.pt into it.
    status["model_object"] = False
    if save_object:
        try:
            robust_torch_save(m, f"{d}/model_object.pt"); status["model_object"] = True
        except Exception as e:
            status["model_object_error"] = repr(e)

    wrapper = DRVergeInference(m).eval()
    with torch.no_grad():
        ref = wrapper(ex_m, ex_d)

    try:
        ep = torch.export.export(wrapper, (ex_m, ex_d))
        torch.export.save(ep, f"{d}/model.pt2"); status["torch_export"] = True
    except Exception as e:
        status["export_error"] = f"torch.export: {e!r}"
        print(f"  [{name}] torch.export FAILED: {e!r}")

    try:
        torch.onnx.export(wrapper, (ex_m, ex_d), f"{d}/model.onnx",
                          input_names=["macula", "disc"], output_names=["p_cumulative"], dynamo=True)
        status["onnx"] = True
        try:
            import onnxruntime as ort
            sess = ort.InferenceSession(f"{d}/model.onnx", providers=["CPUExecutionProvider"])
            got = sess.run(None, {"macula": ex_m.numpy(), "disc": ex_d.numpy()})[0]
            status["onnx_parity_max_abs_diff"] = float(np.max(np.abs(got - ref.numpy())))
        except Exception as e:
            status["onnx_error"] = f"parity: {e!r}"
    except Exception as e:
        status["onnx_error"] = f"export: {e!r}"
        print(f"  [{name}] ONNX export FAILED: {e!r}")

    meta = {"model_name": name, "architecture": type(m).__name__, "dataset": "DRTiD",
            "grade_mapping": list(range(NUM_CLASSES)), "num_thresholds": NUM_THRESHOLDS,
            **PREPROCESSING_META, "torch_version": torch.__version__,
            "torchao_version": ENVIRONMENT.get("torchao"), "git_commit": ENVIRONMENT["git_commit"],
            "artifacts": {k: v for k, v in status.items()}, **(extra_meta or {})}
    save_json(meta, f"{d}/metadata.json")
    ok = status["state_dict"]
    print(f"  [{name}] state_dict={status['state_dict']} pt2={status['torch_export']} "
          f"onnx={status['onnx']} parity={status['onnx_parity_max_abs_diff']}")
    return d, status, meta

EXPORTS = {}
EXPORTS["teacher_fp32"] = export_model(TEACHER, "teacher_fp32",
    extra_meta={"role": "upper_bound_teacher", "training_seed": PRIMARY_SEED})
EXPORTS["best_student_fp32"] = export_model(BEST_FP32_MODEL, "best_student_fp32",
    extra_meta={"role": "deployment_candidate", "condition": BEST_CONDITION,
                "training_seed": BEST_SEED, "best_val_qwk": float(BEST_ROW["QWK"]), "quantization": "FP32"})
if BEST_CONDITION != "dual_csd":
    EXPORTS["best_csd_fp32"] = export_model(load_student(BEST_CSD_CKPT), "best_csd_fp32",
        extra_meta={"role": "best_csd_artifact", "condition": "dual_csd",
                    "training_seed": BEST_CSD_SEED, "quantization": "FP32"})
if PTQ_OK:
    EXPORTS["best_student_ptq_int8"] = export_model(PTQ_MODEL, "best_student_ptq_int8", on_cpu_model=True,
        save_object=True, extra_meta={"role": "deployment_int8", "quantization": "PTQ_INT8", **PTQ_INFO})
if QAT_OK:
    EXPORTS["best_student_qat_int8"] = export_model(QAT_MODEL, "best_student_qat_int8", on_cpu_model=True,
        save_object=True,
        extra_meta={"role": "deployment_int8", "quantization": "QAT_INT8",
                    "training_seed": QAT_DEPLOY_SEED,
                    "deploy_seed_selected_on": "DRTiD validation QWK", **QAT_INFO})

_export_ok = all(s["state_dict"] for _, s, _ in EXPORTS.values())
_pt2_ok = sum(1 for _, s, _ in EXPORTS.values() if s["torch_export"])
record_gate("Gate8_Export", _export_ok,
            f"{len(EXPORTS)} models; state_dict all={_export_ok}; torch.export ok for {_pt2_ok}/{len(EXPORTS)}")

  [teacher_fp32] ONNX export FAILED: ModuleNotFoundError("No module named 'onnxscript'")
  [teacher_fp32] state_dict=True pt2=True onnx=False parity=None
  [best_student_fp32] ONNX export FAILED: ModuleNotFoundError("No module named 'onnxscript'")
  [best_student_fp32] state_dict=True pt2=True onnx=False parity=None
  [best_csd_fp32] ONNX export FAILED: ModuleNotFoundError("No module named 'onnxscript'")
  [best_csd_fp32] state_dict=True pt2=True onnx=False parity=None
  [best_student_ptq_int8] torch.export FAILED: AttributeError("__torch__.torch.classes.quantized.Conv2dPackedParamsBase (of Python compilation unit at: 0) does not have a field with name '__obj_flatten__'")
  [best_student_ptq_int8] ONNX export FAILED: ModuleNotFoundError("No module named 'onnxscript'")
  [best_student_ptq_int8] state_dict=True pt2=False onnx=False parity=None
  [best_student_qat_int8] torch.export FAILED: AttributeError("__torch__.torch.classes.quantized.Conv2dPackedParamsBase (of Python compilation uni

True

In [41]:
# ---- Deployment verification (spec 59): reload from disk and confirm it still behaves ----
def verify_deployment(model_dir, builder, on_cpu_reference=None, n_runs=100):
    """Proves the SAVED ARTIFACT works -- not the object still alive in RAM.

    Every model -- FP32 and INT8 alike -- is rebuilt by a `builder()` and then loaded from
    `checkpoint.pt`. For INT8 the builder reconstructs the *quantized skeleton* (fuse -> prepare ->
    convert), which is the supported way to restore a quantized model: `convert()` produces the
    module tree, `load_state_dict()` restores the packed INT8 weights and their qparams. Plain
    pickling of a quantized module is only a fallback here, because it does not reliably round-trip
    (loading one raises AttributeError: 'ConvReLU2d' object has no attribute '_modules').

    Either way the check is the same: load from disk into a fresh object, compare numerically against
    the in-memory reference, confirm the predicted grade is in range, and run repeatedly. The earlier
    version passed `builder=None` for quantized models and recorded `reloaded_from_disk = False`,
    which meant PTQ/QAT were never shown to be deployment-ready at all.
    """
    checks = {}
    ex_m, ex_d = _sample_pair[0][:1].cpu(), _sample_pair[1][:1].cpu()
    ck  = f"{model_dir}/checkpoint.pt"
    obj = f"{model_dir}/model_object.pt"
    checks["checkpoint_exists"] = os.path.exists(ck)
    checks["model_object_exists"] = os.path.exists(obj)

    ref_out = None
    if on_cpu_reference is not None:
        with torch.no_grad():
            o = on_cpu_reference(ex_m, ex_d)
            ref_out = (o["p_dual"] if isinstance(o, dict) else o).cpu()

    fresh, how = None, None
    if builder is not None and checks["checkpoint_exists"]:
        try:
            fresh = builder()                                   # brand new object, no shared state
            state = robust_torch_load(ck, map_location="cpu")
            fresh.load_state_dict(state if not isinstance(state, dict) or "model_state" not in state
                                  else state["model_state"])
            how = "rebuilt skeleton + state_dict from disk"
        except Exception as e:
            fresh = None
            checks["builder_reload_error"] = repr(e)
    if fresh is None and checks["model_object_exists"]:
        try:
            fresh = robust_torch_load(obj, map_location="cpu")   # fallback: whole pickled module
            how = "pickled module artifact (fallback)"
        except Exception as e:
            checks["reload_error"] = repr(e)

    if fresh is not None:
        try:
            fresh.eval()
            with torch.no_grad():
                o = fresh(ex_m, ex_d)
                got = (o["p_dual"] if isinstance(o, dict) else o).cpu()
            checks["reloaded_from_disk"] = True
            checks["reload_method"] = how
            g = int((got > 0.5).sum())
            checks["grade_in_range"] = bool(0 <= g <= NUM_CLASSES - 1)
            if ref_out is not None:
                checks["reload_max_abs_diff"] = float(torch.max(torch.abs(got - ref_out)))
                checks["reload_parity_ok"] = checks["reload_max_abs_diff"] < 1e-5
            with torch.no_grad():
                for _ in range(n_runs): fresh(ex_m, ex_d)
            checks["stable_over_100_runs"] = True
        except Exception as e:
            checks["reloaded_from_disk"] = False
            checks["reload_error"] = repr(e)
    else:
        checks["reloaded_from_disk"] = False
        checks.setdefault("reload_error", "neither checkpoint.pt+builder nor model_object.pt usable")

    pt2 = f"{model_dir}/model.pt2"
    if os.path.exists(pt2) and ref_out is not None:
        try:
            loaded = torch.export.load(pt2)
            with torch.no_grad(): got = loaded.module()(ex_m, ex_d)
            got = got["p_dual"] if isinstance(got, dict) else got
            checks["pt2_reload_max_abs_diff"] = float(torch.max(torch.abs(got.cpu() - ref_out)))
            checks["pt2_parity_ok"] = checks["pt2_reload_max_abs_diff"] < 1e-4
        except Exception as e:
            checks["pt2_parity_ok"] = False; checks["pt2_error"] = repr(e)

    checks["deployment_verified"] = bool(checks.get("reloaded_from_disk")
                                         and checks.get("stable_over_100_runs")
                                         and checks.get("grade_in_range", False)
                                         and checks.get("reload_parity_ok", True))
    return checks

def build_ptq_skeleton():
    """An architecturally identical INT8 model with placeholder qparams. Loading the saved
    state_dict into it restores the real packed weights, scales and zero-points."""
    m = _fresh_student_from(BEST_FP32_CKPT)
    m.backbone.fuse_model(qat=False)
    m.backbone = QuantizableBackbone(m.backbone)
    m.backbone.qconfig = get_default_qconfig(QUANT_ENGINE)
    prep = prepare(m, inplace=False)
    _calibrate(prep, 1)          # observers must see one batch before convert() can build the tree
    return convert(prep, inplace=False)

def build_qat_skeleton():
    m = _build_qat_prepared(BEST_FP32_CKPT, load_fp32=True)
    return convert(m.to("cpu").eval(), inplace=False)

DEPLOY_CHECKS = {}
_fp32_cpu = copy.deepcopy(BEST_FP32_MODEL).to("cpu").eval()
DEPLOY_CHECKS["best_student_fp32"] = verify_deployment(
    EXPORTS["best_student_fp32"][0],
    builder=lambda: DualViewLightStudent(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS),
    on_cpu_reference=_fp32_cpu)
if PTQ_OK:
    DEPLOY_CHECKS["best_student_ptq_int8"] = verify_deployment(
        EXPORTS["best_student_ptq_int8"][0], builder=build_ptq_skeleton, on_cpu_reference=PTQ_MODEL)
if QAT_OK:
    DEPLOY_CHECKS["best_student_qat_int8"] = verify_deployment(
        EXPORTS["best_student_qat_int8"][0], builder=build_qat_skeleton, on_cpu_reference=QAT_MODEL)

record_gate("Gate8b_ArtifactReload",
            bool(DEPLOY_CHECKS) and all(v.get("deployment_verified", False)
                                        for v in DEPLOY_CHECKS.values()),
            "; ".join(f"{k}={'verified' if v.get('deployment_verified') else 'NOT verified'}"
                      f" via {v.get('reload_method', 'n/a')}" for k, v in DEPLOY_CHECKS.items()))
save_json(DEPLOY_CHECKS, f"{RESULTS_DIR}/deployment_verification.json")
print(json.dumps(DEPLOY_CHECKS, indent=2, default=str))

/usr/local/lib/python3.12/dist-packages/torch/export/pt2_archive/_package.py:790: UserWarning: The given buffer is not writable, and PyTorch does not support non-writable tensors. This means you can write to the underlying (supposedly non-writable) buffer using the tensor. You may want to copy the buffer to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:1586.)
  tensor = torch.frombuffer(
/tmp/ipykernel_1065/239569499.py:95: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migra

PASS | Gate8b_ArtifactReload | best_student_fp32=verified via rebuilt skeleton + state_dict from disk; best_student_ptq_int8=verified via rebuilt skeleton + state_dict from disk; best_student_qat_int8=verified via rebuilt skeleton + state_dict from disk
{
  "best_student_fp32": {
    "checkpoint_exists": true,
    "model_object_exists": false,
    "reloaded_from_disk": true,
    "reload_method": "rebuilt skeleton + state_dict from disk",
    "grade_in_range": true,
    "reload_max_abs_diff": 0.0,
    "reload_parity_ok": true,
    "stable_over_100_runs": true,
    "pt2_reload_max_abs_diff": 0.0,
    "pt2_parity_ok": true,
    "deployment_verified": true
  },
  "best_student_ptq_int8": {
    "checkpoint_exists": true,
    "model_object_exists": true,
    "reloaded_from_disk": true,
    "reload_method": "rebuilt skeleton + state_dict from disk",
    "grade_in_range": true,
    "reload_max_abs_diff": 0.0,
    "reload_parity_ok": true,
    "stable_over_100_runs": true,
    "deployment_verif

In [42]:
# ---- Final inference interface (spec 21) -- what a web prototype would call ----
GRADE_NAMES = {0: "No DR", 1: "Mild NPDR", 2: "Moderate NPDR", 3: "Severe NPDR", 4: "Proliferative DR"}

_INFER_CACHE = {}

def get_inference_model(device="cpu"):
    """Load once, reuse. A server should call this at startup, not per request."""
    if device not in _INFER_CACHE:
        _INFER_CACHE[device] = copy.deepcopy(BEST_FP32_MODEL).to(device).eval()
    return _INFER_CACHE[device]

def predict_dr(macula_image, optic_disc_image, model=None, model_version=None, device="cpu"):
    """macula_image / optic_disc_image: HxWx3 uint8 RGB arrays or file paths."""
    version = model_version or f"{BEST_CONDITION}_seed{BEST_SEED}_FP32"
    m = (model.to(device).eval() if model is not None else get_inference_model(device))
    def prep(x):
        arr = _rgb(x) if isinstance(x, str) else np.asarray(x)
        return eval_transform(image=arr)["image"].unsqueeze(0).to(device)
    a, b = prep(macula_image), prep(optic_disc_image)
    t0 = time.perf_counter()
    with torch.no_grad():
        p = m(a, b)["p_dual"][0]
    dt = (time.perf_counter() - t0) * 1000
    cum = p.cpu().numpy()
    grade = int((cum > 0.5).sum())
    # cumulative P(y>k) -> per-class probabilities
    ext = np.concatenate([[1.0], cum, [0.0]])
    probs = np.clip(ext[:-1] - ext[1:], 0, None); probs = probs / max(probs.sum(), 1e-9)
    return {"predicted_grade": grade, "predicted_label": GRADE_NAMES[grade],
            "cumulative_threshold_scores": cum.tolist(), "grade_probabilities": probs.tolist(),
            # NOT a calibrated clinical probability -- weighted BCE training distorts the sigmoid
            # outputs, so this is a derived score. See the calibration section (ECE/Brier).
            "uncalibrated_confidence": float(probs[grade]), "model_version": version,
            "inference_time_ms": dt, "preprocessing": PREPROCESSING_META}

_demo = pd.read_csv(DRTID_TEST_CSV).iloc[0]
_out = predict_dr(_demo["macula_path"], _demo["disc_path"])
print("predict_dr() demo ->", json.dumps({k: v for k, v in _out.items() if k != "preprocessing"},
                                          indent=2, default=str))
print("true grade:", int(_demo["grade"]))

predict_dr() demo -> {
  "predicted_grade": 2,
  "predicted_label": "Moderate NPDR",
  "cumulative_threshold_scores": [
    0.672029435634613,
    0.5884768962860107,
    0.2807667553424835,
    0.07143926620483398
  ],
  "grade_probabilities": [
    0.32797056436538696,
    0.0835525393486023,
    0.3077101409435272,
    0.20932748913764954,
    0.07143926620483398
  ],
  "uncalibrated_confidence": 0.3077101409435272,
  "model_version": "dual_featkd_seed42_FP32",
  "inference_time_ms": 14.576669000234688
}
true grade: 3


In [43]:
# ---- Pre-registered DEPLOYMENT selection rule (declared before test results are read) ----
DEPLOY_RULE = {"min_qwk_retention_pct": 95.0, "severe_error_must_not_credibly_worsen": True,
               "tiebreak": "lowest median CPU latency", "fallback": "best_fp32"}
save_json(DEPLOY_RULE, f"{CONFIG_DIR}/deployment_selection_rule.json")

def choose_deployment_model():
    base = RAW[RAW.condition == "best_fp32"]
    if not len(base): return "best_fp32", "no FP32 reference available"
    q0, l0 = base["QWK"].mean(), base["Latency_median_ms"].mean()
    cands = []
    for c in ["ptq_int8", "qat_int8"]:
        sub = RAW[RAW.condition == c]
        if not len(sub): continue
        ret = 100.0 * sub["QWK"].mean() / q0 if q0 else float("nan")
        worse = sub["SevereErrorRate"].mean() > base["SevereErrorRate"].mean()
        st = STATS[(STATS.comparison == f"{c}_vs_best_fp32") & (STATS.metric == "QWK")]
        credible_drop = bool(len(st) and st.iloc[0]["excludes_zero"] and st.iloc[0]["mean_diff"] < 0)
        if ret >= DEPLOY_RULE["min_qwk_retention_pct"] and not (worse and credible_drop):
            cands.append((sub["Latency_median_ms"].mean(), c, ret))
    if not cands:
        return "best_fp32", f"no INT8 variant met >={DEPLOY_RULE['min_qwk_retention_pct']}% QWK retention"
    cands.sort()
    return cands[0][1], f"retention {cands[0][2]:.1f}%, lowest latency {cands[0][0]:.2f} ms"

DEPLOY_CHOICE, DEPLOY_REASON = choose_deployment_model()
print(f"Deployment model (pre-registered rule): {DEPLOY_CHOICE} -- {DEPLOY_REASON}")
save_json({"chosen": DEPLOY_CHOICE, "reason": DEPLOY_REASON, "rule": DEPLOY_RULE},
          f"{RESULTS_DIR}/deployment_choice.json")

# ---- Model registry (spec 58) ----
reg = []
_ext_lookup = {}
if len(EXT_DF):
    _prim_ext = EXT_DF[(EXT_DF.field_order == DEEPDRID_PRIMARY_FIELD_ORDER) &
                       (EXT_DF.subset == DEEPDRID_PRIMARY_SUBSET)]
    for _, r in _prim_ext.iterrows():
        _ext_lookup[r["condition"]] = r["QWK"]

for name, (d, status, meta) in EXPORTS.items():
    cond = meta.get("condition", meta.get("role", name))
    quant = meta.get("quantization", "FP32")
    match = RAW[(RAW.condition == {"teacher_fp32": "teacher", "best_student_fp32": "best_fp32",
                                   "best_student_ptq_int8": "ptq_int8", "best_student_qat_int8": "qat_int8",
                                   "best_csd_fp32": "dual_csd"}.get(name, name))]
    if name == "best_csd_fp32": match = match[match.seed == BEST_CSD_SEED]
    if name == "best_student_qat_int8" and QAT_DEPLOY_SEED is not None:
        match = match[match.seed == QAT_DEPLOY_SEED]
    row = match.iloc[0] if len(match) else None
    reg.append({
        "model_id": name, "condition": cond, "seed": meta.get("training_seed", "-"),
        "checkpoint_path": f"{d}/checkpoint.pt",
        "pt2_path": f"{d}/model.pt2" if status["torch_export"] else "",
        "onnx_path": f"{d}/model.onnx" if status["onnx"] else "",
        "val_qwk": meta.get("best_val_qwk", np.nan),
        "test_qwk": float(row["QWK"]) if row is not None else np.nan,
        "test_macro_f1": float(row["MacroF1"]) if row is not None else np.nan,
        "test_accuracy": float(row["Accuracy"]) if row is not None else np.nan,
        "external_qwk": _ext_lookup.get({"teacher_fp32": "teacher", "best_student_fp32": "best_fp32",
                                          "best_student_ptq_int8": "ptq_int8",
                                          "best_student_qat_int8": "qat_int8"}.get(name, name), np.nan),
        "params": int(row["ParamCount"]) if row is not None and not pd.isna(row.get("ParamCount")) else np.nan,
        "size_mb": file_size_mb(f"{d}/checkpoint.pt"),
        "latency_median_ms": float(row["Latency_median_ms"]) if row is not None and "Latency_median_ms" in row else np.nan,
        "quantization": quant,
        "deployment_artifact_size_mb": (file_size_mb(f"{d}/model.pt2") if status["torch_export"]
                                        else file_size_mb(f"{d}/model_object.pt")),
        "deployable": bool(status["state_dict"] and (name not in DEPLOY_CHECKS or
                                                     DEPLOY_CHECKS[name].get("deployment_verified", False))),
    })
REGISTRY = pd.DataFrame(reg)
REGISTRY.to_csv(f"{ART}/model_registry.csv", index=False)
print(REGISTRY.to_string(index=False))

Deployment model (pre-registered rule): qat_int8 -- retention 154.4%, lowest latency 10.35 ms
             model_id           condition seed                                                                                   checkpoint_path                                                                                  pt2_path onnx_path  val_qwk  test_qwk  test_macro_f1  test_accuracy  external_qwk   params    size_mb  latency_median_ms quantization  deployment_artifact_size_mb  deployable
         teacher_fp32 upper_bound_teacher   42          /content/drive/MyDrive/DR-VERGE/artifacts_preflight_v1/models/teacher_fp32/checkpoint.pt      /content/drive/MyDrive/DR-VERGE/artifacts_preflight_v1/models/teacher_fp32/model.pt2                NaN  0.607735       0.344319       0.501818      0.767931 40322124 154.125275         247.299846         FP32                   173.470682        True
    best_student_fp32         dual_featkd   42     /content/drive/MyDrive/DR-VERGE/artifacts_preflight_v

## 38 — Figures & companion CSVs

Every figure is written as **PNG (400 dpi) + PDF + SVG**, and every figure ships a
`*_data.csv` with exactly the numbers plotted. No value exists only inside an image.

In [44]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 400, "font.size": 11,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True,
                     "figure.autolayout": False})

def save_figure(fig, stem, data_df, caption=""):
    for ext in ("png", "pdf", "svg"):
        fig.savefig(f"{FIGURES_DIR}/{stem}.{ext}", bbox_inches="tight")
    data_df.to_csv(f"{FIGURES_DIR}/{stem}_data.csv", index=False)
    if caption:
        with open(f"{FIGURES_DIR}/{stem}_caption.txt", "w") as f: f.write(caption)
    plt.close(fig)
    print(f"  saved {stem} (.png/.pdf/.svg + _data.csv)")

def order_present(order, df=None):
    df = RAW if df is None else df
    return [c for c in order if c in set(df["condition"])]

DISPLAY_ORDER = ["teacher", "macula_only", "disc_only", "dual_no_distill", "dual_logitkd",
                 "dual_featkd", "dual_csd", "abl_csd_raw_smoothl1", "abl_csd_kl_softmax",
                 "abl_csd_counterfactual", "best_fp32", "fp32_ft_control", "fp32_ft_plain",
                 "ptq_int8", "qat_int8", "ptq_int8_pt2e"]

# PRECISE METHOD NAMES. `dual_featkd` and `dual_csd` BOTH sit on top of the tuned logit-KD baseline
# (each adds one extra term to it), so labelling them "Feature-KD" and "CSD" would misdescribe the
# comparison. The controlled ladder actually being tested is:
#     no distillation  ->  logit-KD  ->  logit-KD + feature-KD  ->  logit-KD + CSD
# The paper must say "CSD augmentation and feature-distillation augmentation over a standard logit-KD
# baseline", never a bare "CSD vs Feature-KD".
METHOD_LABELS = {
    "teacher": "Teacher (ResNet-50, dual-view)",
    "macula_only": "Student, macula only",
    "disc_only": "Student, optic-disc only",
    "dual_no_distill": "Dual-view, no distillation",
    "dual_logitkd": "Logit-KD",
    "dual_featkd": "Logit-KD + Feature-KD",
    "dual_csd": "Logit-KD + CSD (proposed)",
    "abl_csd_raw_smoothl1": "CSD ablation: unscaled Huber",
    "abl_csd_kl_softmax": "CSD ablation: KL-softmax (negative control)",
    "abl_csd_counterfactual": "CSD ablation: same-head counterfactual",
    "best_fp32": "M* (FP32)",
    "fp32_ft_control": "FP32 fine-tune control (matched graph)",
    "fp32_ft_plain": "FP32 fine-tune control (plain, secondary)",
    "ptq_int8": "PTQ INT8",
    "qat_int8": "QAT INT8",
    "ptq_int8_pt2e": "PT2E PTQ INT8 (supplementary)",
}
def pretty(c):
    return METHOD_LABELS.get(c, c)

pd.DataFrame([{"condition": k, "paper_label": v} for k, v in METHOD_LABELS.items()]).to_csv(
    f"{TABLES_DIR}/table_condition_labels.csv", index=False)

def agg_stat(df, conds, col):
    means, sds, ns = [], [], []
    for c in conds:
        v = df[df.condition == c][col].dropna()
        means.append(v.mean() if len(v) else np.nan)
        sds.append(v.std() if len(v) > 1 else 0.0)
        ns.append(len(v))
    return np.array(means), np.array(sds), np.array(ns)

In [45]:
# ---- Figure 1: architecture / workflow schematic (hero figure) ----
fig, ax = plt.subplots(figsize=(11, 6)); ax.axis("off"); ax.grid(False)
def box(x, y, w, h, txt, fc):
    ax.add_patch(plt.Rectangle((x, y), w, h, facecolor=fc, edgecolor="#333", lw=1.4, zorder=2))
    ax.text(x + w/2, y + h/2, txt, ha="center", va="center", fontsize=10, zorder=3)
def arrow(x1, y1, x2, y2):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="->", lw=1.6, color="#333"))

box(0.02, 0.72, 0.15, 0.10, "Macula view", "#DCE9F7"); box(0.02, 0.58, 0.15, 0.10, "Optic-disc view", "#DCE9F7")
box(0.24, 0.62, 0.20, 0.20, "Teacher\nResNet-50 dual-view\n+ CORAL heads", "#F7DCDC")
box(0.50, 0.72, 0.24, 0.10, "p_dual, p_macula, p_disc", "#FFF3CD")
box(0.50, 0.58, 0.24, 0.10, r"$\Delta^T=p_{dual}-p_{agg}$", "#FFF3CD")
box(0.24, 0.30, 0.20, 0.18, "Lightweight student\ndepthwise-separable\n+ InteractionFusion", "#DCF7E3")
box(0.50, 0.34, 0.24, 0.10, "L = task + aux\n+ α·KD + β·CSD", "#FFF3CD")
box(0.80, 0.46, 0.17, 0.09, "Best FP32 (M*)", "#DCF7E3")
box(0.80, 0.32, 0.17, 0.09, "PTQ INT8", "#E8DCF7"); box(0.80, 0.18, 0.17, 0.09, "QAT INT8", "#E8DCF7")
box(0.02, 0.30, 0.15, 0.10, "Macula view", "#DCE9F7"); box(0.02, 0.16, 0.15, 0.10, "Optic-disc view", "#DCE9F7")
arrow(0.17, 0.77, 0.24, 0.74); arrow(0.17, 0.63, 0.24, 0.68)
arrow(0.44, 0.74, 0.50, 0.77); arrow(0.44, 0.68, 0.50, 0.63)
arrow(0.17, 0.35, 0.24, 0.40); arrow(0.17, 0.21, 0.24, 0.36)
arrow(0.44, 0.39, 0.50, 0.39); arrow(0.62, 0.58, 0.62, 0.44)
arrow(0.74, 0.39, 0.80, 0.50); arrow(0.885, 0.46, 0.885, 0.41); arrow(0.885, 0.32, 0.885, 0.27)
ax.text(0.63, 0.535, "CSD", fontsize=9, ha="center", color="#B03A2E")
ax.set_xlim(0, 1); ax.set_ylim(0.1, 0.9)
ax.set_title("Figure 1 — DR-VERGE: complementarity-shift distillation and INT8 deployment", fontsize=12)
save_figure(fig, "fig_01_architecture",
            pd.DataFrame([{"component": "teacher", "detail": "ResNet-50 dual-view + CORAL"},
                          {"component": "student", "detail": f"depthwise-separable {STUDENT_CHANNELS}"},
                          {"component": "distillation", "detail": "task + aux + logit-KD + CSD"},
                          {"component": "deployment", "detail": "FP32 / PTQ INT8 / QAT INT8"}]),
            "DR-VERGE architecture. Teacher produces dual and single-view cumulative probabilities; "
            "their difference is the complementarity shift distilled into the lightweight student.")

  saved fig_01_architecture (.png/.pdf/.svg + _data.csv)


In [46]:
# ---- Figure 2: experimental workflow ----
fig, ax = plt.subplots(figsize=(11, 6.5)); ax.axis("off"); ax.grid(False)
steps = [("DRTiD official train (1000 eyes)", 0.86, "#DCE9F7"),
         ("stratified split -> train 800 / val 200", 0.74, "#DCE9F7"),
         ("train all conditions (5 seeds)", 0.62, "#DCF7E3"),
         ("pre-registered grids -> select on VALIDATION only", 0.50, "#FFF3CD"),
         ("M* = best FP32 (validation-selected)", 0.38, "#DCF7E3"),
         ("PTQ INT8 / QAT INT8 / FP32-FT control", 0.26, "#E8DCF7"),
         ("DRTiD official test (no selection here)", 0.14, "#F7DCDC"),
         ("DeepDRiD external -- FROZEN, evaluated last", 0.03, "#F7DCDC")]
for txt, y, fc in steps:
    ax.add_patch(plt.Rectangle((0.18, y), 0.64, 0.075, facecolor=fc, edgecolor="#333", lw=1.3))
    ax.text(0.5, y + 0.037, txt, ha="center", va="center", fontsize=10)
for i in range(len(steps) - 1):
    y1 = steps[i][1]; y2 = steps[i + 1][1] + 0.075
    ax.annotate("", xy=(0.5, y2), xytext=(0.5, y1), arrowprops=dict(arrowstyle="<-", lw=1.5, color="#333"))
ax.set_xlim(0, 1); ax.set_ylim(0, 0.95)
ax.set_title("Figure 2 — Experimental workflow (selection never touches test or external data)", fontsize=12)
save_figure(fig, "fig_02_experimental_workflow", pd.DataFrame([{"step": t} for t, _, _ in steps]),
            "Experimental workflow. All selection happens on validation; DRTiD test is evaluated once "
            "within this run and DeepDRiD is frozen until the very end.")

  saved fig_02_experimental_workflow (.png/.pdf/.svg + _data.csv)


In [47]:
# ---- Figure 3: predictive performance comparison (QWK / Macro-F1 / Accuracy) ----
conds = order_present(["teacher", "macula_only", "disc_only", "dual_no_distill",
                       "dual_logitkd", "dual_featkd", "dual_csd"])
metrics3 = ["QWK", "MacroF1", "Accuracy"]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
rows3 = []
for ax, met in zip(axes, metrics3):
    mu, sd, ns = agg_stat(RAW, conds, met)
    ax.bar(range(len(conds)), mu, yerr=sd, capsize=4, color="#4C72B0", edgecolor="#25405e")
    for i, c in enumerate(conds):
        pts = RAW[RAW.condition == c][met].dropna().values
        if len(pts) > 1: ax.scatter([i] * len(pts), pts, s=16, color="#C44E52", zorder=3, alpha=.85)
    ax.set_xticks(range(len(conds))); ax.set_xticklabels(conds, rotation=35, ha="right", fontsize=9)
    ax.set_title(f"{met} (higher is better)"); ax.set_ylabel(met)
    for c, m_, s_, n_ in zip(conds, mu, sd, ns):
        rows3.append({"metric": met, "condition": c, "mean": m_, "sd": s_, "n_seeds": n_})
fig.suptitle("Figure 3 — Predictive performance on the internal DRTiD test set (mean ± SD over seeds; dots = individual seeds)", y=1.02)
save_figure(fig, "fig_03_performance_comparison", pd.DataFrame(rows3),
            "Predictive performance. Error bars are SD across seeds; red dots are individual seed values.")

  saved fig_03_performance_comparison (.png/.pdf/.svg + _data.csv)


In [48]:
# ---- Figure 4: efficiency Pareto frontier ----
eff_conds = order_present(["teacher", "best_fp32", "ptq_int8", "qat_int8"])
rows4 = []
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, (xcol, xlabel) in zip(axes, [("Latency_median_ms", "CPU latency, median (ms) — lower is better"),
                                     ("CheckpointSize_MB", "Serialized size (MB) — lower is better")]):
    for c in eff_conds:
        sub = RAW[RAW.condition == c]
        if not len(sub) or xcol not in sub: continue
        x, y = sub[xcol].mean(), sub["QWK"].mean()
        if pd.isna(x): continue
        mk = "D" if c == "teacher" else ("*" if "int8" in c else "o")
        ax.scatter(x, y, s=320 if mk != "o" else 150, marker=mk, label=c, edgecolor="#222", zorder=3)
        ax.annotate(c, (x, y), textcoords="offset points", xytext=(8, 6), fontsize=9)
        rows4.append({"axis": xcol, "condition": c, "x": x, "QWK": y})
    ax.set_xlabel(xlabel); ax.set_ylabel("QWK (higher is better)"); ax.set_xscale("log")
    ax.set_title("Top-left is better")
fig.suptitle("Figure 4 — Efficiency–Performance Pareto Frontier", y=1.02)
save_figure(fig, "fig_04_efficiency_pareto", pd.DataFrame(rows4),
            "Efficiency-performance frontier: QWK against CPU latency and serialized state_dict size (log x-axis).")

  saved fig_04_efficiency_pareto (.png/.pdf/.svg + _data.csv)


In [49]:
# ---- Figure 5: quantization retention (FP32 vs PTQ vs QAT) ----
qconds = order_present(["best_fp32", "ptq_int8", "qat_int8"])
qmetrics = ["QWK", "Accuracy", "MacroF1", "MacroRecall"]
rows5 = []
if len(qconds) >= 2:
    fig, ax = plt.subplots(figsize=(11, 5.5))
    w = 0.8 / len(qconds)
    for i, c in enumerate(qconds):
        vals = [RAW[RAW.condition == c][m].mean() for m in qmetrics]
        ax.bar(np.arange(len(qmetrics)) + i * w, vals, width=w, label=c, edgecolor="#25405e")
        for m, v in zip(qmetrics, vals): rows5.append({"condition": c, "metric": m, "value": v})
    ax.set_xticks(np.arange(len(qmetrics)) + w * (len(qconds) - 1) / 2); ax.set_xticklabels(qmetrics)
    ax.set_ylabel("score (higher is better)"); ax.legend()
    ax.set_title("Figure 5 — Quantization: FP32 vs PTQ INT8 vs QAT INT8")
    save_figure(fig, "fig_05_quantization_retention", pd.DataFrame(rows5),
                "Diagnostic performance retained after INT8 quantization.")
else:
    print("  fig_05 skipped: fewer than two quantization variants available")

  saved fig_05_quantization_retention (.png/.pdf/.svg + _data.csv)


In [50]:
# ---- Figure 6: per-grade sensitivity (the rev2 failure mode made permanently visible) ----
pg_conds = order_present(["teacher", "best_fp32", "dual_csd", "ptq_int8", "qat_int8"])
rows6 = []
fig, ax = plt.subplots(figsize=(11, 5.5))
w = 0.8 / max(len(pg_conds), 1)
for i, c in enumerate(pg_conds):
    vals = [RAW[RAW.condition == c][f"Sensitivity_Grade{g}"].mean() for g in range(NUM_CLASSES)]
    ax.bar(np.arange(NUM_CLASSES) + i * w, vals, width=w, label=c, edgecolor="#25405e")
    for g, v in enumerate(vals): rows6.append({"condition": c, "grade": g, "sensitivity": v})
ax.axhline(0.05, color="#C44E52", ls="--", lw=1.2, label="collapse floor (0.05)")
ax.set_xticks(np.arange(NUM_CLASSES) + w * (len(pg_conds) - 1) / 2)
ax.set_xticklabels([f"Grade {g}" for g in range(NUM_CLASSES)])
ax.set_ylabel("Sensitivity / recall (higher is better)"); ax.set_ylim(0, 1); ax.legend(fontsize=9)
ax.set_title("Figure 6 — Per-grade sensitivity: are intermediate grades actually predicted?")
save_figure(fig, "fig_06_per_grade_sensitivity", pd.DataFrame(rows6),
            "Per-grade recall. Grades 1-3 near zero indicates the ordinal collapse seen in rev2.")

  saved fig_06_per_grade_sensitivity (.png/.pdf/.svg + _data.csv)


In [51]:
# ---- Figure 7: normalized confusion matrices ----
cm_conds = order_present(["teacher", "best_fp32", "ptq_int8", "qat_int8"])
rows7 = []
if cm_conds:
    fig, axes = plt.subplots(1, len(cm_conds), figsize=(4.6 * len(cm_conds), 4.4))
    if len(cm_conds) == 1: axes = [axes]
    for ax, c in zip(axes, cm_conds):
        key = next((k for k in PRED_STORE if k[0] == c), None)
        if key is None: ax.axis("off"); continue
        d = PRED_STORE[key]
        cm = confusion_matrix(d["y_true"], d["y_pred"], labels=list(range(NUM_CLASSES)))
        cmn = cm.astype(float) / np.maximum(cm.sum(1, keepdims=True), 1)
        im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1); ax.grid(False)
        ax.set_title(c, fontsize=11); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
        for i in range(NUM_CLASSES):
            for j in range(NUM_CLASSES):
                ax.text(j, i, f"{cmn[i,j]:.2f}", ha="center", va="center", fontsize=8,
                        color="white" if cmn[i, j] > .5 else "black")
                rows7.append({"condition": c, "true": i, "pred": j, "count": int(cm[i, j]),
                              "normalized": float(cmn[i, j])})
    fig.suptitle("Figure 7 — Row-normalized confusion matrices (internal test set)", y=1.03)
    save_figure(fig, "fig_07_confusion_matrices", pd.DataFrame(rows7),
                "Row-normalized confusion matrices; diagonal = per-grade recall.")

  saved fig_07_confusion_matrices (.png/.pdf/.svg + _data.csv)


In [52]:
# ---- Figure 8: CSD mechanism (the primary RQ1 mechanism figure) ----
mech_conds = order_present(["dual_no_distill", "dual_logitkd", "dual_featkd", "dual_csd",
                            "abl_csd_counterfactual"])
mech = ["ShiftL1", "CosAgree", "BenefitCorr"]
rows8 = []
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, met in zip(axes, mech):
    mu, sd, ns = agg_stat(RAW, mech_conds, met)
    better = "lower is better" if met in ("ShiftL1", "ShiftMAE") else "higher is better"
    ax.bar(range(len(mech_conds)), mu, yerr=sd, capsize=4, color="#55A868", edgecolor="#2f5d3f")
    ax.set_xticks(range(len(mech_conds)))
    ax.set_xticklabels([pretty(c) for c in mech_conds], rotation=35, ha="right", fontsize=8)
    ax.set_title(f"{met} ({better})"); ax.set_ylabel(met)
    for c, m_, s_, n_ in zip(mech_conds, mu, sd, ns):
        rows8.append({"metric": met, "condition": c, "mean": m_, "sd": s_, "n_seeds": n_})
fig.suptitle("Figure 8 — CSD mechanism: is the teacher's complementarity shift actually transferred?", y=1.02)
save_figure(fig, "fig_08_csd_mechanism", pd.DataFrame(rows8),
            "Mechanism fidelity. ShiftMAE lower = student shift closer to teacher; CosAgree higher = same "
            "shift direction; BenefitCorr higher = student gains from dual-view on the same samples as the teacher.")

  saved fig_08_csd_mechanism (.png/.pdf/.svg + _data.csv)


In [53]:
# ---- Figure 9: gradient contributions over training ----
rows9 = []
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for cond in ["dual_csd", "dual_logitkd", "dual_no_distill", "dual_featkd"]:
    f = f"{LOGS_DIR}/gradient_contributions_{cond}_{PRIMARY_SEED}.csv"
    if not os.path.exists(f): continue
    h = pd.read_csv(f)
    if "gnorm_task" in h:
        axes[0].plot(h["epoch"], h["gnorm_task"], label=f"{cond}: task", lw=1.4)
    if "gnorm_csd" in h:
        axes[0].plot(h["epoch"], h["gnorm_csd"], label=f"{cond}: CSD", lw=1.8, ls="--")
    if "gnorm_ratio_csd_over_task" in h:
        axes[1].plot(h["epoch"], h["gnorm_ratio_csd_over_task"], label=cond, lw=1.8)
    for _, r in h.iterrows():
        rows9.append({"condition": cond, **{k: r[k] for k in h.columns}})
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("gradient L2 norm"); axes[0].set_yscale("log")
axes[0].set_title("Per-component gradient norm"); axes[0].legend(fontsize=8)
axes[1].axhline(1.0, color="#888", ls=":", lw=1)
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("‖∇L_CSD‖ / ‖∇L_task‖")
axes[1].set_title("CSD gradient share (rev2 was ≈0 — CSD had no influence)"); axes[1].legend(fontsize=8)
fig.suptitle("Figure 9 — Optimization signal: does CSD actually contribute gradient?", y=1.02)
save_figure(fig, "fig_09_gradient_contribution", pd.DataFrame(rows9),
            "Gradient norms per loss component. The ratio panel shows whether CSD is a real training "
            "signal or numerical decoration.")

  saved fig_09_gradient_contribution (.png/.pdf/.svg + _data.csv)


In [54]:
# ---- Figure 10: reliability diagrams (calibration) ----
rel_conds = order_present(["best_fp32", "fp32_ft_control", "ptq_int8", "qat_int8"])
rows10 = []
if rel_conds:
    fig, ax = plt.subplots(figsize=(7.5, 6.5))
    ax.plot([0, 1], [0, 1], ls="--", color="#888", label="perfectly calibrated")
    for c in rel_conds:
        # Resolve the prediction file from PRED_STORE's OWN keys. The earlier code hard-coded
        # seed{BEST_SEED} for every condition, but QAT is trained on SEEDS_QAT and need not contain
        # BEST_SEED at all -- so the QAT reliability curve could silently disappear from the figure.
        keys = [k for k in PRED_STORE if k[0] == c]
        if not keys: continue
        key = next((k for k in keys if k[1] == QAT_DEPLOY_SEED), keys[0]) if c == "qat_int8" else keys[0]
        _c, _sd, _q = key
        pf = f"{PREDS_DIR}/DRTiD_test_{_c}_seed{_sd}_{_q}.csv"
        if not os.path.exists(pf): continue
        dfp = pd.read_csv(pf)
        p = torch.tensor(dfp[[f"p_threshold_{k}" for k in range(NUM_THRESHOLDS)]].values, dtype=torch.float32)
        rc = reliability_curve(p, dfp["true_grade"].values)
        ax.plot(rc["mean_predicted"], rc["observed_frequency"], marker="o", lw=1.8,
                label=f"{pretty(c)} (seed {_sd})")
        cal = compute_calibration(p, dfp["true_grade"].values)
        for _, r in rc.iterrows():
            rows10.append({"condition": c, "seed": _sd, **r.to_dict(),
                           "OrdinalThreshold_ECE": cal["OrdinalThreshold_ECE"],
                           "OrdinalThreshold_Brier": cal["OrdinalThreshold_Brier"]})
    ax.set_xlabel("Mean predicted P(y>k)"); ax.set_ylabel("Observed frequency")
    ax.set_title("Figure 10 — Reliability diagram (pooled cumulative thresholds)")
    ax.legend(fontsize=9)
    save_figure(fig, "fig_10_calibration_reliability", pd.DataFrame(rows10),
                "Reliability diagram. Deviation from the diagonal indicates miscalibration; "
                "companion CSV carries ECE and Brier per condition.")
print(f"\nAll figures written to {FIGURES_DIR}")

  saved fig_10_calibration_reliability (.png/.pdf/.svg + _data.csv)

All figures written to /content/drive/MyDrive/DR-VERGE/artifacts_preflight_v1/results/figures


In [55]:
# ---- Figure 11: internal vs external generalization ----
if len(EXT_DF):
    prim = EXT_DF[(EXT_DF.field_order == DEEPDRID_PRIMARY_FIELD_ORDER) &
                  (EXT_DF.subset == DEEPDRID_PRIMARY_SUBSET)]
    mets = ["QWK", "MacroF1", "Accuracy"]
    conds = [c for c in ["teacher", "best_fp32", "best_csd_fp32", "ptq_int8", "qat_int8"]
             if c in set(prim.condition)]
    rows = []
    fig, axes = plt.subplots(1, len(mets), figsize=(5.2 * len(mets), 5))
    for ax, met in zip(axes, mets):
        internal = [RAW[RAW.condition == c][met].mean() for c in conds]
        external = [prim[prim.condition == c][met].mean() for c in conds]
        x = np.arange(len(conds)); w = 0.38
        ax.bar(x - w/2, internal, w, label="DRTiD (internal)", edgecolor="#25405e")
        ax.bar(x + w/2, external, w, label="DeepDRiD (external, frozen)", edgecolor="#5e2540")
        ax.set_xticks(x); ax.set_xticklabels([pretty(c) for c in conds], rotation=30, ha="right", fontsize=8)
        ax.set_title(f"{met} (higher is better)"); ax.legend(fontsize=8)
        for c, i_, e_ in zip(conds, internal, external):
            rows.append({"metric": met, "condition": c, "internal_DRTiD": i_, "external_DeepDRiD": e_,
                         "delta_external_minus_internal": e_ - i_})
    fig.suptitle("Figure 11 — Internal vs external generalization "
                 f"(DeepDRiD {DEEPDRID_PRIMARY_SUBSET} partition, primary field ordering)", y=1.02)
    save_figure(fig, "fig_11_external_generalization", pd.DataFrame(rows),
                "Internal (DRTiD) vs external (DeepDRiD) performance. A drop is a domain-shift "
                "finding to report, not something to tune away.")
else:
    print("  fig_11 skipped: no external validation results available")

  saved fig_11_external_generalization (.png/.pdf/.svg + _data.csv)


## 39 — Final research tables

In [56]:
# table_diagnostic_performance
diag_cols = ["QWK", "Accuracy", "MacroPrecision", "MacroRecall", "MacroF1", "WeightedF1",
             "BalancedAccuracy", "MAE", "SevereErrorRate", "ECE", "Brier"]
t_diag = RAW.groupby("condition")[diag_cols].agg(["mean", "std"]).round(4)
t_diag.columns = ["_".join(c) for c in t_diag.columns]
t_diag = t_diag.reindex([c for c in DISPLAY_ORDER if c in t_diag.index])
t_diag.to_csv(f"{TABLES_DIR}/table_diagnostic_performance.csv")

# table_efficiency
eff_cols = [c for c in ["ParamCount", "CheckpointSize_MB", "Latency_mean_ms", "Latency_median_ms",
                        "Latency_sd_ms", "Latency_p95_ms", "Latency_p99_ms",
                        "Throughput_pairs_per_s", "Throughput_images_per_s"]
            if c in RAW.columns]
t_eff = RAW.groupby("condition")[eff_cols].mean().round(4)
ref_lat = t_eff.loc["teacher", "Latency_median_ms"] if "teacher" in t_eff.index else np.nan
ref_size = t_eff.loc["teacher", "CheckpointSize_MB"] if "teacher" in t_eff.index else np.nan
t_eff["Speedup_vs_teacher"] = (ref_lat / t_eff["Latency_median_ms"]).round(2)
t_eff["CompressionRatio_vs_teacher"] = (ref_size / t_eff["CheckpointSize_MB"]).round(2)
t_eff["SizeReduction_vs_teacher_pct"] = ((1 - t_eff["CheckpointSize_MB"] / ref_size) * 100).round(2)
t_eff = t_eff.reindex([c for c in DISPLAY_ORDER if c in t_eff.index])
t_eff.to_csv(f"{TABLES_DIR}/table_efficiency.csv")

# table_quantization (FP32 vs PTQ vs QAT) + retention
qrows = []
fp32_m = RAW[RAW.condition == "best_fp32"][diag_cols].mean().to_dict() if "best_fp32" in set(RAW.condition) else {}
for c in ["best_fp32", "fp32_ft_control", "fp32_ft_plain", "ptq_int8", "qat_int8", "ptq_int8_pt2e"]:
    if c not in set(RAW.condition): continue
    m = RAW[RAW.condition == c][diag_cols].mean().to_dict()
    e = RAW[RAW.condition == c][eff_cols].mean().to_dict()
    row = {"model": c, **{k: round(v, 4) for k, v in m.items()}, **{k: round(v, 4) for k, v in e.items()}}
    if fp32_m and c != "best_fp32":
        row.update({k: round(v, 3) for k, v in retention_metrics(m, fp32_m).items()})
        row.update(efficiency_derived(e.get("CheckpointSize_MB"), fp32_m and RAW[RAW.condition=="best_fp32"]["CheckpointSize_MB"].mean(),
                                      e.get("Latency_median_ms"), RAW[RAW.condition=="best_fp32"]["Latency_median_ms"].mean()))
    qrows.append(row)
t_quant = pd.DataFrame(qrows)
t_quant.to_csv(f"{TABLES_DIR}/table_quantization.csv", index=False)

# table_csd_mechanism
mech_cols = ["ShiftL1", "ShiftMAE", "CosAgree", "BenefitCorr", "BenefitCorrSpearman",
             "DualViewGain_G_internal", "DualViewGain_G_external"]
t_mech = RAW[RAW.view_mode == "dual"].groupby("condition")[
    [c for c in mech_cols if c in RAW.columns]].agg(["mean", "std"]).round(4)
t_mech.columns = ["_".join(c) for c in t_mech.columns]
t_mech.to_csv(f"{TABLES_DIR}/table_csd_mechanism.csv")

print("Diagnostic performance:\n", t_diag[[c for c in t_diag.columns if c.endswith("_mean")]].to_string())
print("\nEfficiency:\n", t_eff.to_string())
print("\nQuantization:\n", t_quant.to_string(index=False))
print("\nCSD mechanism:\n", t_mech.to_string())

Diagnostic performance:
                         QWK_mean  Accuracy_mean  MacroPrecision_mean  MacroRecall_mean  MacroF1_mean  WeightedF1_mean  BalancedAccuracy_mean  MAE_mean  SevereErrorRate_mean  ECE_mean  Brier_mean
condition                                                                                                                                                                                         
teacher                   0.6077         0.5018               0.3342            0.3847        0.3443           0.4796                 0.3847    0.7636                0.2327    0.0579      0.1254
macula_only               0.1373         0.2636               0.2487            0.2128        0.1650           0.2373                 0.2128    1.1200                0.3636    0.0872      0.1736
disc_only                 0.1236         0.3273               0.1650            0.1833        0.1688           0.3205                 0.1833    1.0836                0.3327    0.0245      0.1645


## 40–41 — Automatic headline generator & final gate report

In [57]:
def _mean(cond, col):
    v = RAW[RAW.condition == cond][col].dropna()
    return float(v.mean()) if len(v) else float("nan")

headlines = []
t_q, s_q = _mean("teacher", "QWK"), _mean("best_fp32", "QWK")
t_p, s_p = _mean("teacher", "ParamCount"), _mean("best_fp32", "ParamCount")
t_l, s_l = _mean("teacher", "Latency_median_ms"), _mean("best_fp32", "Latency_median_ms")
t_s, s_s = _mean("teacher", "CheckpointSize_MB"), _mean("best_fp32", "CheckpointSize_MB")
if not math.isnan(t_p) and s_p:
    headlines.append(f"Teacher -> Student: {t_p/s_p:.0f}x fewer parameters, {t_s/s_s:.1f}x smaller artifact, "
                     f"{t_l/s_l:.1f}x faster CPU inference, {100*s_q/t_q:.1f}% of teacher QWK retained")
for q in ["ptq_int8", "qat_int8"]:
    if q not in set(RAW.condition): continue
    qq, ql, qs = _mean(q, "QWK"), _mean(q, "Latency_median_ms"), _mean(q, "CheckpointSize_MB")
    headlines.append(f"FP32 -> {q.upper()}: {100*qq/s_q:.1f}% QWK retained, "
                     f"{s_l/ql:.2f}x CPU speedup, {s_s/qs:.2f}x smaller"
                     if not math.isnan(qs) and qs else
                     f"FP32 -> {q.upper()}: {100*qq/s_q:.1f}% QWK retained, {s_l/ql:.2f}x CPU speedup")

print("=" * 78); print("AUTOMATIC HEADLINES (computed, NOT significance claims)"); print("=" * 78)
for h in headlines: print("  " + h)
pd.DataFrame({"headline": headlines}).to_csv(f"{TABLES_DIR}/table_headlines.csv", index=False)

AUTOMATIC HEADLINES (computed, NOT significance claims)
  Teacher -> Student: 122x fewer parameters, 118.4x smaller artifact, 14.0x faster CPU inference, 14.0% of teacher QWK retained
  FP32 -> PTQ_INT8: 124.6% QWK retained, 1.21x CPU speedup, 1.36x smaller
  FP32 -> QAT_INT8: 154.4% QWK retained, 1.70x CPU speedup, 1.36x smaller


In [58]:
# ---- Gate 5: RQ1 verdict on predictive AND mechanistic axes ----
print("=" * 78); print("GATE 5 -- RQ1 VERDICT"); print("=" * 78)
csd_q = _mean("dual_csd", "QWK")
verdict = {}
for base in ["dual_no_distill", "dual_logitkd", "dual_featkd"]:
    if base not in set(RAW.condition): continue
    bq = _mean(base, "QWK")
    st = STATS[(STATS.comparison == f"dual_csd_vs_{base}") & (STATS.metric == "QWK")]
    ci = f"[{st.iloc[0]['ci_low']:+.4f}, {st.iloc[0]['ci_high']:+.4f}]" if len(st) else "n/a"
    cred = bool(st.iloc[0]["excludes_zero"]) if len(st) else None
    verdict[base] = {"csd_qwk": csd_q, "baseline_qwk": bq, "diff": csd_q - bq,
                     "ci_95": ci, "credible": cred}
    print(f"  CSD vs {base:18s}: {csd_q:.4f} vs {bq:.4f} (diff {csd_q-bq:+.4f}, 95% CI {ci}, credible={cred})")

print("\n  Mechanism (does CSD transfer the shift, independently of QWK?)")
for c in order_present(["dual_no_distill", "dual_logitkd", "dual_featkd", "dual_csd"]):
    print(f"    {c:18s} ShiftL1={_mean(c,'ShiftL1'):.4f}  CosAgree={_mean(c,'CosAgree'):+.4f}  "
          f"BenefitCorr(r)={_mean(c,'BenefitCorr'):+.4f}  (rho)={_mean(c,'BenefitCorrSpearman'):+.4f}")
save_json(verdict, f"{RESULTS_DIR}/rq1_verdict.json")
record_gate("Gate5_RQ1_Comparison", True, "RQ1 comparisons computed on predictive and mechanistic axes")
print("\n  A negative or null RQ1 result is a valid, reportable finding -- do not re-tune the")
print("  method in response to this table. The protocol was locked before the run.")

GATE 5 -- RQ1 VERDICT
  CSD vs dual_no_distill   : 0.0649 vs 0.1139 (diff -0.0490, 95% CI [-0.1044, +0.0076], credible=False)
  CSD vs dual_logitkd      : 0.0649 vs 0.1141 (diff -0.0492, 95% CI [-0.1033, +0.0066], credible=False)
  CSD vs dual_featkd       : 0.0649 vs 0.0848 (diff -0.0198, 95% CI [-0.0783, +0.0488], credible=False)

  Mechanism (does CSD transfer the shift, independently of QWK?)
    dual_no_distill    ShiftL1=0.4816  CosAgree=+0.2159  BenefitCorr(r)=-0.0143  (rho)=+0.1562
    dual_logitkd       ShiftL1=0.4876  CosAgree=+0.2170  BenefitCorr(r)=+0.0214  (rho)=+0.1751
    dual_featkd        ShiftL1=0.4735  CosAgree=+0.2173  BenefitCorr(r)=-0.0043  (rho)=+0.1575
    dual_csd           ShiftL1=0.4571  CosAgree=+0.1733  BenefitCorr(r)=+0.0193  (rho)=+0.1782
PASS | Gate5_RQ1_Comparison | RQ1 comparisons computed on predictive and mechanistic axes

  A negative or null RQ1 result is a valid, reportable finding -- do not re-tune the
  method in response to this table. The prot

In [59]:
# ---- Final consolidated gate report ----
gate_df = pd.DataFrame([{"gate": k, "passed": v["passed"], "detail": v["detail"]} for k, v in GATES.items()])
gate_df.to_csv(f"{TABLES_DIR}/table_gate_report.csv", index=False)
print("=" * 78); print("FINAL GATE REPORT"); print("=" * 78)
print(gate_df.to_string(index=False))
n_pass = int(gate_df.passed.sum())
print(f"\n{n_pass}/{len(gate_df)} gates passed.")
failed = gate_df[~gate_df.passed]
if len(failed):
    print("\nFAILED / NOT-RUN gates (report these honestly rather than hiding them):")
    for _, r in failed.iterrows(): print(f"  - {r['gate']}: {r['detail']}")

RUN_SUMMARY = {
    "environment": ENVIRONMENT, "config": CONFIG_SNAPSHOT,
    "selection": {"best_condition": BEST_CONDITION, "best_seed": BEST_SEED,
                  "best_val_qwk": float(BEST_ROW["QWK"]), "best_csd_seed": BEST_CSD_SEED},
    "csd_selected": {"variant": BEST_CSD_VARIANT, "alpha": BEST_ALPHA, "beta": BEST_BETA},
    "gates": GATES, "headlines": headlines, "rq1_verdict": verdict,
    "n_evaluated_runs": int(len(RAW)),
    "external_validation": "completed" if len(EXT_DF) else "skipped/unavailable",
}
save_json(RUN_SUMMARY, f"{RESULTS_DIR}/run_summary.json")

print(f"""
{'='*78}
ARTIFACTS
{'='*78}
  checkpoints : {CKPT_DIR}
  models      : {MODELS_DIR}   (checkpoint.pt / model.pt2 / model.onnx / metadata.json)
  figures     : {FIGURES_DIR}  (png+pdf+svg + *_data.csv per figure)
  tables      : {TABLES_DIR}
  metrics     : {METRICS_DIR}
  predictions : {PREDS_DIR}    (per-sample, so metrics can be recomputed without re-inference)
  logs        : {LOGS_DIR}
  registry    : {ART}/model_registry.csv
  summary     : {RESULTS_DIR}/run_summary.json
""")

FINAL GATE REPORT
                             gate  passed                                                                                                                                                                                                                         detail
                Gate0_Environment    True                                                                                                                                               torch=2.11.0+cu128 torchao=0.10.0 engines=['qnnpack', 'onednn', 'x86', 'fbgemm']
                    Gate1_Dataset    True                                                                                                                                       train/val/test = 800/200/550 eyes; no ID overlap; all grades present; all images resolve
             Gate_CORAL_UnitTests    True                                                                                                                                                  

## Done

Read in this order when writing the paper:

1. **`table_gate_report.csv`** — did anything fail? Report failures honestly.
2. **Gate 5 / `rq1_verdict.json`** — RQ1 on both axes (predictive *and* mechanistic).
3. **`table_quantization.csv`** — RQ2: FP32 vs PTQ vs QAT, plus both FP32 fine-tuning controls.
4. **`table_05_statistical_tests.csv`** — effect sizes with CIs; a difference is not a claim unless
   the CI excludes zero. Check `matched_seeds` before describing a row as paired.
5. **`table_06_external_validation_deepdrid.csv`** — external generalization. Quote the
   `role=PRIMARY` rows (validation partition, primary field ordering) as the headline; the other
   rows are supplementary robustness.
6. **`table_condition_labels.csv`** — the exact label to use for each condition in every table.
7. **`configs/quantization_info.json`** — states the locked scope, which path each result came from,
   and that `quantization_coverage_pct` is an integrity check rather than a reportable number.
8. **`deployment_verification.json`** — which artifacts were actually re-loaded from disk and passed.

Use `docs/judge.md` Section I's safe phrasing for every claim. Do not describe CSD as successful on
mechanism metrics alone if predictive performance did not move; do not call the model
deployment-ready on the basis of size alone.